In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

print("Drive connected successfully.")

Mounted at /content/drive
Drive connected successfully.


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
os.makedirs(WORK_DIR, exist_ok=True)

CSV_PATH = os.path.join(WORK_DIR, "fitzpatrick17k.csv")

!wget -q -O "$CSV_PATH" \
"https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"

print("Downloaded:", os.path.exists(CSV_PATH))
print("File size:", os.path.getsize(CSV_PATH), "bytes")

df = pd.read_csv(CSV_PATH)

print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nNumber of unique condition labels:")
print(df["label"].nunique())

print("\nTop 20 conditions:")
print(df["label"].value_counts().head(20))

Downloaded: True
File size: 4104341 bytes

Dataset shape: (16577, 9)

Columns:
['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']

Number of unique condition labels:
114

Top 20 conditions:
label
psoriasis                      653
squamous cell carcinoma        581
lichen planus                  491
basal cell carcinoma           468
allergic contact dermatitis    430
lupus erythematosus            410
neutrophilic dermatoses        361
sarcoidosis                    349
photodermatoses                348
folliculitis                   342
scabies                        339
acne vulgaris                  335
scleroderma                    309
pityriasis rubra pilaris       278
melanoma                       261
nematode infection             260
erythema multiforme            236
granuloma annulare             211
eczema                         204
drug eruption                  200
Name: count,

In [ ]:
# ==========================================
# CELL 3 — ANALYZE ALL 114 CONDITION CLASSES
# ==========================================

label_counts = (
    df["label"]
    .value_counts()
    .rename_axis("condition")
    .reset_index(name="image_count")
)

print("Total conditions:", len(label_counts))
print("Total metadata records:", len(df))

print("\nClass statistics:")
print("Largest class:", label_counts["image_count"].max())
print("Smallest class:", label_counts["image_count"].min())
print("Median images/class:", label_counts["image_count"].median())

print("\nClasses with >= 100 images:",
      (label_counts["image_count"] >= 100).sum())

print("Classes with >= 50 images:",
      (label_counts["image_count"] >= 50).sum())

print("Classes with >= 20 images:",
      (label_counts["image_count"] >= 20).sum())

print("\nAll 114 conditions:")
display(label_counts)

# Save this for project documentation
COUNTS_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_114_class_counts.csv"
)

label_counts.to_csv(COUNTS_PATH, index=False)

print("\nSaved class-count report:")
print(COUNTS_PATH)

Total conditions: 114
Total metadata records: 16577

Class statistics:
Largest class: 653
Smallest class: 53
Median images/class: 102.0

Classes with >= 100 images: 60
Classes with >= 50 images: 114
Classes with >= 20 images: 114

All 114 conditions:


,condition,image_count
0,psoriasis,653
1,squamous cell carcinoma,581
2,lichen planus,491
3,basal cell carcinoma,468
4,allergic contact dermatitis,430
...,...,...
109,paronychia,59
110,erythema elevatum diutinum,55
111,pustular psoriasis,53
112,pilomatricoma,53



Saved class-count report:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick17k_114_class_counts.csv


In [ ]:
# ==========================================
# CELL 4 — DOWNLOAD FITZPATRICK17K IMAGES
# ==========================================

import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

IMAGE_DIR = os.path.join(WORK_DIR, "images")
os.makedirs(IMAGE_DIR, exist_ok=True)

DOWNLOAD_MANIFEST = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_download_manifest.csv"
)

def download_image(row):
    idx = row.name
    url = row["url"]

    # Stable filename using md5hash
    filename = f"{row['md5hash']}.jpg"
    filepath = os.path.join(IMAGE_DIR, filename)

    # Don't download again if already present
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1000:
        return idx, filepath, "existing"

    try:
        response = requests.get(
            url,
            timeout=12,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        if response.status_code == 200 and len(response.content) > 1000:
            with open(filepath, "wb") as f:
                f.write(response.content)

            return idx, filepath, "downloaded"

        return idx, None, f"http_{response.status_code}"

    except Exception as e:
        return idx, None, "failed"


records = []

with ThreadPoolExecutor(max_workers=16) as executor:

    futures = {
        executor.submit(download_image, row): idx
        for idx, row in df.iterrows()
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Downloading Fitzpatrick17k"
    ):
        idx, filepath, status = future.result()

        records.append({
            "index": idx,
            "filepath": filepath,
            "status": status
        })


download_df = pd.DataFrame(records)

# Join download results back to metadata
manifest = (
    df.reset_index()
      .merge(download_df, on="index", how="left")
)

manifest.to_csv(DOWNLOAD_MANIFEST, index=False)


# --------------------------
# RESULTS
# --------------------------

valid = manifest["filepath"].notna().sum()

print("\n==============================")
print("DOWNLOAD COMPLETE")
print("==============================")

print("Metadata records :", len(manifest))
print("Usable downloads :", valid)
print("Unavailable      :", len(manifest) - valid)

print("\nDownload status:")
print(manifest["status"].value_counts())

print("\nUsable images per class:")

usable_counts = (
    manifest[manifest["filepath"].notna()]
    ["label"]
    .value_counts()
)

print(usable_counts)

print("\nClasses remaining:",
      usable_counts.index.nunique())

print("\nSmallest usable class:",
      usable_counts.min())

print("\nManifest saved:")
print(DOWNLOAD_MANIFEST)


DOWNLOAD COMPLETE
Metadata records : 16577
Usable downloads : 3887
Unavailable      : 12690

Download status:
status
failed        12672
downloaded     3887
http_404         18
Name: count, dtype: int64

Usable images per class:
label
basal cell carcinoma        257
lichen planus               230
squamous cell carcinoma     212
pityriasis rubra pilaris    175
scabies                     138
                           ... 
congenital nevus              2
paronychia                    2
epidermal nevus               2
lyme disease                  1
perioral dermatitis           1
Name: count, Length: 90, dtype: int64

Classes remaining: 90

Smallest usable class: 1

Manifest saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick17k_download_manifest.csv


In [ ]:
import os

folder = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

print("Files in Fitzpatrick114:\n")

for item in os.listdir(folder):
    path = os.path.join(folder, item)
    if os.path.isfile(path):
        print(item, "—", round(os.path.getsize(path) / (1024**2), 2), "MB")

Files in Fitzpatrick114:

fitzpatrick17k.csv — 3.91 MB
fitzpatrick17k_114_class_counts.csv — 0.0 MB
fitzpatrick17k_download_manifest.csv — 4.48 MB
fitzpatrick17k-main.zip — 6.86 MB


In [ ]:
import os

folder = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

print("Files found:\n")

zip_files = []

for file in os.listdir(folder):
    path = os.path.join(folder, file)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"{file} — {size_mb:.2f} MB")

        if file.lower().endswith(".zip"):
            zip_files.append(path)

print("\nZIP files found:", len(zip_files))

for z in zip_files:
    print(z)

Files found:

fitzpatrick17k.csv — 3.91 MB
fitzpatrick17k_114_class_counts.csv — 0.00 MB
fitzpatrick17k_download_manifest.csv — 4.48 MB
fitzpatrick17k-main.zip — 6.86 MB

ZIP files found: 1
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick17k-main.zip


In [ ]:
import os
import zipfile

folder = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
EXTRACT_DIR = "/content/fitzpatrick114_images"

# Find ZIP files automatically
zip_files = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.lower().endswith(".zip")
]

if len(zip_files) == 0:
    print("❌ No ZIP file found in:", folder)

else:
    ZIP_PATH = zip_files[0]

    print("Using ZIP:")
    print(ZIP_PATH)

    os.makedirs(EXTRACT_DIR, exist_ok=True)

    print("\nExtracting...")

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    print("✅ Extraction complete")

    # Find actual images
    extensions = (".jpg", ".jpeg", ".png", ".webp")
    image_files = []

    for root, dirs, files in os.walk(EXTRACT_DIR):
        for file in files:
            if file.lower().endswith(extensions):
                image_files.append(
                    os.path.join(root, file)
                )

    print("\nActual image files found:", len(image_files))

    print("\nFirst 10 image paths:")
    for path in image_files[:10]:
        print(path)

Using ZIP:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick17k-main.zip

Extracting...
✅ Extraction complete

Actual image files found: 0

First 10 image paths:


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
CSV_PATH = os.path.join(WORK_DIR, "fitzpatrick17k.csv")
MANIFEST_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_download_manifest.csv"
)

df = pd.read_csv(CSV_PATH)
manifest = pd.read_csv(MANIFEST_PATH)

print("Metadata:", df.shape)
print("Manifest:", manifest.shape)

print("\nMetadata columns:")
print(df.columns.tolist())

print("\nManifest columns:")
print(manifest.columns.tolist())

print("\nFirst manifest rows:")
display(manifest.head())

Metadata: (16577, 9)
Manifest: (16577, 12)

Metadata columns:
['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']

Manifest columns:
['index', 'md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'filepath', 'status']

First manifest rows:


,index,md5hash,fitzpatrick_scale,fitzpatrick_centaur,label,nine_partition_label,three_partition_label,qc,url,url_alphanum,filepath,status
0,0,5e82a45bc5d78bd24ae9202d194423f8,3,3,drug induced pigmentary changes,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicmminoc...,NaN,failed
1,1,fa2911a9b13b6f8af79cb700937cc14f,1,1,photodermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicpphoto...,NaN,failed
2,2,d2bac3c9e4499032ca8e9b07c7d3bc40,2,3,dermatofibroma,benign dermal,benign,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicdderma...,NaN,failed
3,3,0a94359e7eaacd7178e06b2823777789,1,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppsori...,NaN,failed
4,4,a39ec3b1f22c08a421fa20535e037bba,1,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppsori...,NaN,failed


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
MANIFEST_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_download_manifest.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)

# Keep only successfully downloaded images
usable = manifest[
    (manifest["status"] == "downloaded") &
    (manifest["filepath"].notna())
].copy()

# Confirm the files really exist
usable = usable[
    usable["filepath"].apply(lambda x: os.path.isfile(str(x)))
].copy()

# Remove duplicate images by MD5
usable = usable.drop_duplicates(
    subset=["md5hash"]
).reset_index(drop=True)

# Clean labels
usable["label"] = usable["label"].astype(str).str.strip()

print("Usable unique images:", len(usable))
print("Conditions represented:", usable["label"].nunique())

counts = usable["label"].value_counts()

print("\nLargest class:", counts.index[0], "-", counts.iloc[0])
print("Smallest class:", counts.index[-1], "-", counts.iloc[-1])

print("\nClasses by image count:")
print(">= 20 images :", (counts >= 20).sum())
print("10-19 images :", ((counts >= 10) & (counts < 20)).sum())
print("5-9 images   :", ((counts >= 5) & (counts < 10)).sum())
print("1-4 images   :", ((counts >= 1) & (counts < 5)).sum())

print("\nAll represented conditions:")
for i, (label, count) in enumerate(counts.items(), start=1):
    print(f"{i:3}. {label:<45} {count}")

# Save clean manifest
CLEAN_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_usable_90class.csv"
)

usable.to_csv(CLEAN_PATH, index=False)

print("\nSaved:")
print(CLEAN_PATH)

Usable unique images: 3887
Conditions represented: 90

Largest class: basal cell carcinoma - 257
Smallest class: perioral dermatitis - 1

Classes by image count:
>= 20 images : 53
10-19 images : 20
5-9 images   : 7
1-4 images   : 10

All represented conditions:
  1. basal cell carcinoma                          257
  2. lichen planus                                 230
  3. squamous cell carcinoma                       212
  4. pityriasis rubra pilaris                      175
  5. scabies                                       138
  6. porokeratosis actinic                         127
  7. nematode infection                            126
  8. lupus erythematosus                           108
  9. tungiasis                                     107
 10. psoriasis                                     104
 11. folliculitis                                  102
 12. hailey hailey disease                         97
 13. pityriasis rosea                              95
 14. neutrophilic dermato

In [ ]:
import os
import pandas as pd
import numpy as np

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

CLEAN_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_usable_90class.csv"
)

df = pd.read_csv(CLEAN_PATH)

rng = np.random.default_rng(42)

train_parts = []
val_parts = []
test_parts = []

# Split each condition separately
for label, group in df.groupby("label"):

    group = group.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    n = len(group)

    if n >= 10:
        # roughly 70 / 15 / 15
        n_test = max(1, round(n * 0.15))
        n_val = max(1, round(n * 0.15))

    elif n >= 5:
        # preserve more rare examples for training
        n_test = 1
        n_val = 1

    elif n >= 3:
        # extremely small class
        n_test = 1
        n_val = 0

    else:
        # 1-2 images: training only
        n_test = 0
        n_val = 0

    test = group.iloc[:n_test]

    val = group.iloc[
        n_test:n_test + n_val
    ]

    train = group.iloc[
        n_test + n_val:
    ]

    train_parts.append(train)

    if len(val):
        val_parts.append(val)

    if len(test):
        test_parts.append(test)


train_df = pd.concat(
    train_parts,
    ignore_index=True
)

val_df = pd.concat(
    val_parts,
    ignore_index=True
)

test_df = pd.concat(
    test_parts,
    ignore_index=True
)

# Shuffle final datasets
train_df = train_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

val_df = val_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

test_df = test_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


# Save
TRAIN_PATH = os.path.join(
    WORK_DIR,
    "train_90class.csv"
)

VAL_PATH = os.path.join(
    WORK_DIR,
    "val_90class.csv"
)

TEST_PATH = os.path.join(
    WORK_DIR,
    "test_90class.csv"
)

train_df.to_csv(TRAIN_PATH, index=False)
val_df.to_csv(VAL_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)


# Verification
print("===== SPLIT RESULTS =====")

print("\nTrain images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nTotal:",
      len(train_df) +
      len(val_df) +
      len(test_df))

print("\nClasses in train:",
      train_df["label"].nunique())

print("Classes in validation:",
      val_df["label"].nunique())

print("Classes in test:",
      test_df["label"].nunique())


# Check image leakage using md5
train_ids = set(train_df["md5hash"])
val_ids = set(val_df["md5hash"])
test_ids = set(test_df["md5hash"])

print("\n===== LEAKAGE CHECK =====")

print(
    "Train ↔ Validation:",
    len(train_ids & val_ids)
)

print(
    "Train ↔ Test:",
    len(train_ids & test_ids)
)

print(
    "Validation ↔ Test:",
    len(val_ids & test_ids)
)


print("\nSaved:")
print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

===== SPLIT RESULTS =====

Train images: 2721
Validation images: 581
Test images: 585

Total: 3887

Classes in train: 90
Classes in validation: 80
Classes in test: 84

===== LEAKAGE CHECK =====
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

Saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/train_90class.csv
/content/drive/MyDrive/DermaWise/Fitzpatrick114/val_90class.csv
/content/drive/MyDrive/DermaWise/Fitzpatrick114/test_90class.csv


In [ ]:
import os
import pandas as pd
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TRAIN_PATH = os.path.join(WORK_DIR, "train_90class.csv")
VAL_PATH   = os.path.join(WORK_DIR, "val_90class.csv")
TEST_PATH  = os.path.join(WORK_DIR, "test_90class.csv")

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

# --------------------------------------------------
# 1. Create one fixed 90-class vocabulary
# --------------------------------------------------

CLASS_NAMES = sorted(train_df["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)

class_to_index = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

index_to_class = {
    i: name for name, i in class_to_index.items()
}

print("Number of classes:", NUM_CLASSES)

print("\nFirst 10 class mappings:")
for i in range(min(10, NUM_CLASSES)):
    print(i, "->", index_to_class[i])


# --------------------------------------------------
# 2. Add numeric labels
# --------------------------------------------------

train_df["class_id"] = train_df["label"].map(class_to_index)
val_df["class_id"]   = val_df["label"].map(class_to_index)
test_df["class_id"]  = test_df["label"].map(class_to_index)

assert train_df["class_id"].notna().all()
assert val_df["class_id"].notna().all()
assert test_df["class_id"].notna().all()


# --------------------------------------------------
# 3. Dataset settings
# --------------------------------------------------

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    # Keep 0-255 range because Keras EfficientNet
    # contains its own input rescaling.
    image = tf.cast(image, tf.float32)

    return image, label


def create_dataset(dataframe, training=False):

    paths = dataframe["filepath"].astype(str).values
    labels = dataframe["class_id"].astype("int32").values

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = create_dataset(train_df, training=True)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)


# --------------------------------------------------
# 4. Test one batch
# --------------------------------------------------

images, labels = next(iter(train_ds))

print("\n===== DATASET CHECK =====")
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("Pixel minimum:", float(tf.reduce_min(images)))
print("Pixel maximum:", float(tf.reduce_max(images)))

print("\nTrain batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())


# --------------------------------------------------
# 5. Save class names
# --------------------------------------------------

CLASS_FILE = os.path.join(
    WORK_DIR,
    "fitzpatrick90_class_names.txt"
)

with open(CLASS_FILE, "w") as f:
    for name in CLASS_NAMES:
        f.write(name + "\n")

print("\nClass mapping saved:")
print(CLASS_FILE)

Number of classes: 90

First 10 class mappings:
0 -> acanthosis nigricans
1 -> acne
2 -> acrodermatitis enteropathica
3 -> actinic keratosis
4 -> allergic contact dermatitis
5 -> aplasia cutis
6 -> basal cell carcinoma
7 -> behcets disease
8 -> calcinosis cutis
9 -> cheilitis

===== DATASET CHECK =====
Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Pixel minimum: 0.0
Pixel maximum: 255.0

Train batches: 86
Validation batches: 19
Test batches: 19

Class mapping saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick90_class_names.txt


In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# --------------------------------------------------
# 1. Calculate class weights
# --------------------------------------------------

y_train = train_df["class_id"].astype(int).values

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_train
)

class_weights = {
    i: float(weight)
    for i, weight in enumerate(weights)
}

print("===== CLASS WEIGHTS =====")
print("Number of weighted classes:", len(class_weights))

print("\nFirst 10:")
for i in range(10):
    print(
        f"{i:2} | {index_to_class[i]:40} | "
        f"{class_weights[i]:.3f}"
    )


# --------------------------------------------------
# 2. Data augmentation
# --------------------------------------------------

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")


# --------------------------------------------------
# 3. EfficientNetB0 backbone
# --------------------------------------------------

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Stage 1: freeze backbone
base_model.trainable = False


# --------------------------------------------------
# 4. Build 90-class classifier
# --------------------------------------------------

inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="image"
)

x = augmentation(inputs)

x = base_model(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.Dropout(
    0.30,
    name="dropout"
)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="condition_predictions"
)(x)

model_90 = tf.keras.Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick90"
)


# --------------------------------------------------
# 5. Compile
# --------------------------------------------------

model_90.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_accuracy"
        )
    ]
)


# --------------------------------------------------
# 6. Verify
# --------------------------------------------------

print("\n===== MODEL CHECK =====")
print("Model:", model_90.name)
print("Classes:", NUM_CLASSES)
print("Output units:", model_90.output_shape[-1])
print("Backbone trainable:", base_model.trainable)
print("Parameters:", f"{model_90.count_params():,}")

assert NUM_CLASSES == 90
assert model_90.output_shape[-1] == 90

print("\n✅ 90-class model constructed successfully.")

===== CLASS WEIGHTS =====
Number of weighted classes: 90

First 10:
 0 | acanthosis nigricans                     | 1.890
 1 | acne                                     | 1.890
 2 | acrodermatitis enteropathica             | 1.120
 3 | actinic keratosis                        | 1.890
 4 | allergic contact dermatitis              | 0.975
 5 | aplasia cutis                            | 3.779
 6 | basal cell carcinoma                     | 0.169
 7 | behcets disease                          | 2.160
 8 | calcinosis cutis                         | 2.748
 9 | cheilitis                                | 2.160
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

===== MODEL CHECK =====
Model: DermaWise_Fitzpatrick90
Classes: 90
Output units: 90
Backbone trainable: False
Parameters: 4,164,861

✅ 90-class model constructed successfully.


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_stage1_best.keras"
)

callbacks = [
    # Save only the best validation-loss model
    tf.keras.callbacks.ModelCheckpoint(
        STAGE1_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    # Stop if validation performance stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when learning stalls
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Starting Stage 1 training...")
print("Classes:", NUM_CLASSES)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Saving best model to:")
print(STAGE1_PATH)

history_stage1 = model_90.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Stage 1 training finished.")
print("Best model saved at:")
print(STAGE1_PATH)

Starting Stage 1 training...
Classes: 90
Training images: 2721
Validation images: 581
Saving best model to:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick90_stage1_best.keras
Epoch 1/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.0529 - loss: 4.6685 - top3_accuracy: 0.1226 - top5_accuracy: 0.1714
Epoch 1: val_loss improved from None to 3.45573, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick90_stage1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick90_stage1_best.keras
86/86 ━━━━━━━━━━━━━━━━━━━━ 324s 4s/step - accuracy: 0.0996 - loss: 4.3797 - top3_accuracy: 0.2051 - top5_accuracy: 0.2723 - val_accuracy: 0.2392 - val_loss: 3.4557 - val_top3_accuracy: 0.3976 - val_top5_accuracy: 0.4991 - learning_rate: 0.0010
Epoch 2/20
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.2586 - loss: 3.3419 - top3_accuracy: 0.4369 - top5_accuracy: 0.5229
Epoch 2: val_los

In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_stage1_best.keras"
)

print("Loading BEST Stage 1 model...")

best_stage1 = tf.keras.models.load_model(STAGE1_PATH)

print("✅ Model loaded")
print("Output classes:", best_stage1.output_shape[-1])

print("\nEvaluating on unseen TEST images...")

results = best_stage1.evaluate(
    test_ds,
    verbose=1,
    return_dict=True
)

print("\n===== STAGE 1 TEST RESULTS =====")

for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

print("\nTop-1 Accuracy:",
      f"{results['accuracy'] * 100:.2f}%")

print("Top-3 Accuracy:",
      f"{results['top3_accuracy'] * 100:.2f}%")

print("Top-5 Accuracy:",
      f"{results['top5_accuracy'] * 100:.2f}%")

Loading BEST Stage 1 model...
✅ Model loaded
Output classes: 90

Evaluating on unseen TEST images...
19/19 ━━━━━━━━━━━━━━━━━━━━ 49s 2s/step - accuracy: 0.5060 - loss: 1.8988 - top3_accuracy: 0.7316 - top5_accuracy: 0.8154

===== STAGE 1 TEST RESULTS =====
accuracy: 0.5060
loss: 1.8988
top3_accuracy: 0.7316
top5_accuracy: 0.8154

Top-1 Accuracy: 50.60%
Top-3 Accuracy: 73.16%
Top-5 Accuracy: 81.54%


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_stage1_best.keras"
)

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_finetuned_best.keras"
)

# Load best Stage 1 model
model_90 = tf.keras.models.load_model(STAGE1_PATH)

# Find EfficientNet backbone
base_model = model_90.get_layer("efficientnetb0")

# Unfreeze backbone first
base_model.trainable = True

# Freeze everything except last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_layers = sum(
    1 for layer in base_model.layers
    if layer.trainable
)

print("Trainable EfficientNet layers:", trainable_layers)

# IMPORTANT: much smaller LR for fine-tuning
model_90.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_accuracy"
        )
    ]
)

callbacks_ft = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE2_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("\nStarting Stage 2 fine-tuning...")

history_stage2 = model_90.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks_ft,
    verbose=1
)

print("\n✅ Stage 2 fine-tuning finished.")
print("Best model saved:")
print(STAGE2_PATH)

Trainable EfficientNet layers: 23

Starting Stage 2 fine-tuning...
Epoch 1/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7029 - loss: 0.6536 - top3_accuracy: 0.8565 - top5_accuracy: 0.9116
Epoch 1: val_loss improved from None to 1.99944, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick90_finetuned_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick90_finetuned_best.keras
86/86 ━━━━━━━━━━━━━━━━━━━━ 367s 4s/step - accuracy: 0.7078 - loss: 0.6340 - top3_accuracy: 0.8684 - top5_accuracy: 0.9210 - val_accuracy: 0.4940 - val_loss: 1.9994 - val_top3_accuracy: 0.6713 - val_top5_accuracy: 0.7762 - learning_rate: 1.0000e-05
Epoch 2/10
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7062 - loss: 0.6092 - top3_accuracy: 0.8735 - top5_accuracy: 0.9190
Epoch 2: val_loss did not improve from 1.99944
86/86 ━━━━━━━━━━━━━━━━━━━━ 308s 4s/step - accuracy: 0.7078 - loss: 0.5863 - top3_accuracy:

In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_stage1_best.keras"
)

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_finetuned_best.keras"
)

print("Loading Stage 1...")
stage1 = tf.keras.models.load_model(STAGE1_PATH)

print("Loading Stage 2...")
stage2 = tf.keras.models.load_model(STAGE2_PATH)

print("\nEvaluating Stage 1...")
r1 = stage1.evaluate(
    test_ds,
    verbose=0,
    return_dict=True
)

print("Evaluating Stage 2...")
r2 = stage2.evaluate(
    test_ds,
    verbose=0,
    return_dict=True
)

print("\n========== TEST COMPARISON ==========")

print("\nSTAGE 1")
print(f"Loss:          {r1['loss']:.4f}")
print(f"Top-1:         {r1['accuracy'] * 100:.2f}%")
print(f"Top-3:         {r1['top3_accuracy'] * 100:.2f}%")
print(f"Top-5:         {r1['top5_accuracy'] * 100:.2f}%")

print("\nSTAGE 2")
print(f"Loss:          {r2['loss']:.4f}")
print(f"Top-1:         {r2['accuracy'] * 100:.2f}%")
print(f"Top-3:         {r2['top3_accuracy'] * 100:.2f}%")
print(f"Top-5:         {r2['top5_accuracy'] * 100:.2f}%")

print("\n========== CHANGE ==========")

print(
    f"Top-1 change: "
    f"{(r2['accuracy'] - r1['accuracy']) * 100:+.2f} percentage points"
)

print(
    f"Top-3 change: "
    f"{(r2['top3_accuracy'] - r1['top3_accuracy']) * 100:+.2f} percentage points"
)

print(
    f"Top-5 change: "
    f"{(r2['top5_accuracy'] - r1['top5_accuracy']) * 100:+.2f} percentage points"
)

Loading Stage 1...
Loading Stage 2...

Evaluating Stage 1...
Evaluating Stage 2...

========== TEST COMPARISON ==========

STAGE 1
Loss:          1.8988
Top-1:         50.60%
Top-3:         73.16%
Top-5:         81.54%

STAGE 2
Loss:          1.8580
Top-1:         53.68%
Top-3:         72.82%
Top-5:         82.05%

========== CHANGE ==========
Top-1 change: +3.08 percentage points
Top-3 change: -0.34 percentage points
Top-5 change: +0.51 percentage points


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score
)

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick90_finetuned_best.keras"
)

# Load best fine-tuned model
model_final = tf.keras.models.load_model(STAGE2_PATH)

print("Generating predictions on test set...")

probabilities = model_final.predict(
    test_ds,
    verbose=1
)

y_pred = np.argmax(probabilities, axis=1)

# test_ds was created without shuffle
y_true = test_df["class_id"].astype(int).values

assert len(y_true) == len(y_pred)

# Overall metrics
accuracy = np.mean(y_true == y_pred)

macro_precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

print("\n========== FINAL 90-CLASS TEST METRICS ==========")

print(f"Top-1 Accuracy : {accuracy * 100:.2f}%")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall   : {macro_recall:.4f}")
print(f"Macro F1       : {macro_f1:.4f}")
print(f"Weighted F1    : {weighted_f1:.4f}")


# Detailed report only for classes actually present in test set
present_ids = sorted(np.unique(y_true))

present_names = [
    index_to_class[i]
    for i in present_ids
]

report = classification_report(
    y_true,
    y_pred,
    labels=present_ids,
    target_names=present_names,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()

REPORT_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick90_test_classification_report.csv"
)

report_df.to_csv(REPORT_PATH)

print("\nClasses represented in test:", len(present_ids))
print("Report saved to:")
print(REPORT_PATH)

print("\nLowest-performing test classes:")
class_rows = report_df.loc[present_names]
display(
    class_rows.sort_values("f1-score").head(15)
)

print("\nHighest-performing test classes:")
display(
    class_rows.sort_values(
        "f1-score",
        ascending=False
    ).head(15)
)

Generating predictions on test set...
19/19 ━━━━━━━━━━━━━━━━━━━━ 50s 2s/step

========== FINAL 90-CLASS TEST METRICS ==========
Top-1 Accuracy : 53.68%
Macro Precision: 0.5343
Macro Recall   : 0.5326
Macro F1       : 0.4958
Weighted F1    : 0.5380

Classes represented in test: 84
Report saved to:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick90_test_classification_report.csv

Lowest-performing test classes:


,precision,recall,f1-score,support
drug induced pigmentary changes,0.000000,0.000000,0.000000,1.0
dyshidrotic eczema,0.000000,0.000000,0.000000,1.0
lymphangioma,0.000000,0.000000,0.000000,2.0
pilomatricoma,0.000000,0.000000,0.000000,1.0
mucous cyst,0.000000,0.000000,0.000000,2.0
lichen simplex,0.000000,0.000000,0.000000,2.0
lupus subacute,0.000000,0.000000,0.000000,1.0
tick bite,0.000000,0.000000,0.000000,1.0
sarcoidosis,0.066667,0.200000,0.100000,5.0
fixed eruptions,0.250000,0.250000,0.250000,4.0



Highest-performing test classes:


,precision,recall,f1-score,support
rosacea,1.000000,1.000000,1.000000,1.0
urticaria pigmentosa,1.000000,1.000000,1.000000,1.0
xeroderma pigmentosum,1.000000,1.000000,1.000000,2.0
striae,1.000000,1.000000,1.000000,1.0
juvenile xanthogranuloma,0.833333,1.000000,0.909091,5.0
neurofibromatosis,0.916667,0.846154,0.880000,13.0
pediculosis lids,0.733333,0.916667,0.814815,12.0
neurotic excoriations,1.000000,0.666667,0.800000,3.0
tungiasis,0.736842,0.875000,0.800000,16.0
scleromyxedema,1.000000,0.666667,0.800000,6.0


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

DATA_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick17k_usable_90class.csv"
)

df = pd.read_csv(DATA_PATH)

# Count images per condition
counts = df["label"].value_counts()

# Keep classes with at least 20 real images
supported_classes = counts[counts >= 20].index.tolist()

df_supported = df[
    df["label"].isin(supported_classes)
].copy()

print("===== SUPPORTED DATASET =====")
print("Conditions:", df_supported["label"].nunique())
print("Images:", len(df_supported))

print("\nSmallest class:",
      df_supported["label"].value_counts().min())

print("Largest class:",
      df_supported["label"].value_counts().max())

print("\nClass counts:")
print(
    df_supported["label"]
    .value_counts()
    .sort_values(ascending=False)
)

SUPPORTED_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick_supported_53class.csv"
)

df_supported.to_csv(
    SUPPORTED_PATH,
    index=False
)

print("\nSaved:")
print(SUPPORTED_PATH)

===== SUPPORTED DATASET =====
Conditions: 53
Images: 3557

Smallest class: 20
Largest class: 257

Class counts:
label
basal cell carcinoma                        257
lichen planus                               230
squamous cell carcinoma                     212
pityriasis rubra pilaris                    175
scabies                                     138
porokeratosis actinic                       127
nematode infection                          126
lupus erythematosus                         108
tungiasis                                   107
psoriasis                                   104
folliculitis                                102
hailey hailey disease                        97
pityriasis rosea                             95
neutrophilic dermatoses                      90
neurofibromatosis                            86
melanoma                                     83
mycosis fungoides                            83
pediculosis lids                             77
dariers disease   

In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

SUPPORTED_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick_supported_53class.csv"
)

df = pd.read_csv(SUPPORTED_PATH)

train_parts = []
val_parts = []
test_parts = []

# Split each class independently
for label, group in df.groupby("label"):

    group = group.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    n = len(group)

    # Approx. 70 / 15 / 15
    n_test = max(3, round(n * 0.15))
    n_val  = max(3, round(n * 0.15))

    test_part = group.iloc[:n_test]
    val_part = group.iloc[n_test:n_test + n_val]
    train_part = group.iloc[n_test + n_val:]

    train_parts.append(train_part)
    val_parts.append(val_part)
    test_parts.append(test_part)

train53_df = pd.concat(
    train_parts,
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

val53_df = pd.concat(
    val_parts,
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

test53_df = pd.concat(
    test_parts,
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)


# Save
TRAIN53_PATH = os.path.join(
    WORK_DIR,
    "train_53class.csv"
)

VAL53_PATH = os.path.join(
    WORK_DIR,
    "val_53class.csv"
)

TEST53_PATH = os.path.join(
    WORK_DIR,
    "test_53class.csv"
)

train53_df.to_csv(TRAIN53_PATH, index=False)
val53_df.to_csv(VAL53_PATH, index=False)
test53_df.to_csv(TEST53_PATH, index=False)


# Verification
print("===== 53-CLASS SPLIT =====")

print("Train images:", len(train53_df))
print("Validation images:", len(val53_df))
print("Test images:", len(test53_df))

print(
    "Total:",
    len(train53_df) +
    len(val53_df) +
    len(test53_df)
)

print("\nTrain classes:",
      train53_df["label"].nunique())

print("Validation classes:",
      val53_df["label"].nunique())

print("Test classes:",
      test53_df["label"].nunique())


# Leakage check
train_ids = set(train53_df["md5hash"])
val_ids = set(val53_df["md5hash"])
test_ids = set(test53_df["md5hash"])

print("\n===== LEAKAGE CHECK =====")

print(
    "Train ↔ Validation:",
    len(train_ids & val_ids)
)

print(
    "Train ↔ Test:",
    len(train_ids & test_ids)
)

print(
    "Validation ↔ Test:",
    len(val_ids & test_ids)
)


# Minimum samples per class in each split
print("\n===== MINIMUM CLASS SUPPORT =====")

print(
    "Train minimum:",
    train53_df["label"].value_counts().min()
)

print(
    "Validation minimum:",
    val53_df["label"].value_counts().min()
)

print(
    "Test minimum:",
    test53_df["label"].value_counts().min()
)

===== 53-CLASS SPLIT =====
Train images: 2493
Validation images: 532
Test images: 532
Total: 3557

Train classes: 53
Validation classes: 53
Test classes: 53

===== LEAKAGE CHECK =====
Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0

===== MINIMUM CLASS SUPPORT =====
Train minimum: 14
Validation minimum: 3
Test minimum: 3


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# =========================================================
# 1. CLASS MAPPING
# =========================================================

CLASS_NAMES_53 = sorted(
    train53_df["label"].unique().tolist()
)

NUM_CLASSES_53 = len(CLASS_NAMES_53)

class_to_index_53 = {
    name: i
    for i, name in enumerate(CLASS_NAMES_53)
}

index_to_class_53 = {
    i: name
    for name, i in class_to_index_53.items()
}

train53_df["class_id"] = train53_df["label"].map(class_to_index_53)
val53_df["class_id"] = val53_df["label"].map(class_to_index_53)
test53_df["class_id"] = test53_df["label"].map(class_to_index_53)

print("Classes:", NUM_CLASSES_53)


# =========================================================
# 2. TENSORFLOW DATASETS
# =========================================================

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image_53(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def make_dataset_53(dataframe, training=False):

    paths = dataframe["filepath"].astype(str).values

    labels = dataframe[
        "class_id"
    ].astype("int32").values

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            len(dataframe),
            seed=42,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image_53,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds


train53_ds = make_dataset_53(
    train53_df,
    training=True
)

val53_ds = make_dataset_53(val53_df)

test53_ds = make_dataset_53(test53_df)


# =========================================================
# 3. CLASS WEIGHTS
# =========================================================

y_train_53 = train53_df[
    "class_id"
].astype(int).values

weights_53 = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES_53),
    y=y_train_53
)

class_weights_53 = {
    i: float(w)
    for i, w in enumerate(weights_53)
}

print(
    "Class weights created:",
    len(class_weights_53)
)


# =========================================================
# 4. AUGMENTATION
# =========================================================

augmentation_53 = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
], name="augmentation")


# =========================================================
# 5. EFFICIENTNETB0
# =========================================================

base53 = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base53.trainable = False


# =========================================================
# 6. CLASSIFIER
# =========================================================

inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="image"
)

x = augmentation_53(inputs)

x = base53(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.30)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES_53,
    activation="softmax",
    name="condition_predictions"
)(x)

model_53 = tf.keras.Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick53"
)


# =========================================================
# 7. COMPILE
# =========================================================

model_53.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3_accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5_accuracy"
        )
    ]
)


# =========================================================
# 8. VERIFY
# =========================================================

images, labels = next(iter(train53_ds))

print("\n===== 53-CLASS MODEL CHECK =====")

print("Image batch:", images.shape)
print("Label batch:", labels.shape)

print("Output classes:", model_53.output_shape[-1])

print(
    "Parameters:",
    f"{model_53.count_params():,}"
)

assert NUM_CLASSES_53 == 53
assert model_53.output_shape[-1] == 53

print("\n✅ 53-class model ready.")


# =========================================================
# 9. SAVE CLASS NAMES
# =========================================================

CLASS_FILE_53 = os.path.join(
    WORK_DIR,
    "fitzpatrick53_class_names.txt"
)

with open(CLASS_FILE_53, "w") as f:
    for name in CLASS_NAMES_53:
        f.write(name + "\n")

print("\nClass names saved:")
print(CLASS_FILE_53)

Classes: 53
Class weights created: 53

===== 53-CLASS MODEL CHECK =====
Image batch: (32, 224, 224, 3)
Label batch: (32,)
Output classes: 53
Parameters: 4,117,464

✅ 53-class model ready.

Class names saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick53_class_names.txt


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_53_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick53_stage1_best.keras"
)

callbacks_53 = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE1_53_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("===== STARTING 53-CLASS STAGE 1 =====")
print("Training images:", len(train53_df))
print("Validation images:", len(val53_df))
print("Classes:", NUM_CLASSES_53)
print("Maximum epochs: 15")

history_53_stage1 = model_53.fit(
    train53_ds,
    validation_data=val53_ds,
    epochs=15,
    class_weight=class_weights_53,
    callbacks=callbacks_53,
    verbose=1
)

print("\n✅ 53-class Stage 1 complete")
print("Best model:")
print(STAGE1_53_PATH)

===== STARTING 53-CLASS STAGE 1 =====
Training images: 2493
Validation images: 532
Classes: 53
Maximum epochs: 15
Epoch 1/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.0750 - loss: 3.9062 - top3_accuracy: 0.1717 - top5_accuracy: 0.2306
Epoch 1: val_loss improved from None to 2.98925, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick53_stage1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick53_stage1_best.keras
78/78 ━━━━━━━━━━━━━━━━━━━━ 269s 3s/step - accuracy: 0.1308 - loss: 3.6799 - top3_accuracy: 0.2623 - top5_accuracy: 0.3426 - val_accuracy: 0.2932 - val_loss: 2.9893 - val_top3_accuracy: 0.4699 - val_top5_accuracy: 0.6034 - learning_rate: 0.0010
Epoch 2/15
78/78 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3005 - loss: 2.8951 - top3_accuracy: 0.4885 - top5_accuracy: 0.5884
Epoch 2: val_loss improved from 2.98925 to 2.64748, saving model to /content/drive/MyDrive/DermaWise

In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_53_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick53_stage1_best.keras"
)

print("Loading best 53-class Stage 1 model...")

best_53_stage1 = tf.keras.models.load_model(
    STAGE1_53_PATH
)

print("✅ Model loaded")
print("Output classes:", best_53_stage1.output_shape[-1])

print("\nEvaluating on 532 unseen test images...")

results_53 = best_53_stage1.evaluate(
    test53_ds,
    verbose=1,
    return_dict=True
)

print("\n========== 53-CLASS TEST RESULTS ==========")

print(f"Loss:  {results_53['loss']:.4f}")
print(f"Top-1: {results_53['accuracy'] * 100:.2f}%")
print(f"Top-3: {results_53['top3_accuracy'] * 100:.2f}%")
print(f"Top-5: {results_53['top5_accuracy'] * 100:.2f}%")

Loading best 53-class Stage 1 model...
✅ Model loaded
Output classes: 53

Evaluating on 532 unseen test images...
17/17 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.5263 - loss: 1.7534 - top3_accuracy: 0.7256 - top5_accuracy: 0.8289

========== 53-CLASS TEST RESULTS ==========
Loss:  1.7534
Top-1: 52.63%
Top-3: 72.56%
Top-5: 82.89%


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
os.makedirs(SAVE_DIR, exist_ok=True)

ZIP_PATH = os.path.join(
    SAVE_DIR,
    "Fitzpatrick17k_CategorizedAbbrvs.zip"
)

!wget -c \
"https://zenodo.org/records/11101338/files/Fitzpatrick17k_CategorizedAbbrvs.zip?download=1" \
-O "$ZIP_PATH"

--2026-09-21 17:02:16--  https://zenodo.org/records/11101338/files/Fitzpatrick17k_CategorizedAbbrvs.zip?download=1
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 188.185.43.153, 188.184.98.114, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1422904980 (1.3G) [application/octet-stream]
Saving to: ‘/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_CategorizedAbbrvs.zip’

/content/drive/MyDr 100%[===================>]   1.32G  4.27MB/s    in 9m 47s  

2026-09-21 17:12:04 (2.31 MB/s) - ‘/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_CategorizedAbbrvs.zip’ saved [1422904980/1422904980]



In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_CategorizedAbbrvs.zip"

size_gb = os.path.getsize(ZIP_PATH) / (1024**3)

print(f"Downloaded ZIP size: {size_gb:.2f} GB")
print("Exists:", os.path.exists(ZIP_PATH))

Downloaded ZIP size: 1.33 GB
Exists: True


In [ ]:
import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_CategorizedAbbrvs.zip"

size_gb = os.path.getsize(ZIP_PATH) / (1024**3)

print(f"ZIP size: {size_gb:.2f} GB")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

print("ZIP opened successfully ✅")
print("Total files inside:", len(files))

print("\nFirst 10 files:")
for f in files[:10]:
    print(f)

ZIP size: 1.33 GB
ZIP opened successfully ✅
Total files inside: 16692

First 10 files:
Categorized_AbbrvName/
Categorized_AbbrvName/ac-au-bu-di-ge_67/
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_0_7632dbe0.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_13_0d8a8465.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_15_11a121c6.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_18_b2b7c4f8.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_19_d56176c7.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_21_2b2e5e50.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_26_4d41674b.jpg
Categorized_AbbrvName/ac-au-bu-di-ge_67/ac-au-bu-di-ge_f1_27_ec8e57d5.jpg


In [ ]:
import zipfile
import os

ZIP_PATH = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_CategorizedAbbrvs.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    files = z.namelist()

# Actual image files
image_files = [
    f for f in files
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Class folders immediately below Categorized_AbbrvName/
class_folders = set()

for f in image_files:
    parts = f.split("/")
    if len(parts) >= 3 and parts[0] == "Categorized_AbbrvName":
        class_folders.add(parts[1])

print("===== FITZPATRICK17K-C ARCHIVE CHECK =====")
print("Actual images:", len(image_files))
print("Class folders:", len(class_folders))

print("\nFirst 15 class folders:")
for name in sorted(class_folders)[:15]:
    print(name)

===== FITZPATRICK17K-C ARCHIVE CHECK =====
Actual images: 16577
Class folders: 114

First 15 class folders:
ac-au-bu-di-ge_67
ac-en_92
ac-ke_175
ac-ni_92
ac-vu_335
ac_183
al-co-de_430
ap-cu_72
ba-ce-ca-mo_62
ba-ce-ca_468
be-di_63
be-ne_63
ca-cu_80
ch_106
co-ne_68


In [ ]:
import os
import zipfile

ZIP_PATH = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_CategorizedAbbrvs.zip"

EXTRACT_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C"

os.makedirs(EXTRACT_DIR, exist_ok=True)

print("Extracting Fitzpatrick17k-C...")
print("This may take several minutes.\n")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("✅ Extraction complete")
print("Saved to:", EXTRACT_DIR)

Extracting Fitzpatrick17k-C...
This may take several minutes.

✅ Extraction complete
Saved to: /content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C


In [ ]:
import os

IMAGE_ROOT = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C/Categorized_AbbrvName"

image_count = 0
folder_count = 0

for folder in os.listdir(IMAGE_ROOT):
    folder_path = os.path.join(IMAGE_ROOT, folder)

    if os.path.isdir(folder_path):
        folder_count += 1

        image_count += len([
            f for f in os.listdir(folder_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

print("===== EXTRACTION CHECK =====")
print("Class folders:", folder_count)
print("Images:", image_count)

assert folder_count == 114, f"Expected 114 classes, found {folder_count}"
assert image_count == 16577, f"Expected 16577 images, found {image_count}"

print("\n✅ Complete 114-condition dataset ready!")

===== EXTRACTION CHECK =====
Class folders: 114
Images: 16577

✅ Complete 114-condition dataset ready!


In [ ]:
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
os.makedirs(WORK_DIR, exist_ok=True)

# Corrected Fitzpatrick17k-C metadata
!wget -q --show-progress \
"https://zenodo.org/records/11101338/files/Fitzpatrick17k-C.csv?download=1" \
-O "$WORK_DIR/Fitzpatrick17k-C.csv"

# Diagnosis-name mapping
!wget -q --show-progress \
"https://zenodo.org/records/11101338/files/Fitzpatrick17k_DiagnosisMapping.xlsx?download=1" \
-O "$WORK_DIR/Fitzpatrick17k_DiagnosisMapping.xlsx"

print("\n===== DOWNLOAD CHECK =====")

for filename in [
    "Fitzpatrick17k-C.csv",
    "Fitzpatrick17k_DiagnosisMapping.xlsx"
]:
    path = os.path.join(WORK_DIR, filename)
    print(
        filename,
        "→",
        os.path.exists(path),
        f"({os.path.getsize(path)/1024:.1f} KB)"
        if os.path.exists(path) else ""
    )

/content/drive/MyDr 100%[===================>]   4.10M   764KB/s    in 5.9s    
/content/drive/MyDr 100%[===================>]  24.04K   146KB/s    in 0.2s    

===== DOWNLOAD CHECK =====
Fitzpatrick17k-C.csv → True (4203.4 KB)
Fitzpatrick17k_DiagnosisMapping.xlsx → True (24.0 KB)


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

CSV_PATH = os.path.join(WORK_DIR, "Fitzpatrick17k-C.csv")
MAP_PATH = os.path.join(WORK_DIR, "Fitzpatrick17k_DiagnosisMapping.xlsx")

# Load files
meta = pd.read_csv(CSV_PATH)
mapping = pd.read_excel(MAP_PATH)

print("===== FITZPATRICK17K-C METADATA =====")
print("Shape:", meta.shape)

print("\nColumns:")
print(meta.columns.tolist())

print("\nFirst 3 rows:")
display(meta.head(3))

print("\n===== DIAGNOSIS MAPPING =====")
print("Shape:", mapping.shape)

print("\nColumns:")
print(mapping.columns.tolist())

print("\nFirst 10 rows:")
display(mapping.head(10))

print("\n===== UNIQUE VALUES PER METADATA COLUMN =====")
for col in meta.columns:
    n = meta[col].nunique(dropna=False)
    print(f"{col}: {n}")

===== FITZPATRICK17K-C METADATA =====
Shape: (11394, 15)

Columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name']

First 3 rows:


,filepath,filename,diag,fst,partition,md5hash,fitzpatrick,label,nine_partition_label,three_partition_label,qc,url,url_alphanum,orig_img_name,new_img_name
0,ne-de_361/ne-de_f2_61_3a96728f.jpg,ne-de_f2_61_3a96728f,ne-de,2,train,3a96728fb161fc25b3597d3996dc4e4a,2,neutrophilic dermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicppyode...,3a96728fb161fc25b3597d3996dc4e4a.jpg,ne-de_f2_61_3a96728f.jpg
1,lu-er_410/lu-er_f1_89_07be6be3.jpg,lu-er_f1_89_07be6be3,lu-er,1,train,07be6be3cbebd04ff9bfcab50b75378b,1,lupus erythematosus,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical...,httpwwwdermaamincomsiteimagesclinicalpicssubac...,07be6be3cbebd04ff9bfcab50b75378b.jpg,lu-er_f1_89_07be6be3.jpg
2,me_261/me_f0_194_72bcd875.jpg,me_f0_194_72bcd875,me,0,train,72bcd875e7a573b4139883059e69328d,-1,melanoma,malignant melanoma,malignant,NaN,http://atlasdermatologico.com.br/img?imageId=4264,httpwwwatlasdermatologicocombrimgimageId4264.jpg,72bcd875e7a573b4139883059e69328d.jpg,me_f0_194_72bcd875.jpg



===== DIAGNOSIS MAPPING =====
Shape: (114, 12)

Columns:
['Abbreviated \nDiagnosis Label', 'Diagnosis Label', 'ICD-11 Name', 'ICD-11 Code', 'ICD-11 URI', 'ICD-11 Notes', 'ICD-11 Coding Tool', 'ICD-11 Classification of \nDermatological Diseases', 'SNOMED-CT Label', 'SCTID', 'Link', 'SNOMED-CT Notes']

First 10 rows:


,Abbreviated \nDiagnosis Label,Diagnosis Label,ICD-11 Name,ICD-11 Code,ICD-11 URI,ICD-11 Notes,ICD-11 Coding Tool,ICD-11 Classification of \nDermatological Diseases,SNOMED-CT Label,SCTID,Link,SNOMED-CT Notes
0,ac-ni,acanthosis nigricans,ED51.0,Acanthosis nigricans,http://id.who.int/icd/entity/71488193,NaN,NaN,NaN,Acanthosis nigricans (disorder),402599005.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
1,ac,acne,ED80,Acne,http://id.who.int/icd/entity/1892393023,NaN,NaN,NaN,Acne (disorder),11381005.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
2,ac-vu,acne vulgaris,ED80.Z,"Acne, unspecified",NaN,NaN,NaN,NaN,Acne vulgaris (disorder),88616000.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
3,ac-au-bu-di-ge,acquired autoimmune bullous diseaseherpes gest...,NaN,NaN,NaN,Second half matches to JA65.10 Herpes gestatio...,—,—,NaN,NaN,NaN,First half matches to Anti-p200 pemphigoid and...
4,ac-en,acrodermatitis enteropathica,5C64.20,Acrodermatitis enteropathica,http://id.who.int/icd/entity/1813939482,NaN,NaN,NaN,Hereditary acrodermatitis enteropathica (disor...,37702000.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
5,ac-ke,actinic keratosis,EK90.0,Actinic keratosis,http://id.who.int/icd/entity/1803982621,NaN,NaN,NaN,Actinic keratosis (disorder),201101007.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
6,al-co-de,allergic contact dermatitis,EK00,Allergic contact dermatitis,http://id.who.int/icd/entity/2020673786,NaN,NaN,NaN,Allergic contact dermatitis (disorder),238575004.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
7,ap-cu,aplasia cutis,LC60,Aplasia cutis congenita,http://id.who.int/icd/entity/350175828,NaN,NaN,NaN,Aplasia cutis congenita (disorder),35484002.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
8,ba-ce-ca,basal cell carcinoma,NaN,NaN,NaN,Many subtypes.,—,—,Basal cell carcinoma (morphologic abnormality),1338007.0,https://browser.ihtsdotools.org/?perspective=f...,NaN
9,ba-ce-ca-mo,basal cell carcinoma morpheiform,NaN,NaN,NaN,Is this the same as XH67Y4 Basal cell carcinom...,—,—,NaN,NaN,NaN,"Is it same as Basal cell carcinoma, morpheic?"



===== UNIQUE VALUES PER METADATA COLUMN =====
filepath: 11394
filename: 11394
diag: 114
fst: 7
partition: 3
md5hash: 11394
fitzpatrick: 7
label: 114
nine_partition_label: 9
three_partition_label: 3
qc: 6
url: 11360
url_alphanum: 11394
orig_img_name: 11394
new_img_name: 11394


In [ ]:
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
CSV_PATH = os.path.join(WORK_DIR, "Fitzpatrick17k-C.csv")

meta = pd.read_csv(CSV_PATH)

print("===== OFFICIAL PARTITIONS =====")
print(meta["partition"].value_counts(dropna=False))

print("\n===== CLASSES PER PARTITION =====")
for split in sorted(meta["partition"].dropna().unique()):
    part = meta[meta["partition"] == split]

    print(
        f"{split}:",
        len(part), "images |",
        part["label"].nunique(), "classes"
    )

print("\n===== LABEL CHECK =====")
print("Total images:", len(meta))
print("Unique diagnosis codes:", meta["diag"].nunique())
print("Unique condition names:", meta["label"].nunique())

print("\n===== DUPLICATE CHECK =====")
print("Duplicate md5hash:", meta["md5hash"].duplicated().sum())
print("Duplicate filepath:", meta["filepath"].duplicated().sum())

print("\nPartition values:", meta["partition"].unique())

===== OFFICIAL PARTITIONS =====
partition
train    7975
test     2280
val      1139
Name: count, dtype: int64

===== CLASSES PER PARTITION =====
test: 2280 images | 114 classes
train: 7975 images | 114 classes
val: 1139 images | 114 classes

===== LABEL CHECK =====
Total images: 11394
Unique diagnosis codes: 114
Unique condition names: 114

===== DUPLICATE CHECK =====
Duplicate md5hash: 0
Duplicate filepath: 0

Partition values: ['train' 'val' 'test']


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

CSV_PATH = os.path.join(WORK_DIR, "Fitzpatrick17k-C.csv")

IMAGE_ROOT = os.path.join(
    WORK_DIR,
    "Fitzpatrick17k_C",
    "Categorized_AbbrvName"
)

meta = pd.read_csv(CSV_PATH)

# The CSV filepath already contains:
# diagnosis_folder/image_filename.jpg
meta["image_path"] = meta["filepath"].apply(
    lambda x: os.path.join(IMAGE_ROOT, str(x))
)

# Check whether every corrected image actually exists
meta["exists"] = meta["image_path"].apply(os.path.exists)

print("===== IMAGE MATCHING =====")
print("Metadata rows:", len(meta))
print("Images found:", meta["exists"].sum())
print("Images missing:", (~meta["exists"]).sum())

# Show missing examples if any
if (~meta["exists"]).sum() > 0:
    print("\nFirst missing paths:")
    print(
        meta.loc[
            ~meta["exists"],
            ["filepath", "image_path"]
        ].head(10).to_string(index=False)
    )

# Only continue if ALL corrected images were found
assert meta["exists"].all(), "❌ Some corrected images could not be located."

# Build official splits
train_df = meta[meta["partition"] == "train"].copy()
val_df   = meta[meta["partition"] == "val"].copy()
test_df  = meta[meta["partition"] == "test"].copy()

# Save clean manifests
train_df.to_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_val.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_test.csv"),
    index=False
)

print("\n===== FINAL 114-CLASS DATASET =====")
print("Train:", len(train_df), "| Classes:", train_df["label"].nunique())
print("Validation:", len(val_df), "| Classes:", val_df["label"].nunique())
print("Test:", len(test_df), "| Classes:", test_df["label"].nunique())

print("\nTotal:", len(train_df) + len(val_df) + len(test_df))

print("\n✅ Official 114-class manifests created successfully.")

===== IMAGE MATCHING =====
Metadata rows: 11394
Images found: 11394
Images missing: 0

===== FINAL 114-CLASS DATASET =====
Train: 7975 | Classes: 114
Validation: 1139 | Classes: 114
Test: 2280 | Classes: 114

Total: 11394

✅ Official 114-class manifests created successfully.


In [ ]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# Load the manifests we just created
train_df = pd.read_csv(os.path.join(WORK_DIR, "fitzpatrick114_train.csv"))
val_df   = pd.read_csv(os.path.join(WORK_DIR, "fitzpatrick114_val.csv"))
test_df  = pd.read_csv(os.path.join(WORK_DIR, "fitzpatrick114_test.csv"))

IMG_SIZE = 224
BATCH_SIZE = 32

# --------------------------------------------------
# 1. Create a fixed 114-class label mapping
# --------------------------------------------------

class_names = sorted(train_df["label"].unique().tolist())

NUM_CLASSES = len(class_names)

class_to_index = {
    name: i for i, name in enumerate(class_names)
}

index_to_class = {
    i: name for name, i in class_to_index.items()
}

print("Number of classes:", NUM_CLASSES)

assert NUM_CLASSES == 114

# Convert text labels -> integer labels
for df in [train_df, val_df, test_df]:
    df["class_index"] = df["label"].map(class_to_index)

assert train_df["class_index"].isna().sum() == 0
assert val_df["class_index"].isna().sum() == 0
assert test_df["class_index"].isna().sum() == 0


# --------------------------------------------------
# 2. Image loading function
# --------------------------------------------------

def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    # Keep 0-255 float range.
    # EfficientNetB0 handles preprocessing internally.
    image = tf.cast(image, tf.float32)

    return image, label


# --------------------------------------------------
# 3. Dataset creator
# --------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

def create_dataset(df, training=False):

    paths = df["image_path"].astype(str).values
    labels = df["class_index"].astype("int32").values

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = create_dataset(train_df, training=True)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)


# --------------------------------------------------
# 4. Verify one batch
# --------------------------------------------------

images, labels = next(iter(train_ds))

print("\n===== DATASET CHECK =====")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nClasses:", NUM_CLASSES)

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)

print("Pixel minimum:", float(tf.reduce_min(images)))
print("Pixel maximum:", float(tf.reduce_max(images)))

print("\nExample labels:")

for i in range(min(5, len(labels))):
    idx = int(labels[i])
    print(idx, "→", index_to_class[idx])

print("\n✅ 114-class TensorFlow pipeline ready.")

Number of classes: 114

===== DATASET CHECK =====
Train images: 7975
Validation images: 1139
Test images: 2280

Classes: 114

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32,)
Pixel minimum: 0.0
Pixel maximum: 255.0

Example labels:
74 → perioral dermatitis
8 → basal cell carcinoma
92 → scabies
93 → scleroderma
29 → erythema nodosum

✅ 114-class TensorFlow pipeline ready.


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

NUM_CLASSES = 114
IMG_SIZE = 224

# -------------------------------
# Data augmentation
# -------------------------------

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")


# -------------------------------
# EfficientNetB0 backbone
# -------------------------------

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Stage 1: freeze backbone
base_model.trainable = False


# -------------------------------
# Build classifier
# -------------------------------

inputs = tf.keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    name="image"
)

x = augmentation(inputs)

x = base_model(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.Dropout(
    0.30,
    name="dropout"
)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="condition_predictions"
)(x)

model_114 = tf.keras.Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick114"
)


# -------------------------------
# Compile
# -------------------------------

model_114.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)


print("===== 114-CLASS MODEL =====")
print("Classes:", NUM_CLASSES)
print("Total parameters:", f"{model_114.count_params():,}")
print("Backbone trainable:", base_model.trainable)

print("\nInput:", model_114.input_shape)
print("Output:", model_114.output_shape)

assert model_114.output_shape[-1] == 114

print("\n✅ 114-class EfficientNetB0 created successfully.")

===== 114-CLASS MODEL =====
Classes: 114
Total parameters: 4,195,605
Backbone trainable: False

Input: (None, 224, 224, 3)
Output: (None, 114)

✅ 114-class EfficientNetB0 created successfully.


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

y_train = train_df["class_index"].values

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=y_train
)

class_weights = {
    i: float(w)
    for i, w in enumerate(weights)
}

print("Class weights created:", len(class_weights))
print("Minimum weight:", min(class_weights.values()))
print("Maximum weight:", max(class_weights.values()))

assert len(class_weights) == 114

print("✅ Class weights ready.")

Class weights created: 114
Minimum weight: 0.18704850361197112
Maximum weight: 4.9968671679198
✅ Class weights ready.


In [ ]:
STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_stage1_best.keras"
)

callbacks_stage1 = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE1_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Checkpoint:")
print(STAGE1_PATH)
print("\n✅ Stage 1 callbacks ready.")

Checkpoint:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_stage1_best.keras

✅ Stage 1 callbacks ready.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

train_df = pd.read_csv(os.path.join(WORK_DIR, "fitzpatrick114_train.csv"))
val_df   = pd.read_csv(os.path.join(WORK_DIR, "fitzpatrick114_val.csv"))
test_df  = pd.read_csv(os.path.join(WORK_DIR, "fitzpatrick114_test.csv"))

IMG_SIZE = 224
BATCH_SIZE = 32

class_names = sorted(train_df["label"].unique())
NUM_CLASSES = len(class_names)

class_to_index = {name: i for i, name in enumerate(class_names)}
index_to_class = {i: name for name, i in class_to_index.items()}

for df in [train_df, val_df, test_df]:
    df["class_index"] = df["label"].map(class_to_index).astype("int32")

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

def create_dataset(df, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            df["image_path"].astype(str).values,
            df["class_index"].values
        )
    )

    if training:
        ds = ds.shuffle(
            len(df),
            reshuffle_each_iteration=True
        )

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)

    return ds

train_ds = create_dataset(train_df, True)
val_ds   = create_dataset(val_df)
test_ds  = create_dataset(test_df)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Classes:", NUM_CLASSES)

assert NUM_CLASSES == 114

print("✅ Datasets restored")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 7975
Validation: 1139
Test: 2280
Classes: 114
✅ Datasets restored


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="image"
)

x = augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)

x = tf.keras.layers.Dropout(
    0.30,
    name="dropout"
)(x)

outputs = tf.keras.layers.Dense(
    114,
    activation="softmax",
    name="condition_predictions"
)(x)

model_114 = tf.keras.Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick114"
)

model_114.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")
    ]
)

print("Output:", model_114.output_shape)
print("✅ model_114 restored")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Output: (None, 114)
✅ model_114 restored


In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["class_index"].values
)

class_weights = {
    i: float(w)
    for i, w in enumerate(weights)
}

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_stage1_best.keras"
)

callbacks_stage1 = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE1_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Class weights:", len(class_weights))
print("Checkpoint:", STAGE1_PATH)
print("✅ Ready for Stage 1")

Class weights: 114
Checkpoint: /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_stage1_best.keras
✅ Ready for Stage 1


In [ ]:
history_stage1 = model_114.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_stage1
)

Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - accuracy: 0.0449 - loss: 4.6961 - top3: 0.1008 - top5: 0.1440 
Epoch 1: val_loss improved from None to 3.94190, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_stage1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_stage1_best.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 3051s 12s/step - accuracy: 0.0757 - loss: 4.3662 - top3: 0.1541 - top5: 0.2092 - val_accuracy: 0.1361 - val_loss: 3.9419 - val_top3: 0.2590 - val_top5: 0.3433 - learning_rate: 0.0010
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.1552 - loss: 3.6845 - top3: 0.2860 - top5: 0.3664
Epoch 2: val_loss improved from 3.94190 to 3.74246, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_stage1_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_stage1_

In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_stage1_best.keras"
)

best_stage1 = tf.keras.models.load_model(STAGE1_PATH)

results = best_stage1.evaluate(
    test_ds,
    verbose=1,
    return_dict=True
)

print("\n===== 114-CLASS STAGE 1 TEST RESULTS =====")
print(f"Loss:  {results['loss']:.4f}")
print(f"Top-1: {results['accuracy'] * 100:.2f}%")
print(f"Top-3: {results['top3'] * 100:.2f}%")
print(f"Top-5: {results['top5'] * 100:.2f}%")

72/72 ━━━━━━━━━━━━━━━━━━━━ 792s 11s/step - accuracy: 0.2263 - loss: 3.3215 - top3: 0.4039 - top5: 0.4965

===== 114-CLASS STAGE 1 TEST RESULTS =====
Loss:  3.3215
Top-1: 22.63%
Top-3: 40.39%
Top-5: 49.65%


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_stage1_best.keras"
)

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_finetuned_best.keras"
)

# Always start from the BEST Stage-1 checkpoint
model_114 = tf.keras.models.load_model(STAGE1_PATH)

# Find EfficientNet inside our model
base_model = None

for layer in model_114.layers:
    if isinstance(layer, tf.keras.Model) and "efficientnet" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise ValueError("Could not find EfficientNet backbone.")

print("Backbone:", base_model.name)
print("Total backbone layers:", len(base_model.layers))

# First freeze everything
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = False

# Unfreeze only last 30 layers, but keep BatchNorm frozen
for layer in base_model.layers[-30:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

trainable_backbone = sum(
    layer.trainable for layer in base_model.layers
)

print("Trainable backbone layers:", trainable_backbone)

model_114.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")
    ]
)

print("✅ Fine-tuning model ready.")

Backbone: efficientnetb0
Total backbone layers: 238
Trainable backbone layers: 23
✅ Fine-tuning model ready.


In [ ]:
callbacks_stage2 = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE2_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=os.path.join(
            WORK_DIR,
            "stage2_training_backup"
        )
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Best model →", STAGE2_PATH)
print("Recovery backup enabled ✅")

Best model → /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_finetuned_best.keras
Recovery backup enabled ✅


In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_stage1_best.keras"
)

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_finetuned_best.keras"
)

BACKUP_DIR = os.path.join(
    WORK_DIR,
    "stage2_training_backup"
)

print("Stage 1 model:", os.path.exists(STAGE1_PATH))
print("Stage 2 best model:", os.path.exists(STAGE2_PATH))
print("Stage 2 recovery backup:", os.path.exists(BACKUP_DIR))

if os.path.exists(STAGE2_PATH):
    print(
        "Stage 2 model size:",
        round(os.path.getsize(STAGE2_PATH) / (1024**2), 2),
        "MB"
    )

if os.path.exists(BACKUP_DIR):
    print("Backup contents:", os.listdir(BACKUP_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 1 model: True
Stage 2 best model: True
Stage 2 recovery backup: True
Stage 2 model size: 29.26 MB
Backup contents: ['latest.weights.h5', 'training_metadata.json']


In [ ]:
import os
import shutil
import time

DRIVE_IMAGE_ROOT = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "Fitzpatrick17k_C/Categorized_AbbrvName"
)

LOCAL_IMAGE_ROOT = "/content/Fitzpatrick17k_C/Categorized_AbbrvName"

print("Drive dataset exists:", os.path.exists(DRIVE_IMAGE_ROOT))

if not os.path.exists(DRIVE_IMAGE_ROOT):
    raise FileNotFoundError(
        f"Dataset not found:\n{DRIVE_IMAGE_ROOT}"
    )

if os.path.exists(LOCAL_IMAGE_ROOT):
    print("Local dataset already exists.")
else:
    print("Copying images from Google Drive to Colab local storage...")
    print("This may take several minutes.")

    start = time.time()

    os.makedirs(
        os.path.dirname(LOCAL_IMAGE_ROOT),
        exist_ok=True
    )

    shutil.copytree(
        DRIVE_IMAGE_ROOT,
        LOCAL_IMAGE_ROOT
    )

    print(
        "Copy completed in",
        round((time.time() - start) / 60, 2),
        "minutes"
    )

# Count copied image files
valid_extensions = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

image_count = 0

for root, dirs, files in os.walk(LOCAL_IMAGE_ROOT):
    image_count += sum(
        f.endswith(valid_extensions)
        for f in files
    )

print("\n===== LOCAL DATASET CHECK =====")
print("Local path:", LOCAL_IMAGE_ROOT)
print("Images found:", image_count)
print("Ready:", os.path.exists(LOCAL_IMAGE_ROOT))

Drive dataset exists: True
Copying images from Google Drive to Colab local storage...
This may take several minutes.
Copy completed in 18.41 minutes

===== LOCAL DATASET CHECK =====
Local path: /content/Fitzpatrick17k_C/Categorized_AbbrvName
Images found: 16577
Ready: True


In [ ]:
import os
import pandas as pd
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

LOCAL_IMAGE_ROOT = "/content/Fitzpatrick17k_C/Categorized_AbbrvName"

TRAIN_CSV = os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
VAL_CSV   = os.path.join(WORK_DIR, "fitzpatrick114_val.csv")
TEST_CSV  = os.path.join(WORK_DIR, "fitzpatrick114_test.csv")

# Load official corrected manifests
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Original counts:")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

# Convert existing Drive paths to local Colab paths
def make_local_path(old_path):
    old_path = str(old_path).replace("\\", "/")

    marker = "Categorized_AbbrvName/"

    if marker not in old_path:
        raise ValueError(f"Unexpected image path: {old_path}")

    relative_path = old_path.split(marker, 1)[1]

    return os.path.join(
        LOCAL_IMAGE_ROOT,
        relative_path
    )

train_df["local_path"] = train_df["image_path"].apply(make_local_path)
val_df["local_path"]   = val_df["image_path"].apply(make_local_path)
test_df["local_path"]  = test_df["image_path"].apply(make_local_path)

# Verify every official image exists locally
train_missing = (~train_df["local_path"].apply(os.path.exists)).sum()
val_missing   = (~val_df["local_path"].apply(os.path.exists)).sum()
test_missing  = (~test_df["local_path"].apply(os.path.exists)).sum()

print("\nMissing local images:")
print("Train:", train_missing)
print("Val:", val_missing)
print("Test:", test_missing)

# Fixed class ordering
class_names = sorted(train_df["label"].unique())
NUM_CLASSES = len(class_names)

class_to_index = {
    name: index
    for index, name in enumerate(class_names)
}

print("\nClasses:", NUM_CLASSES)

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def build_dataset(df, training=False):

    paths = df["local_path"].astype(str).values

    labels = (
        df["label"]
        .map(class_to_index)
        .astype("int32")
        .values
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(df),
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds = build_dataset(train_df, training=True)
val_ds   = build_dataset(val_df)
test_ds  = build_dataset(test_df)

print("\n===== LOCAL DATASETS READY =====")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Classes:", NUM_CLASSES)

# Test one batch
images, labels = next(iter(train_ds))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Pixel range:",
      float(tf.reduce_min(images)),
      "to",
      float(tf.reduce_max(images)))

print("\n✅ Training will now read images from local Colab storage.")

Original counts:
Train: 7975
Val: 1139
Test: 2280

Missing local images:
Train: 0
Val: 0
Test: 0

Classes: 114

===== LOCAL DATASETS READY =====
Train: 7975
Validation: 1139
Test: 2280
Classes: 114
Batch image shape: (32, 224, 224, 3)
Batch label shape: (32,)
Pixel range: 0.0 to 255.0

✅ Training will now read images from local Colab storage.


In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_finetuned_best.keras"
)

BACKUP_DIR = os.path.join(
    WORK_DIR,
    "stage2_training_backup"
)

# -------------------------------------------------
# 1. Load the best Stage-2 model already saved
# -------------------------------------------------

model_114 = tf.keras.models.load_model(STAGE2_PATH)

print("✅ Loaded Stage-2 checkpoint:")
print(STAGE2_PATH)

# -------------------------------------------------
# 2. Find EfficientNet backbone
# -------------------------------------------------

base_model = None

for layer in model_114.layers:
    if isinstance(layer, tf.keras.Model) and "efficientnet" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise ValueError("EfficientNet backbone could not be found.")

print("\nBackbone:", base_model.name)

# -------------------------------------------------
# 3. Preserve our fine-tuning configuration
# -------------------------------------------------

base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in base_model.layers)
)

# -------------------------------------------------
# 4. Recompile with Stage-2 learning rate
# -------------------------------------------------

model_114.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

# -------------------------------------------------
# 5. Recreate class weights
# -------------------------------------------------

train_labels = (
    train_df["label"]
    .map(class_to_index)
    .astype("int32")
    .values
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)

class_weights = {
    i: float(w)
    for i, w in enumerate(weights)
}

print("Class weights restored:", len(class_weights))

# -------------------------------------------------
# 6. Recreate callbacks
# -------------------------------------------------

callbacks_stage2 = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE2_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=BACKUP_DIR,
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("\nRecovery backup:", os.path.exists(BACKUP_DIR))
print("Best Stage-2 model:", os.path.exists(STAGE2_PATH))

print("\n✅ STAGE 2 READY TO RESUME")

✅ Loaded Stage-2 checkpoint:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_finetuned_best.keras

Backbone: efficientnetb0
Trainable backbone layers: 23
Class weights restored: 114

Recovery backup: True
Best Stage-2 model: True

✅ STAGE 2 READY TO RESUME


In [ ]:
history_stage2 = model_114.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_stage2
)

Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4149 - loss: 1.8298 - top3: 0.6159 - top5: 0.7141
Epoch 3: val_loss improved from None to 3.43802, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_finetuned_best.keras

Epoch 3: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_finetuned_best.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 1015s 4s/step - accuracy: 0.4239 - loss: 1.8249 - top3: 0.6247 - top5: 0.7135 - val_accuracy: 0.2441 - val_loss: 3.4380 - val_top3: 0.4223 - val_top5: 0.4934 - learning_rate: 1.0000e-05
Epoch 4/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4392 - loss: 1.7854 - top3: 0.6279 - top5: 0.7129
Epoch 4: val_loss did not improve from 3.43802
250/250 ━━━━━━━━━━━━━━━━━━━━ 994s 4s/step - accuracy: 0.4327 - loss: 1.8067 - top3: 0.6241 - top5: 0.7107 - val_accuracy: 0.2441 - val_loss: 3.4434 - val_top3: 0.4223 - val_top5: 0.4952 - learning_rate: 1.0000e-05
Epoch 

In [ ]:
import tensorflow as tf

best_stage2 = tf.keras.models.load_model(
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_finetuned_best.keras"
)

results_stage2 = best_stage2.evaluate(
    test_ds,
    verbose=1,
    return_dict=True
)

print("\n===== STAGE 2 TEST RESULTS =====")
print(f"Loss:  {results_stage2['loss']:.4f}")
print(f"Top-1: {results_stage2['accuracy'] * 100:.2f}%")
print(f"Top-3: {results_stage2['top3'] * 100:.2f}%")
print(f"Top-5: {results_stage2['top5'] * 100:.2f}%")

print("\n===== CHANGE FROM STAGE 1 =====")
print(f"Top-1: {results_stage2['accuracy']*100 - 22.63:+.2f}")
print(f"Top-3: {results_stage2['top3']*100 - 40.39:+.2f}")
print(f"Top-5: {results_stage2['top5']*100 - 49.65:+.2f}")

72/72 ━━━━━━━━━━━━━━━━━━━━ 210s 3s/step - accuracy: 0.2496 - loss: 3.3406 - top3: 0.4259 - top5: 0.5158

===== STAGE 2 TEST RESULTS =====
Loss:  3.3406
Top-1: 24.96%
Top-3: 42.59%
Top-5: 51.58%

===== CHANGE FROM STAGE 1 =====
Top-1: +2.33
Top-3: +2.20
Top-5: +1.93


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# Use our best Stage-2 model
best_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_finetuned_best.keras"
)

print("Generating predictions...")

# True labels
y_true = test_df["label"].map(class_to_index).astype(int).values

# Predictions
pred_probs = best_model.predict(
    test_ds,
    verbose=1
)

y_pred = np.argmax(pred_probs, axis=1)

# Classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

print("\n===== OVERALL PERFORMANCE =====")
print(f"Accuracy:    {report['accuracy'] * 100:.2f}%")
print(f"Macro F1:    {report['macro avg']['f1-score']:.4f}")
print(f"Weighted F1: {report['weighted avg']['f1-score']:.4f}")
print(f"Macro Recall:{report['macro avg']['recall']:.4f}")

# Per-class table
per_class = report_df.loc[class_names][
    ["precision", "recall", "f1-score", "support"]
].copy()

per_class = per_class.sort_values(
    "f1-score",
    ascending=True
)

print("\n===== 15 LOWEST F1 CLASSES =====")
print(per_class.head(15).to_string())

print("\n===== 15 HIGHEST F1 CLASSES =====")
print(per_class.tail(15).sort_values(
    "f1-score",
    ascending=False
).to_string())

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

confusions = []

for true_idx in range(len(class_names)):
    for pred_idx in range(len(class_names)):

        if true_idx == pred_idx:
            continue

        count = cm[true_idx, pred_idx]

        if count > 0:
            confusions.append({
                "True": class_names[true_idx],
                "Predicted": class_names[pred_idx],
                "Count": int(count)
            })

confusion_df = pd.DataFrame(confusions)

confusion_df = confusion_df.sort_values(
    "Count",
    ascending=False
)

print("\n===== TOP 20 CONFUSIONS =====")
print(confusion_df.head(20).to_string(index=False))

# Save detailed results
REPORT_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "fitzpatrick114_stage2_classification_report.csv"
)

CONFUSION_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "fitzpatrick114_stage2_confusions.csv"
)

per_class.to_csv(REPORT_PATH)
confusion_df.to_csv(CONFUSION_PATH, index=False)

print("\nSaved:")
print(REPORT_PATH)
print(CONFUSION_PATH)

Generating predictions...
72/72 ━━━━━━━━━━━━━━━━━━━━ 198s 3s/step

===== OVERALL PERFORMANCE =====
Accuracy:    24.96%
Macro F1:    0.2332
Weighted F1: 0.2392
Macro Recall:0.2762

===== 15 LOWEST F1 CLASSES =====
                                    precision    recall  f1-score  support
calcinosis cutis                     0.000000  0.000000  0.000000      9.0
fixed eruptions                      0.000000  0.000000  0.000000     17.0
drug induced pigmentary changes      0.000000  0.000000  0.000000     10.0
disseminated actinic porokeratosis   0.000000  0.000000  0.000000      4.0
erythema elevatum diutinum           0.000000  0.000000  0.000000      4.0
milia                                0.000000  0.000000  0.000000     17.0
lichen simplex                       0.000000  0.000000  0.000000     13.0
neurodermatitis                      0.000000  0.000000  0.000000     11.0
tuberous sclerosis                   0.000000  0.000000  0.000000     13.0
pilomatricoma                        

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.backend.clear_session()

IMG_SIZE_B2 = 260
NUM_CLASSES = 114

augmentation_b2 = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="augmentation_b2")

base_b2 = tf.keras.applications.EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE_B2, IMG_SIZE_B2, 3)
)

base_b2.trainable = False

inputs = layers.Input(
    shape=(IMG_SIZE_B2, IMG_SIZE_B2, 3)
)

x = augmentation_b2(inputs)

x = base_b2(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.40)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="condition_predictions"
)(x)

model_b2 = Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick114_B2"
)

print("✅ EfficientNetB2 created successfully")
print("Input:", model_b2.input_shape)
print("Output:", model_b2.output_shape)
print(f"Parameters: {model_b2.count_params():,}")

model_b2.summary()

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
✅ EfficientNetB2 created successfully
Input: (None, 260, 260, 3)
Output: (None, 114)
Parameters: 7,929,195


Model: "DermaWise_Fitzpatrick114_B2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation_b2 (Sequential)    │ (None, 260, 260, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb2 (Functional)     │ (None, 9, 9, 1408)     │     7,768,569 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ condition_predictions (Dense)   │ (None, 114)            │       160,626 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,929,195 (30.25 MB)

 Trainable params: 160,626 (627.45 KB)

 Non-trainable params: 7,768,569 (29.63 MB)

In [ ]:
import tensorflow as tf

IMG_SIZE_B2 = 260
BATCH_SIZE_B2 = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image_b2(path, label):
    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_B2, IMG_SIZE_B2]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def build_dataset_b2(df, training=False):

    paths = df["local_path"].astype(str).values

    labels = (
        df["label"]
        .map(class_to_index)
        .astype("int32")
        .values
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            len(df),
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image_b2,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE_B2)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds_b2 = build_dataset_b2(
    train_df,
    training=True
)

val_ds_b2 = build_dataset_b2(val_df)

test_ds_b2 = build_dataset_b2(test_df)

print("===== B2 DATASETS =====")
print("Image size:", IMG_SIZE_B2)
print("Batch size:", BATCH_SIZE_B2)
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

images, labels = next(iter(train_ds_b2))

print("\nBatch images:", images.shape)
print("Batch labels:", labels.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("\n✅ B2 datasets ready")

===== B2 DATASETS =====
Image size: 260
Batch size: 16
Train images: 7975
Validation images: 1139
Test images: 2280

Batch images: (16, 260, 260, 3)
Batch labels: (16,)
Pixel range: 0.0 to 255.0

✅ B2 datasets ready


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

B2_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_stage1_best.keras"
)

B2_BACKUP_DIR = os.path.join(
    WORK_DIR,
    "b2_stage1_training_backup"
)

# Make sure EfficientNetB2 remains frozen for Stage 1
base_b2.trainable = False

# Compile
model_b2.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

# Restart-safe callbacks
callbacks_b2_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        B2_STAGE1_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=B2_BACKUP_DIR,
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("===== B2 STAGE 1 READY =====")
print("Model:", model_b2.name)
print("Backbone frozen:", not base_b2.trainable)
print("Trainable parameters:",
      sum(tf.keras.backend.count_params(w)
          for w in model_b2.trainable_weights))

print("\nBest checkpoint:")
print(B2_STAGE1_PATH)

print("\nRecovery directory:")
print(B2_BACKUP_DIR)

print("\n✅ Safe to start B2 Stage 1")

===== B2 STAGE 1 READY =====
Model: DermaWise_Fitzpatrick114_B2
Backbone frozen: True
Trainable parameters: 160626

Best checkpoint:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_stage1_best.keras

Recovery directory:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/b2_stage1_training_backup

✅ Safe to start B2 Stage 1


In [ ]:
import os

LOCAL_IMAGE_ROOT = "/content/Fitzpatrick17k_C/Categorized_AbbrvName"

print("Local images exist:", os.path.exists(LOCAL_IMAGE_ROOT))

if os.path.exists(LOCAL_IMAGE_ROOT):
    image_count = 0

    for root, dirs, files in os.walk(LOCAL_IMAGE_ROOT):
        image_count += sum(
            f.lower().endswith((".jpg", ".jpeg", ".png"))
            for f in files
        )

    print("Images found:", image_count)

Local images exist: False


In [ ]:
import os
import shutil
import time

DRIVE_IMAGE_ROOT = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "Fitzpatrick17k_C/Categorized_AbbrvName"
)

LOCAL_IMAGE_ROOT = (
    "/content/Fitzpatrick17k_C/Categorized_AbbrvName"
)

print("Drive dataset exists:", os.path.exists(DRIVE_IMAGE_ROOT))

if not os.path.exists(DRIVE_IMAGE_ROOT):
    raise FileNotFoundError(
        "Drive dataset not found. Check Google Drive mount."
    )

print("Copying dataset to fast local Colab storage...")
print("Please leave this cell running.")

start = time.time()

os.makedirs(
    os.path.dirname(LOCAL_IMAGE_ROOT),
    exist_ok=True
)

shutil.copytree(
    DRIVE_IMAGE_ROOT,
    LOCAL_IMAGE_ROOT,
    dirs_exist_ok=True
)

minutes = (time.time() - start) / 60

print(f"\nCopy completed in {minutes:.2f} minutes")

# Verify
image_count = 0

for root, dirs, files in os.walk(LOCAL_IMAGE_ROOT):
    image_count += sum(
        f.lower().endswith((".jpg", ".jpeg", ".png"))
        for f in files
    )

print("\n===== VERIFICATION =====")
print("Images found:", image_count)
print("Ready:", image_count == 16577)

Drive dataset exists: True
Copying dataset to fast local Colab storage...
Please leave this cell running.

Copy completed in 30.93 minutes

===== VERIFICATION =====
Images found: 16577
Ready: True


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from sklearn.utils.class_weight import compute_class_weight

# ============================================================
# PATHS
# ============================================================

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

LOCAL_IMAGE_ROOT = (
    "/content/Fitzpatrick17k_C/Categorized_AbbrvName"
)

TRAIN_CSV = os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
VAL_CSV   = os.path.join(WORK_DIR, "fitzpatrick114_val.csv")
TEST_CSV  = os.path.join(WORK_DIR, "fitzpatrick114_test.csv")

B2_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_stage1_best.keras"
)

B2_BACKUP_DIR = os.path.join(
    WORK_DIR,
    "b2_stage1_training_backup"
)

# ============================================================
# LOAD OFFICIAL SPLITS
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

def make_local_path(old_path):
    old_path = str(old_path).replace("\\", "/")
    marker = "Categorized_AbbrvName/"

    if marker not in old_path:
        raise ValueError(f"Unexpected path: {old_path}")

    relative = old_path.split(marker, 1)[1]

    return os.path.join(
        LOCAL_IMAGE_ROOT,
        relative
    )

train_df["local_path"] = train_df["image_path"].apply(make_local_path)
val_df["local_path"]   = val_df["image_path"].apply(make_local_path)
test_df["local_path"]  = test_df["image_path"].apply(make_local_path)

# ============================================================
# VERIFY FILES
# ============================================================

print("Missing images:")
print("Train:", (~train_df["local_path"].apply(os.path.exists)).sum())
print("Val:", (~val_df["local_path"].apply(os.path.exists)).sum())
print("Test:", (~test_df["local_path"].apply(os.path.exists)).sum())

# ============================================================
# CLASSES
# ============================================================

class_names = sorted(train_df["label"].unique())

class_to_index = {
    name: i
    for i, name in enumerate(class_names)
}

NUM_CLASSES = len(class_names)

print("\nClasses:", NUM_CLASSES)

# ============================================================
# B2 DATASETS
# ============================================================

IMG_SIZE_B2 = 260
BATCH_SIZE_B2 = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image_b2(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_B2, IMG_SIZE_B2]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def build_dataset_b2(df, training=False):

    paths = df["local_path"].astype(str).values

    labels = (
        df["label"]
        .map(class_to_index)
        .astype("int32")
        .values
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            len(df),
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image_b2,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE_B2)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds_b2 = build_dataset_b2(train_df, True)
val_ds_b2   = build_dataset_b2(val_df)
test_ds_b2  = build_dataset_b2(test_df)

# ============================================================
# CLASS WEIGHTS
# ============================================================

train_labels = (
    train_df["label"]
    .map(class_to_index)
    .astype("int32")
    .values
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)

class_weights = {
    i: float(w)
    for i, w in enumerate(weights)
}

# ============================================================
# BUILD EFFICIENTNETB2
# ============================================================

tf.keras.backend.clear_session()

augmentation_b2 = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10)
], name="augmentation_b2")

base_b2 = tf.keras.applications.EfficientNetB2(
    weights="imagenet",
    include_top=False,
    input_shape=(260, 260, 3)
)

base_b2.trainable = False

inputs = layers.Input(
    shape=(260, 260, 3)
)

x = augmentation_b2(inputs)

x = base_b2(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.40)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="condition_predictions"
)(x)

model_b2 = Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick114_B2"
)

# ============================================================
# COMPILE
# ============================================================

model_b2.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

# ============================================================
# CALLBACKS
# ============================================================

callbacks_b2_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        B2_STAGE1_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=B2_BACKUP_DIR,
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# ============================================================
# FINAL CHECK
# ============================================================

images, labels = next(iter(train_ds_b2))

print("\n===== B2 RESTORATION COMPLETE =====")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("Classes:", NUM_CLASSES)

print("Batch:", images.shape)

print("Backbone:", base_b2.name)
print("Backbone frozen:", not base_b2.trainable)

print(
    "Trainable parameters:",
    sum(
        tf.keras.backend.count_params(w)
        for w in model_b2.trainable_weights
    )
)

print("Class weights:", len(class_weights))

print("\nBest model path:")
print(B2_STAGE1_PATH)

print("\nRecovery enabled:")
print(B2_BACKUP_DIR)

print("\n✅ B2 READY FOR TRAINING")

Missing images:
Train: 0
Val: 0
Test: 0

Classes: 114
31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

===== B2 RESTORATION COMPLETE =====
Train: 7975
Validation: 1139
Test: 2280
Classes: 114
Batch: (16, 260, 260, 3)
Backbone: efficientnetb2
Backbone frozen: True
Trainable parameters: 160626
Class weights: 114

Best model path:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_stage1_best.keras

Recovery enabled:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/b2_stage1_training_backup

✅ B2 READY FOR TRAINING


In [ ]:
history_b2_stage1 = model_b2.fit(
    train_ds_b2,
    validation_data=val_ds_b2,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_b2_stage1
)

Epoch 1/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.0528 - loss: 4.6167 - top3: 0.1138 - top5: 0.1621
Epoch 1: val_loss improved from None to 3.87891, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_stage1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_stage1_best.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 1290s 3s/step - accuracy: 0.0844 - loss: 4.3179 - top3: 0.1713 - top5: 0.2267 - val_accuracy: 0.1484 - val_loss: 3.8789 - val_top3: 0.2599 - val_top5: 0.3406 - learning_rate: 0.0010
Epoch 2/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1683 - loss: 3.4850 - top3: 0.3095 - top5: 0.3975
Epoch 2: val_loss improved from 3.87891 to 3.62359, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_stage1_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b

In [ ]:
import tensorflow as tf

B2_STAGE1_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_b2_stage1_best.keras"
)

best_b2_stage1 = tf.keras.models.load_model(
    B2_STAGE1_PATH
)

results_b2 = best_b2_stage1.evaluate(
    test_ds_b2,
    verbose=1,
    return_dict=True
)

print("\n===== EFFICIENTNETB2 STAGE 1 TEST =====")
print(f"Loss:  {results_b2['loss']:.4f}")
print(f"Top-1: {results_b2['accuracy'] * 100:.2f}%")
print(f"Top-3: {results_b2['top3'] * 100:.2f}%")
print(f"Top-5: {results_b2['top5'] * 100:.2f}%")

print("\n===== CURRENT B0 FINAL BENCHMARK =====")
print("Top-1: 24.96%")
print("Top-3: 42.59%")
print("Top-5: 51.58%")

print("\n===== B2 STAGE 1 vs B0 FINAL =====")
print(
    f"Top-1: "
    f"{results_b2['accuracy']*100 - 24.96:+.2f} percentage points"
)

print(
    f"Top-3: "
    f"{results_b2['top3']*100 - 42.59:+.2f} percentage points"
)

print(
    f"Top-5: "
    f"{results_b2['top5']*100 - 51.58:+.2f} percentage points"
)

143/143 ━━━━━━━━━━━━━━━━━━━━ 300s 2s/step - accuracy: 0.2395 - loss: 3.2930 - top3: 0.4075 - top5: 0.4987

===== EFFICIENTNETB2 STAGE 1 TEST =====
Loss:  3.2930
Top-1: 23.95%
Top-3: 40.75%
Top-5: 49.87%

===== CURRENT B0 FINAL BENCHMARK =====
Top-1: 24.96%
Top-3: 42.59%
Top-5: 51.58%

===== B2 STAGE 1 vs B0 FINAL =====
Top-1: -1.01 percentage points
Top-3: -1.84 percentage points
Top-5: -1.71 percentage points


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

B2_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_stage1_best.keras"
)

B2_STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_finetuned_best.keras"
)

B2_STAGE2_BACKUP_DIR = os.path.join(
    WORK_DIR,
    "b2_stage2_training_backup"
)

# ------------------------------------------------------------
# 1. Load the BEST Stage-1 model
# ------------------------------------------------------------

model_b2 = tf.keras.models.load_model(B2_STAGE1_PATH)

# Find EfficientNetB2 backbone
base_b2 = None

for layer in model_b2.layers:
    if isinstance(layer, tf.keras.Model) and "efficientnet" in layer.name.lower():
        base_b2 = layer
        break

if base_b2 is None:
    raise ValueError("EfficientNetB2 backbone could not be found.")

print("Backbone found:", base_b2.name)
print("Total backbone layers:", len(base_b2.layers))

# ------------------------------------------------------------
# 2. Unfreeze backbone
# ------------------------------------------------------------

base_b2.trainable = True

# Freeze everything first
for layer in base_b2.layers:
    layer.trainable = False

# ------------------------------------------------------------
# 3. Fine-tune ONLY the last 40 layers
# ------------------------------------------------------------

for layer in base_b2.layers[-40:]:

    # Keep Batch Normalization frozen
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

# ------------------------------------------------------------
# 4. Count trainable layers
# ------------------------------------------------------------

trainable_backbone_layers = [
    layer.name
    for layer in base_b2.layers
    if layer.trainable
]

print("\nTrainable backbone layers:",
      len(trainable_backbone_layers))

print("\nLast trainable layers:")
for name in trainable_backbone_layers[-10:]:
    print(" -", name)

# ------------------------------------------------------------
# 5. Compile with LOW learning rate
# ------------------------------------------------------------

model_b2.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

# ------------------------------------------------------------
# 6. Stage-2 callbacks
# ------------------------------------------------------------

callbacks_b2_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(
        B2_STAGE2_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=B2_STAGE2_BACKUP_DIR,
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in model_b2.trainable_weights
)

print("\n====================================")
print("B2 STAGE 2 READY")
print("====================================")

print("Backbone:", base_b2.name)
print("Trainable backbone layers:",
      len(trainable_backbone_layers))

print("Trainable parameters:",
      f"{trainable_params:,}")

print("Learning rate:",
      float(model_b2.optimizer.learning_rate.numpy()))

print("\nCheckpoint:")
print(B2_STAGE2_PATH)

print("\nRecovery directory:")
print(B2_STAGE2_BACKUP_DIR)

print("\n✅ READY FOR FINE-TUNING")

Backbone found: efficientnetb2
Total backbone layers: 340

Trainable backbone layers: 32

Last trainable layers:
 - block7b_se_squeeze
 - block7b_se_reshape
 - block7b_se_reduce
 - block7b_se_expand
 - block7b_se_excite
 - block7b_project_conv
 - block7b_drop
 - block7b_add
 - top_conv
 - top_activation

B2 STAGE 2 READY
Backbone: efficientnetb2
Trainable backbone layers: 32
Trainable parameters: 3,767,890
Learning rate: 9.999999747378752e-06

Checkpoint:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_finetuned_best.keras

Recovery directory:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/b2_stage2_training_backup

✅ READY FOR FINE-TUNING


In [ ]:
import os

BEST_MODEL = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_b2_finetuned_best.keras"
)

BACKUP_DIR = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "b2_stage2_training_backup"
)

print("Best model exists:", os.path.exists(BEST_MODEL))

if os.path.exists(BEST_MODEL):
    print(
        "Best model size:",
        round(os.path.getsize(BEST_MODEL) / (1024**2), 2),
        "MB"
    )

print("Backup exists:", os.path.exists(BACKUP_DIR))

if os.path.exists(BACKUP_DIR):
    print("Backup contents:", os.listdir(BACKUP_DIR))

Best model exists: True
Best model size: 60.15 MB
Backup exists: True
Backup contents: ['training_metadata.json', 'latest.weights.h5']


In [ ]:
import os

LOCAL_IMAGE_ROOT = "/content/Fitzpatrick17k_C/Categorized_AbbrvName"

print("Local images exist:", os.path.exists(LOCAL_IMAGE_ROOT))

Local images exist: False


In [ ]:
import os
import shutil
import time

DRIVE_IMAGE_ROOT = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "Fitzpatrick17k_C/Categorized_AbbrvName"
)

LOCAL_IMAGE_ROOT = (
    "/content/Fitzpatrick17k_C/Categorized_AbbrvName"
)

print("Drive dataset exists:", os.path.exists(DRIVE_IMAGE_ROOT))

if not os.path.exists(DRIVE_IMAGE_ROOT):
    raise FileNotFoundError("Dataset not found in Google Drive.")

print("Copying images to local Colab storage...")
start = time.time()

os.makedirs(
    os.path.dirname(LOCAL_IMAGE_ROOT),
    exist_ok=True
)

shutil.copytree(
    DRIVE_IMAGE_ROOT,
    LOCAL_IMAGE_ROOT,
    dirs_exist_ok=True
)

elapsed = (time.time() - start) / 60

# Verify
image_count = 0

for root, dirs, files in os.walk(LOCAL_IMAGE_ROOT):
    image_count += sum(
        f.lower().endswith((".jpg", ".jpeg", ".png"))
        for f in files
    )

print("\n===== COPY COMPLETE =====")
print(f"Time: {elapsed:.2f} minutes")
print("Images found:", image_count)
print("Ready:", image_count == 16577)

Drive dataset exists: True
Copying images to local Colab storage...

===== COPY COMPLETE =====
Time: 22.43 minutes
Images found: 16577
Ready: True


In [ ]:
import os
import pandas as pd
import tensorflow as tf

# ============================================================
# PATHS
# ============================================================

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TEST_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_test.csv"
)

BEST_MODEL = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_finetuned_best.keras"
)

LOCAL_IMAGE_ROOT = (
    "/content/Fitzpatrick17k_C/Categorized_AbbrvName"
)

# ============================================================
# LOAD TEST MANIFEST
# ============================================================

test_df = pd.read_csv(TEST_CSV)

# We need the training CSV only to reconstruct the SAME
# alphabetical 114-class mapping used during training.
train_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
)

class_names = sorted(train_df["label"].unique())

class_to_index = {
    name: i
    for i, name in enumerate(class_names)
}

print("Classes:", len(class_names))

# ============================================================
# CHANGE DRIVE PATHS TO LOCAL PATHS
# ============================================================

def make_local_path(old_path):

    old_path = str(old_path).replace("\\", "/")

    marker = "Categorized_AbbrvName/"

    if marker not in old_path:
        raise ValueError(f"Unexpected path: {old_path}")

    relative = old_path.split(marker, 1)[1]

    return os.path.join(
        LOCAL_IMAGE_ROOT,
        relative
    )


test_df["local_path"] = (
    test_df["image_path"].apply(make_local_path)
)

missing = (
    ~test_df["local_path"].apply(os.path.exists)
).sum()

print("Test images:", len(test_df))
print("Missing:", missing)

# ============================================================
# TEST DATASET — 260x260 FOR B2
# ============================================================

IMG_SIZE = 260
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE


def load_test_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


test_paths = (
    test_df["local_path"]
    .astype(str)
    .values
)

test_labels = (
    test_df["label"]
    .map(class_to_index)
    .astype("int32")
    .values
)

test_ds_b2 = (
    tf.data.Dataset
    .from_tensor_slices(
        (test_paths, test_labels)
    )
    .map(
        load_test_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ============================================================
# LOAD SAVED B2 MODEL
# ============================================================

print("\nLoading saved B2 model...")

best_b2 = tf.keras.models.load_model(
    BEST_MODEL
)

print("Model loaded:", best_b2.name)

# ============================================================
# OFFICIAL TEST EVALUATION
# ============================================================

print("\nEvaluating 2,280 official test images...\n")

results = best_b2.evaluate(
    test_ds_b2,
    verbose=1,
    return_dict=True
)

top1 = results["accuracy"] * 100
top3 = results["top3"] * 100
top5 = results["top5"] * 100

print("\n====================================")
print("B2 FINE-TUNED — OFFICIAL TEST")
print("====================================")

print(f"Loss:  {results['loss']:.4f}")
print(f"Top-1: {top1:.2f}%")
print(f"Top-3: {top3:.2f}%")
print(f"Top-5: {top5:.2f}%")

print("\n====================================")
print("B0 FINAL BENCHMARK")
print("====================================")

print("Top-1: 24.96%")
print("Top-3: 42.59%")
print("Top-5: 51.58%")

print("\n====================================")
print("B2 CHANGE vs B0")
print("====================================")

print(f"Top-1: {top1 - 24.96:+.2f} percentage points")
print(f"Top-3: {top3 - 42.59:+.2f} percentage points")
print(f"Top-5: {top5 - 51.58:+.2f} percentage points")

Classes: 114
Test images: 2280
Missing: 0

Loading saved B2 model...
Model loaded: DermaWise_Fitzpatrick114_B2

Evaluating 2,280 official test images...

143/143 ━━━━━━━━━━━━━━━━━━━━ 321s 2s/step - accuracy: 0.2746 - loss: 3.2407 - top3: 0.4447 - top5: 0.5412

B2 FINE-TUNED — OFFICIAL TEST
Loss:  3.2407
Top-1: 27.46%
Top-3: 44.47%
Top-5: 54.12%

B0 FINAL BENCHMARK
Top-1: 24.96%
Top-3: 42.59%
Top-5: 51.58%

B2 CHANGE vs B0
Top-1: +2.50 percentage points
Top-3: +1.88 percentage points
Top-5: +2.54 percentage points


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

LOCAL_IMAGE_ROOT = (
    "/content/Fitzpatrick17k_C/Categorized_AbbrvName"
)

TRAIN_CSV = os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
VAL_CSV   = os.path.join(WORK_DIR, "fitzpatrick114_val.csv")

# ------------------------------------------------------------
# LOAD MANIFESTS
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

def make_local_path(old_path):
    old_path = str(old_path).replace("\\", "/")
    marker = "Categorized_AbbrvName/"

    if marker not in old_path:
        raise ValueError(f"Unexpected path: {old_path}")

    relative = old_path.split(marker, 1)[1]

    return os.path.join(
        LOCAL_IMAGE_ROOT,
        relative
    )

train_df["local_path"] = train_df["image_path"].apply(make_local_path)
val_df["local_path"]   = val_df["image_path"].apply(make_local_path)

print("Missing train:",
      (~train_df["local_path"].apply(os.path.exists)).sum())

print("Missing val:",
      (~val_df["local_path"].apply(os.path.exists)).sum())

# ------------------------------------------------------------
# SAME 114-CLASS MAPPING
# ------------------------------------------------------------

class_names = sorted(train_df["label"].unique())

class_to_index = {
    name: i
    for i, name in enumerate(class_names)
}

NUM_CLASSES = len(class_names)

# ------------------------------------------------------------
# DATA PIPELINE
# ------------------------------------------------------------

IMG_SIZE = 260
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def make_dataset(df, training=False):

    paths = df["local_path"].astype(str).values

    labels = (
        df["label"]
        .map(class_to_index)
        .astype("int32")
        .values
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        ds = ds.shuffle(
            len(df),
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        load_image,
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(AUTOTUNE)

    return ds


train_ds_b2 = make_dataset(
    train_df,
    training=True
)

val_ds_b2 = make_dataset(
    val_df,
    training=False
)

# ------------------------------------------------------------
# REBUILD SAME CLASS WEIGHTS
# ------------------------------------------------------------

train_labels = (
    train_df["label"]
    .map(class_to_index)
    .astype("int32")
    .values
)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)

class_weights = {
    i: float(w)
    for i, w in enumerate(weights)
}

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

images, labels = next(iter(train_ds_b2))

print("\n===== DATASETS RESTORED =====")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Classes:", NUM_CLASSES)
print("Batch:", images.shape)
print("Class weights:", len(class_weights))

print("\n✅ READY FOR MODEL RECOVERY")

Missing train: 0
Missing val: 0

===== DATASETS RESTORED =====
Train images: 7975
Validation images: 1139
Classes: 114
Batch: (16, 260, 260, 3)
Class weights: 114

✅ READY FOR MODEL RECOVERY


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# Current best model — KEEP SAFE
CURRENT_BEST = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_finetuned_best.keras"
)

# New file for continued training
CONTINUED_BEST = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_continued_best.keras"
)

CONTINUED_BACKUP = os.path.join(
    WORK_DIR,
    "b2_continued_training_backup"
)

# ------------------------------------------------------------
# LOAD CURRENT BEST MODEL
# ------------------------------------------------------------

model_b2 = tf.keras.models.load_model(CURRENT_BEST)

print("Loaded:", model_b2.name)

# ------------------------------------------------------------
# FIND EFFICIENTNETB2 BACKBONE
# ------------------------------------------------------------

base_b2 = None

for layer in model_b2.layers:
    if (
        isinstance(layer, tf.keras.Model)
        and "efficientnet" in layer.name.lower()
    ):
        base_b2 = layer
        break

if base_b2 is None:
    raise ValueError("EfficientNetB2 backbone not found.")

print("Backbone:", base_b2.name)
print("Total backbone layers:", len(base_b2.layers))

# ------------------------------------------------------------
# RESTORE SAME STAGE-2 TRAINABLE CONFIGURATION
# ------------------------------------------------------------

base_b2.trainable = True

# Freeze everything first
for layer in base_b2.layers:
    layer.trainable = False

# Last 40 layers eligible for fine-tuning,
# but BatchNormalization remains frozen.
for layer in base_b2.layers[-40:]:

    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = True

trainable_backbone_layers = [
    layer
    for layer in base_b2.layers
    if layer.trainable
]

# ------------------------------------------------------------
# USE THE REDUCED LR REACHED DURING ORIGINAL TRAINING
# ------------------------------------------------------------

model_b2.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=3e-6
    ),

    loss=tf.keras.losses.SparseCategoricalCrossentropy(),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

# ------------------------------------------------------------
# CALLBACKS
# ------------------------------------------------------------

callbacks_continue = [

    tf.keras.callbacks.ModelCheckpoint(
        CONTINUED_BEST,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=CONTINUED_BACKUP,
        save_freq="epoch",
        delete_checkpoint=False
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in model_b2.trainable_weights
)

print("\n====================================")
print("B2 CONTINUATION READY")
print("====================================")

print(
    "Trainable backbone layers:",
    len(trainable_backbone_layers)
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print(
    "Learning rate:",
    float(model_b2.optimizer.learning_rate.numpy())
)

print("\nOriginal best remains:")
print(CURRENT_BEST)

print("\nContinuation will save to:")
print(CONTINUED_BEST)

print("\n✅ SAFE TO CONTINUE")

Loaded: DermaWise_Fitzpatrick114_B2
Backbone: efficientnetb2
Total backbone layers: 340

B2 CONTINUATION READY
Trainable backbone layers: 32
Trainable parameters: 3,767,890
Learning rate: 3.000000106112566e-06

Original best remains:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_finetuned_best.keras

Continuation will save to:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_continued_best.keras

✅ SAFE TO CONTINUE


In [ ]:
history_b2_continue = model_b2.fit(
    train_ds_b2,
    validation_data=val_ds_b2,
    initial_epoch=10,
    epochs=12,
    class_weight=class_weights,
    callbacks=callbacks_continue
)

Epoch 11/12
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4779 - loss: 1.5690 - top3: 0.6716 - top5: 0.7485
Epoch 11: val_loss improved from None to 3.26295, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_continued_best.keras

Epoch 11: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_continued_best.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 1844s 4s/step - accuracy: 0.4746 - loss: 1.5657 - top3: 0.6707 - top5: 0.7483 - val_accuracy: 0.2845 - val_loss: 3.2629 - val_top3: 0.4548 - val_top5: 0.5382 - learning_rate: 3.0000e-06
Epoch 12/12
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4811 - loss: 1.5429 - top3: 0.6793 - top5: 0.7571
Epoch 12: val_loss improved from 3.26295 to 3.26214, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_continued_best.keras

Epoch 12: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawi

In [ ]:
import os

LOCAL_IMAGE_ROOT = "/content/Fitzpatrick17k_C/Categorized_AbbrvName"

ORIGINAL_BEST = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_b2_finetuned_best.keras"
)

CONTINUED_BEST = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_b2_continued_best.keras"
)

CONTINUED_BACKUP = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "b2_continued_training_backup"
)

print("===== DERMAWISE STATUS =====")
print("Local images:", os.path.exists(LOCAL_IMAGE_ROOT))
print("Original 27.46% model:", os.path.exists(ORIGINAL_BEST))
print("Continued model:", os.path.exists(CONTINUED_BEST))
print("Continuation backup:", os.path.exists(CONTINUED_BACKUP))

if os.path.exists(CONTINUED_BEST):
    print(
        "Continued model size:",
        round(os.path.getsize(CONTINUED_BEST)/(1024**2), 2),
        "MB"
    )

if os.path.exists(CONTINUED_BACKUP):
    print(
        "Backup contents:",
        os.listdir(CONTINUED_BACKUP)
    )

===== DERMAWISE STATUS =====
Local images: False
Original 27.46% model: True
Continued model: True
Continuation backup: True
Continued model size: 60.15 MB
Backup contents: ['training_metadata.json', 'latest.weights.h5']


In [ ]:
import os
import shutil
import time
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

DRIVE_IMAGE_ROOT = os.path.join(
    WORK_DIR,
    "Fitzpatrick17k_C",
    "Categorized_AbbrvName"
)

LOCAL_TEST_ROOT = "/content/DermaWise_Test_Images"

TEST_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_test.csv"
)

test_df = pd.read_csv(TEST_CSV)

os.makedirs(LOCAL_TEST_ROOT, exist_ok=True)

def get_relative_path(old_path):
    old_path = str(old_path).replace("\\", "/")
    marker = "Categorized_AbbrvName/"

    if marker not in old_path:
        raise ValueError(f"Unexpected path: {old_path}")

    return old_path.split(marker, 1)[1]


print("Copying ONLY 2,280 test images...")
start = time.time()

copied = 0
missing = 0

local_paths = []

for old_path in test_df["image_path"]:

    relative = get_relative_path(old_path)

    source = os.path.join(
        DRIVE_IMAGE_ROOT,
        relative
    )

    destination = os.path.join(
        LOCAL_TEST_ROOT,
        relative
    )

    os.makedirs(
        os.path.dirname(destination),
        exist_ok=True
    )

    if os.path.exists(source):

        shutil.copy2(
            source,
            destination
        )

        copied += 1
        local_paths.append(destination)

    else:

        missing += 1
        local_paths.append(destination)


test_df["local_path"] = local_paths

elapsed = (time.time() - start) / 60

print("\n===== TEST COPY COMPLETE =====")
print("Copied:", copied)
print("Missing:", missing)
print(f"Time: {elapsed:.2f} minutes")
print("Ready:", copied == 2280 and missing == 0)

Copying ONLY 2,280 test images...

===== TEST COPY COMPLETE =====
Copied: 2280
Missing: 0
Time: 12.44 minutes
Ready: True


In [ ]:
import tensorflow as tf
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TEST_CSV = os.path.join(WORK_DIR, "fitzpatrick114_test.csv")
LOCAL_TEST_ROOT = "/content/DermaWise_Test_Images"

IMG_SIZE_B2 = 260
BATCH_SIZE_B2 = 16

# Reload official test manifest
test_df = pd.read_csv(TEST_CSV)

# Create the exact same local paths used during copying
def get_relative_path(old_path):
    old_path = str(old_path).replace("\\", "/")
    marker = "Categorized_AbbrvName/"
    return old_path.split(marker, 1)[1]

test_df["local_path"] = test_df["image_path"].apply(
    lambda x: os.path.join(LOCAL_TEST_ROOT, get_relative_path(x))
)

# IMPORTANT: same alphabetical class mapping used during training
class_names = sorted(test_df["label"].unique())

print("Number of classes:", len(class_names))

class_to_index = {
    name: i for i, name in enumerate(class_names)
}

test_df["label_index"] = test_df["label"].map(class_to_index)

# Check everything before evaluation
missing = (~test_df["local_path"].apply(os.path.exists)).sum()

print("Test images:", len(test_df))
print("Missing images:", missing)
print("Classes:", len(class_names))


def load_test_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_B2, IMG_SIZE_B2]
    )

    image = tf.cast(image, tf.float32)

    return image, label


test_ds_b2 = tf.data.Dataset.from_tensor_slices(
    (
        test_df["local_path"].values,
        test_df["label_index"].values
    )
)

test_ds_b2 = (
    test_ds_b2
    .map(load_test_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE_B2)
    .prefetch(tf.data.AUTOTUNE)
)

# Verify one batch
images, labels = next(iter(test_ds_b2))

print("\n===== TEST DATASET READY =====")
print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("✅ READY FOR FINAL B2 COMPARISON")

Number of classes: 114
Test images: 2280
Missing images: 0
Classes: 114

===== TEST DATASET READY =====
Image batch: (16, 260, 260, 3)
Label batch: (16,)
Pixel range: 0.0 to 255.0
✅ READY FOR FINAL B2 COMPARISON


In [ ]:
import tensorflow as tf
import os

CONTINUED_BEST = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_b2_continued_best.keras"
)

print("Loading continued B2 model...")

model_continued = tf.keras.models.load_model(
    CONTINUED_BEST,
    compile=False
)

model_continued.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-6),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        ),
    ]
)

print("Evaluating continued B2 on 2,280 official test images...\n")

results = model_continued.evaluate(
    test_ds_b2,
    verbose=1
)

loss, top1, top3, top5 = results

print("\n====================================")
print("CONTINUED B2 — OFFICIAL TEST RESULTS")
print("====================================")
print(f"Loss  : {loss:.4f}")
print(f"Top-1 : {top1 * 100:.2f}%")
print(f"Top-3 : {top3 * 100:.2f}%")
print(f"Top-5 : {top5 * 100:.2f}%")

print("\nOriginal preserved B2:")
print("Top-1 : 27.46%")
print("Top-3 : 44.47%")
print("Top-5 : 54.12%")

print("\nDifference:")
print(f"Top-1 : {(top1 * 100) - 27.46:+.2f} percentage points")
print(f"Top-3 : {(top3 * 100) - 44.47:+.2f} percentage points")
print(f"Top-5 : {(top5 * 100) - 54.12:+.2f} percentage points")

Loading continued B2 model...
Evaluating continued B2 on 2,280 official test images...

143/143 ━━━━━━━━━━━━━━━━━━━━ 338s 2s/step - accuracy: 0.2768 - loss: 3.2396 - top3: 0.4456 - top5: 0.5434

CONTINUED B2 — OFFICIAL TEST RESULTS
Loss  : 3.2396
Top-1 : 27.68%
Top-3 : 44.56%
Top-5 : 54.34%

Original preserved B2:
Top-1 : 27.46%
Top-3 : 44.47%
Top-5 : 54.12%

Difference:
Top-1 : +0.22 percentage points
Top-3 : +0.09 percentage points
Top-5 : +0.22 percentage points


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

print("Generating predictions for 2,280 test images...")

# Predict
pred_probs = model_continued.predict(
    test_ds_b2,
    verbose=1
)

y_pred = np.argmax(pred_probs, axis=1)

# True labels
y_true = test_df["label_index"].values

print("\nPredictions complete.")
print("True labels:", len(y_true))
print("Predictions:", len(y_pred))


# -----------------------------
# Classification report
# -----------------------------

report = classification_report(
    y_true,
    y_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

print("\n===== OVERALL RESULTS =====")
print(
    f"Accuracy     : {report['accuracy'] * 100:.2f}%"
)
print(
    f"Macro F1     : {report['macro avg']['f1-score']:.4f}"
)
print(
    f"Weighted F1  : {report['weighted avg']['f1-score']:.4f}"
)
print(
    f"Macro Recall : {report['macro avg']['recall']:.4f}"
)


# -----------------------------
# Save full report
# -----------------------------

REPORT_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "fitzpatrick114_b2_final_classification_report.csv"
)

report_df.to_csv(REPORT_PATH)

print("\nSaved report:")
print(REPORT_PATH)


# -----------------------------
# Show weakest classes
# -----------------------------

class_report = report_df.loc[class_names].copy()

weakest = class_report.sort_values(
    "f1-score",
    ascending=True
).head(15)

print("\n===== 15 LOWEST F1 CLASSES =====")
print(
    weakest[
        ["precision", "recall", "f1-score", "support"]
    ].round(4)
)


# -----------------------------
# Show strongest classes
# -----------------------------

strongest = class_report.sort_values(
    "f1-score",
    ascending=False
).head(15)

print("\n===== 15 HIGHEST F1 CLASSES =====")
print(
    strongest[
        ["precision", "recall", "f1-score", "support"]
    ].round(4)
)

Generating predictions for 2,280 test images...
143/143 ━━━━━━━━━━━━━━━━━━━━ 353s 2s/step

Predictions complete.
True labels: 2280
Predictions: 2280

===== OVERALL RESULTS =====
Accuracy     : 27.68%
Macro F1     : 0.2618
Weighted F1  : 0.2680
Macro Recall : 0.3026

Saved report:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick114_b2_final_classification_report.csv

===== 15 LOWEST F1 CLASSES =====
                                    precision  recall  f1-score  support
calcinosis cutis                       0.0000  0.0000    0.0000      9.0
disseminated actinic porokeratosis     0.0000  0.0000    0.0000      4.0
erythema elevatum diutinum             0.0000  0.0000    0.0000      4.0
lichen simplex                         0.0000  0.0000    0.0000     13.0
neurotic excoriations                  0.0000  0.0000    0.0000      9.0
scleroderma                            0.2000  0.0408    0.0678     49.0
photodermatoses                        0.3333  0.0408    0.0727     49.0
str

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(class_names)))
)

confusions = []

for true_idx in range(len(class_names)):
    for pred_idx in range(len(class_names)):

        if true_idx == pred_idx:
            continue

        count = int(cm[true_idx, pred_idx])

        if count > 0:
            confusions.append({
                "True Class": class_names[true_idx],
                "Predicted As": class_names[pred_idx],
                "Count": count
            })

confusions_df = pd.DataFrame(confusions)

confusions_df = confusions_df.sort_values(
    "Count",
    ascending=False
).reset_index(drop=True)

print("===== TOP 30 CONFUSIONS =====")

print(
    confusions_df.head(30).to_string(index=False)
)


# Save ALL confusion pairs
CONFUSION_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "fitzpatrick114_b2_final_confusions.csv"
)

confusions_df.to_csv(
    CONFUSION_PATH,
    index=False
)

print("\nSaved full confusion analysis:")
print(CONFUSION_PATH)


# Extra summary
zero_f1 = class_report[
    class_report["f1-score"] == 0
]

print("\n===== ZERO-F1 CLASSES =====")
print("Number of zero-F1 classes:", len(zero_f1))

for name in zero_f1.index:
    print(
        f"{name} | test images: "
        f"{int(zero_f1.loc[name, 'support'])}"
    )

print("\n✅ ERROR ANALYSIS COMPLETE")

===== TOP 30 CONFUSIONS =====
                        True Class                       Predicted As  Count
                         psoriasis                         paronychia     14
           squamous cell carcinoma               basal cell carcinoma      9
                       scleroderma                       lyme disease      6
superficial spreading melanoma ssm                 malignant melanoma      6
                     lichen planus                  prurigo nodularis      5
       allergic contact dermatitis                       lyme disease      5
                pyogenic granuloma                 granuloma pyogenic      5
                         psoriasis           pityriasis rubra pilaris      5
               lupus erythematosus              seborrheic dermatitis      5
                         psoriasis                 dyshidrotic eczema      5
 solid cystic basal cell carcinoma   basal cell carcinoma morpheiform      4
                 actinic keratosis   basal cel

In [ ]:
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TRAIN_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

train_df = pd.read_csv(TRAIN_CSV)

# Count training images per condition
train_counts = (
    train_df["label"]
    .value_counts()
    .rename("Train Images")
)

# Add final test performance
analysis_df = class_report[
    ["precision", "recall", "f1-score", "support"]
].copy()

analysis_df["Train Images"] = (
    analysis_df.index.map(train_counts)
)

analysis_df = analysis_df.rename(
    columns={
        "precision": "Precision",
        "recall": "Recall",
        "f1-score": "F1",
        "support": "Test Images"
    }
)

analysis_df = analysis_df.sort_values(
    "F1",
    ascending=True
)

print("===== WEAKEST 20 CLASSES + TRAINING SIZE =====")

print(
    analysis_df[
        ["Train Images", "Test Images", "Precision", "Recall", "F1"]
    ]
    .head(20)
    .round(4)
    .to_string()
)

print("\n===== TRAINING DISTRIBUTION =====")
print("Smallest class:", int(train_counts.min()))
print("Largest class :", int(train_counts.max()))
print("Median class  :", float(train_counts.median()))

print("\nClasses with <20 training images:",
      int((train_counts < 20).sum()))

print("Classes with <50 training images:",
      int((train_counts < 50).sum()))

print("Classes with >=100 training images:",
      int((train_counts >= 100).sum()))

print("\n✅ IMBALANCE ANALYSIS COMPLETE")

===== WEAKEST 20 CLASSES + TRAINING SIZE =====
                                    Train Images  Test Images  Precision  Recall      F1
calcinosis cutis                              33          9.0     0.0000  0.0000  0.0000
disseminated actinic porokeratosis            14          4.0     0.0000  0.0000  0.0000
erythema elevatum diutinum                    17          4.0     0.0000  0.0000  0.0000
lichen simplex                                45         13.0     0.0000  0.0000  0.0000
neurotic excoriations                         33          9.0     0.0000  0.0000  0.0000
scleroderma                                  172         49.0     0.2000  0.0408  0.0678
photodermatoses                              169         49.0     0.3333  0.0408  0.0727
striae                                        34          9.0     0.0556  0.1111  0.0741
neurodermatitis                               38         11.0     0.0769  0.0909  0.0833
sarcoidosis                                  180         51.0  

In [ ]:
import numpy as np
import pandas as pd

# Highest predicted probability for every test image
confidence = np.max(pred_probs, axis=1)

correct_mask = (y_pred == y_true)
wrong_mask = ~correct_mask

correct_conf = confidence[correct_mask]
wrong_conf = confidence[wrong_mask]

print("===== CONFIDENCE ANALYSIS =====")

print(f"Correct predictions : {correct_mask.sum()}")
print(f"Wrong predictions   : {wrong_mask.sum()}")

print(
    f"\nAverage confidence when CORRECT : "
    f"{correct_conf.mean()*100:.2f}%"
)

print(
    f"Average confidence when WRONG   : "
    f"{wrong_conf.mean()*100:.2f}%"
)

print("\n===== HIGH-CONFIDENCE ERRORS =====")

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:

    count = np.sum(
        wrong_mask & (confidence >= threshold)
    )

    print(
        f"Wrong predictions >= {threshold*100:.0f}% confidence: "
        f"{count}"
    )


# Top high-confidence mistakes
error_df = pd.DataFrame({
    "True Class": [
        class_names[i] for i in y_true[wrong_mask]
    ],
    "Predicted Class": [
        class_names[i] for i in y_pred[wrong_mask]
    ],
    "Confidence": confidence[wrong_mask]
})

error_df = error_df.sort_values(
    "Confidence",
    ascending=False
)

print("\n===== TOP 20 MOST CONFIDENT WRONG PREDICTIONS =====")

print(
    error_df.head(20).to_string(
        index=False,
        formatters={
            "Confidence": lambda x: f"{x*100:.2f}%"
        }
    )
)

ERROR_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "fitzpatrick114_b2_confidence_errors.csv"
)

error_df.to_csv(
    ERROR_PATH,
    index=False
)

print("\nSaved:")
print(ERROR_PATH)

print("\n✅ CONFIDENCE ANALYSIS COMPLETE")

===== CONFIDENCE ANALYSIS =====
Correct predictions : 631
Wrong predictions   : 1649

Average confidence when CORRECT : 54.88%
Average confidence when WRONG   : 33.87%

===== HIGH-CONFIDENCE ERRORS =====
Wrong predictions >= 50% confidence: 319
Wrong predictions >= 60% confidence: 177
Wrong predictions >= 70% confidence: 88
Wrong predictions >= 80% confidence: 46
Wrong predictions >= 90% confidence: 14

===== TOP 20 MOST CONFIDENT WRONG PREDICTIONS =====
                                           True Class                    Predicted Class Confidence
                                             melanoma                          tungiasis     99.89%
                             porokeratosis of mibelli                 nematode infection     98.78%
                         acrodermatitis enteropathica                         paronychia     97.25%
acquired autoimmune bullous diseaseherpes gestationis                          porphyria     96.33%
                         nevus sebaceous 

In [ ]:
import os
import shutil
import pandas as pd
import time

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

DRIVE_IMAGE_ROOT = os.path.join(
    WORK_DIR,
    "Fitzpatrick17k_C",
    "Categorized_AbbrvName"
)

LOCAL_TRAIN_ROOT = "/content/DermaWise_Train_Images"
LOCAL_VAL_ROOT = "/content/DermaWise_Val_Images"

TRAIN_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

VAL_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_val.csv"
)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)


def get_relative_path(old_path):
    old_path = str(old_path).replace("\\", "/")
    marker = "Categorized_AbbrvName/"
    return old_path.split(marker, 1)[1]


def copy_partition(df, destination_root, name):

    os.makedirs(destination_root, exist_ok=True)

    copied = 0
    missing = 0
    local_paths = []

    print(f"\nCopying {name}...")

    start = time.time()

    for old_path in df["image_path"]:

        relative = get_relative_path(old_path)

        source = os.path.join(
            DRIVE_IMAGE_ROOT,
            relative
        )

        destination = os.path.join(
            destination_root,
            relative
        )

        os.makedirs(
            os.path.dirname(destination),
            exist_ok=True
        )

        if os.path.exists(source):

            if not os.path.exists(destination):
                shutil.copy2(source, destination)

            copied += 1
            local_paths.append(destination)

        else:

            missing += 1
            local_paths.append(destination)

    elapsed = (time.time() - start) / 60

    print(f"{name} images: {len(df)}")
    print("Available:", copied)
    print("Missing:", missing)
    print(f"Time: {elapsed:.2f} minutes")

    return local_paths, copied, missing


train_paths, train_found, train_missing = copy_partition(
    train_df,
    LOCAL_TRAIN_ROOT,
    "TRAIN"
)

val_paths, val_found, val_missing = copy_partition(
    val_df,
    LOCAL_VAL_ROOT,
    "VALIDATION"
)

train_df["local_path"] = train_paths
val_df["local_path"] = val_paths


print("\n================================")
print("DERMAWISE TRAINING DATA STATUS")
print("================================")

print("Train:", train_found, "/", len(train_df))
print("Validation:", val_found, "/", len(val_df))
print("Test: 2280 / 2280")

READY = (
    train_missing == 0
    and val_missing == 0
    and train_found == 7975
    and val_found == 1139
)

print("\nReady for focal experiment:", READY)

if READY:
    print("✅ ALL REQUIRED DATA READY")
else:
    print("❌ STOP — DATASET CHECK FAILED")


Copying TRAIN...
TRAIN images: 7975
Available: 7975
Missing: 0
Time: 41.63 minutes

Copying VALIDATION...
VALIDATION images: 1139
Available: 1139
Missing: 0
Time: 6.23 minutes

DERMAWISE TRAINING DATA STATUS
Train: 7975 / 7975
Validation: 1139 / 1139
Test: 2280 / 2280

Ready for focal experiment: True
✅ ALL REQUIRED DATA READY


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os

# ==========================================
# SETTINGS
# ==========================================

IMG_SIZE_B2 = 260
BATCH_SIZE_B2 = 16
NUM_CLASSES = 114

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TRAIN_CSV = os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
VAL_CSV   = os.path.join(WORK_DIR, "fitzpatrick114_val.csv")

LOCAL_TRAIN_ROOT = "/content/DermaWise_Train_Images"
LOCAL_VAL_ROOT   = "/content/DermaWise_Val_Images"


# ==========================================
# LOAD MANIFESTS
# ==========================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)


def get_relative_path(old_path):
    old_path = str(old_path).replace("\\", "/")
    marker = "Categorized_AbbrvName/"
    return old_path.split(marker, 1)[1]


train_df["local_path"] = train_df["image_path"].apply(
    lambda x: os.path.join(
        LOCAL_TRAIN_ROOT,
        get_relative_path(x)
    )
)

val_df["local_path"] = val_df["image_path"].apply(
    lambda x: os.path.join(
        LOCAL_VAL_ROOT,
        get_relative_path(x)
    )
)


# ==========================================
# CLASS MAPPING
# ==========================================

# IMPORTANT:
# Same alphabetical mapping used previously.
class_names_focal = sorted(train_df["label"].unique())

class_to_index_focal = {
    name: i
    for i, name in enumerate(class_names_focal)
}

train_df["label_index"] = train_df["label"].map(
    class_to_index_focal
)

val_df["label_index"] = val_df["label"].map(
    class_to_index_focal
)


# ==========================================
# SAFETY CHECKS
# ==========================================

missing_train = (
    ~train_df["local_path"].apply(os.path.exists)
).sum()

missing_val = (
    ~val_df["local_path"].apply(os.path.exists)
).sum()

print("===== DATA CHECK =====")
print("Train images:", len(train_df))
print("Validation images:", len(val_df))
print("Classes:", len(class_names_focal))
print("Missing train:", missing_train)
print("Missing validation:", missing_val)

# Compare against mapping already used for test evaluation
print(
    "Class mapping identical to test:",
    class_names_focal == class_names
)


# ==========================================
# IMAGE LOADER
# ==========================================

def load_image_focal(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_B2, IMG_SIZE_B2]
    )

    # EfficientNet receives 0–255 float input
    image = tf.cast(image, tf.float32)

    return image, label


# ==========================================
# BUILD DATASETS
# ==========================================

train_ds_focal = tf.data.Dataset.from_tensor_slices(
    (
        train_df["local_path"].values,
        train_df["label_index"].values
    )
)

train_ds_focal = (
    train_ds_focal
    .shuffle(
        buffer_size=len(train_df),
        reshuffle_each_iteration=True
    )
    .map(
        load_image_focal,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE_B2)
    .prefetch(tf.data.AUTOTUNE)
)


val_ds_focal = tf.data.Dataset.from_tensor_slices(
    (
        val_df["local_path"].values,
        val_df["label_index"].values
    )
)

val_ds_focal = (
    val_ds_focal
    .map(
        load_image_focal,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE_B2)
    .prefetch(tf.data.AUTOTUNE)
)


# ==========================================
# FINAL VERIFICATION
# ==========================================

images, labels = next(iter(train_ds_focal))

print("\n===== DATASET VERIFICATION =====")
print("Train batch:", images.shape)
print("Label batch:", labels.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print(
    "Label range:",
    int(tf.reduce_min(labels)),
    "to",
    int(tf.reduce_max(labels))
)

READY = (
    len(train_df) == 7975
    and len(val_df) == 1139
    and len(class_names_focal) == 114
    and missing_train == 0
    and missing_val == 0
    and class_names_focal == class_names
)

print("\nReady:", READY)

if READY:
    print("✅ DATA PIPELINE SAFE FOR FOCAL EXPERIMENT")
else:
    print("❌ STOP — DO NOT START TRAINING")

===== DATA CHECK =====
Train images: 7975
Validation images: 1139
Classes: 114
Missing train: 0
Missing validation: 0
Class mapping identical to test: True

===== DATASET VERIFICATION =====
Train batch: (16, 260, 260, 3)
Label batch: (16,)
Pixel range: 0.0 to 255.0
Label range: 2 to 106

Ready: True
✅ DATA PIPELINE SAFE FOR FOCAL EXPERIMENT


In [ ]:
import tensorflow as tf
import os
import shutil

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# Current strongest model — DO NOT overwrite
SOURCE_MODEL = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_continued_best.keras"
)

# New experiment output
FOCAL_BEST = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_focal_best.keras"
)

FOCAL_BACKUP = os.path.join(
    WORK_DIR,
    "b2_focal_training_backup"
)

print("Source model exists:", os.path.exists(SOURCE_MODEL))

# ==========================================
# LOAD CURRENT BEST MODEL
# ==========================================

model_focal = tf.keras.models.load_model(
    SOURCE_MODEL,
    compile=False
)

print("Loaded:", model_focal.name)


# ==========================================
# FIND EFFICIENTNET BACKBONE
# ==========================================

base_model = None

for layer in model_focal.layers:
    if "efficientnet" in layer.name.lower():
        base_model = layer
        break

if base_model is None:
    raise ValueError("EfficientNet backbone not found.")

print("Backbone:", base_model.name)


# ==========================================
# CONTROLLED FINE-TUNING
# ==========================================

base_model.trainable = True

# Freeze entire backbone first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 40 eligible layers
unfrozen = 0

for layer in reversed(base_model.layers):

    if unfrozen >= 40:
        break

    # Keep BatchNorm frozen
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        continue

    layer.trainable = True
    unfrozen += 1

print("Unfrozen non-BN backbone layers:", unfrozen)


# ==========================================
# FOCAL LOSS
# ==========================================

# Multiclass sparse focal loss.
# gamma=2 focuses more strongly on difficult examples.
def sparse_focal_loss(gamma=2.0):

    def loss(y_true, y_pred):

        y_true = tf.cast(
            tf.reshape(y_true, [-1]),
            tf.int32
        )

        y_pred = tf.clip_by_value(
            y_pred,
            tf.keras.backend.epsilon(),
            1.0 - tf.keras.backend.epsilon()
        )

        # Probability assigned to the correct class
        indices = tf.stack(
            [
                tf.range(tf.shape(y_true)[0]),
                y_true
            ],
            axis=1
        )

        p_t = tf.gather_nd(
            y_pred,
            indices
        )

        focal_factor = tf.pow(
            1.0 - p_t,
            gamma
        )

        ce = -tf.math.log(p_t)

        return tf.reduce_mean(
            focal_factor * ce
        )

    return loss


# ==========================================
# COMPILE
# ==========================================

model_focal.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-6
    ),

    loss=sparse_focal_loss(
        gamma=2.0
    ),

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)


# ==========================================
# CALLBACKS
# ==========================================

callbacks_focal = [

    tf.keras.callbacks.ModelCheckpoint(
        FOCAL_BEST,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=FOCAL_BACKUP
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]


# ==========================================
# SAFETY CHECK
# ==========================================

trainable_params = sum(
    tf.keras.backend.count_params(w)
    for w in model_focal.trainable_weights
)

print("\n===== FOCAL EXPERIMENT =====")
print("Input model:", SOURCE_MODEL)
print("Output model:", FOCAL_BEST)
print("Learning rate: 1e-6")
print("Focal gamma: 2.0")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nOriginal 27.68% model remains untouched.")

print("\n✅ SAFE TO START FOCAL TRAINING")

Source model exists: True
Loaded: DermaWise_Fitzpatrick114_B2
Backbone: efficientnetb2
Unfrozen non-BN backbone layers: 40

===== FOCAL EXPERIMENT =====
Input model: /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_continued_best.keras
Output model: /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_focal_best.keras
Learning rate: 1e-6
Focal gamma: 2.0
Trainable parameters: 4,318,258

Original 27.68% model remains untouched.

✅ SAFE TO START FOCAL TRAINING


In [ ]:
print("Starting B2 focal-loss fine-tuning...")
print("Maximum epochs: 10")
print("Early stopping patience: 4")
print("Current baseline Top-1: 27.68%")
print("=" * 60)

history_focal = model_focal.fit(
    train_ds_focal,
    validation_data=val_ds_focal,

    epochs=10,

    callbacks=callbacks_focal,

    verbose=1
)

print("\n======================================")
print("✅ FOCAL TRAINING FINISHED")
print("======================================")

best_val_acc = max(
    history_focal.history["val_accuracy"]
)

best_val_top3 = max(
    history_focal.history["val_top3"]
)

best_val_top5 = max(
    history_focal.history["val_top5"]
)

best_val_loss = min(
    history_focal.history["val_loss"]
)

print(f"Best Val Accuracy : {best_val_acc*100:.2f}%")
print(f"Best Val Top-3    : {best_val_top3*100:.2f}%")
print(f"Best Val Top-5    : {best_val_top5*100:.2f}%")
print(f"Best Val Loss     : {best_val_loss:.4f}")

print("\nBest checkpoint:")
print(FOCAL_BEST)

Starting B2 focal-loss fine-tuning...
Maximum epochs: 10
Early stopping patience: 4
Current baseline Top-1: 27.68%
Epoch 1/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4769 - loss: 1.7437 - top3: 0.6621 - top5: 0.7483
Epoch 1: val_loss improved from None to 2.93592, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_focal_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_focal_best.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 1818s 4s/step - accuracy: 0.4819 - loss: 1.7115 - top3: 0.6713 - top5: 0.7521 - val_accuracy: 0.2924 - val_loss: 2.9359 - val_top3: 0.4557 - val_top5: 0.5487 - learning_rate: 1.0000e-06
Epoch 2/10
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4977 - loss: 1.6552 - top3: 0.6844 - top5: 0.7676
Epoch 2: val_loss improved from 2.93592 to 2.90058, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_b2_focal_best

In [ ]:
import tensorflow as tf
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

FOCAL_BEST = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_focal_best.keras"
)

print("Focal checkpoint exists:", os.path.exists(FOCAL_BEST))

print("\nLoading best focal model...")

# We used a custom focal loss during training,
# so compile=False avoids serialization issues.
focal_best_model = tf.keras.models.load_model(
    FOCAL_BEST,
    compile=False
)

# Compile with ordinary CE ONLY for evaluation.
# This does NOT modify the trained weights.
focal_best_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

print("\nEvaluating on 2,280 official test images...")

results = focal_best_model.evaluate(
    test_ds_b2,
    verbose=1
)

loss, top1, top3, top5 = results

print("\n======================================")
print("FOCAL B2 — OFFICIAL TEST RESULTS")
print("======================================")

print(f"Loss  : {loss:.4f}")
print(f"Top-1 : {top1*100:.2f}%")
print(f"Top-3 : {top3*100:.2f}%")
print(f"Top-5 : {top5*100:.2f}%")

print("\n======================================")
print("COMPARISON WITH CURRENT BEST")
print("======================================")

print("Current B2:")
print("Top-1 : 27.68%")
print("Top-3 : 44.56%")
print("Top-5 : 54.34%")

print("\nFocal B2:")
print(f"Top-1 : {top1*100:.2f}%")
print(f"Top-3 : {top3*100:.2f}%")
print(f"Top-5 : {top5*100:.2f}%")

print("\nChange:")
print(f"Top-1 : {top1*100 - 27.68:+.2f} percentage points")
print(f"Top-3 : {top3*100 - 44.56:+.2f} percentage points")
print(f"Top-5 : {top5*100 - 54.34:+.2f} percentage points")

Focal checkpoint exists: True

Loading best focal model...

Evaluating on 2,280 official test images...
143/143 ━━━━━━━━━━━━━━━━━━━━ 327s 2s/step - accuracy: 0.2895 - loss: 3.0772 - top3: 0.4706 - top5: 0.5667

FOCAL B2 — OFFICIAL TEST RESULTS
Loss  : 3.0772
Top-1 : 28.95%
Top-3 : 47.06%
Top-5 : 56.67%

COMPARISON WITH CURRENT BEST
Current B2:
Top-1 : 27.68%
Top-3 : 44.56%
Top-5 : 54.34%

Focal B2:
Top-1 : 28.95%
Top-3 : 47.06%
Top-5 : 56.67%

Change:
Top-1 : +1.27 percentage points
Top-3 : +2.50 percentage points
Top-5 : +2.33 percentage points


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

print("Generating focal-model predictions...")

focal_probs = focal_best_model.predict(
    test_ds_b2,
    verbose=1
)

focal_pred = np.argmax(focal_probs, axis=1)

# Same official true labels
focal_true = test_df["label_index"].values


# ==========================================
# CLASSIFICATION REPORT
# ==========================================

focal_report = classification_report(
    focal_true,
    focal_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

focal_report_df = pd.DataFrame(
    focal_report
).transpose()


print("\n===== FOCAL MODEL OVERALL =====")

print(
    f"Accuracy     : {focal_report['accuracy']*100:.2f}%"
)

print(
    f"Macro F1     : "
    f"{focal_report['macro avg']['f1-score']:.4f}"
)

print(
    f"Weighted F1  : "
    f"{focal_report['weighted avg']['f1-score']:.4f}"
)

print(
    f"Macro Recall : "
    f"{focal_report['macro avg']['recall']:.4f}"
)


# ==========================================
# SAVE REPORT
# ==========================================

REPORT_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_b2_focal_classification_report.csv"
)

focal_report_df.to_csv(REPORT_PATH)

print("\nSaved:")
print(REPORT_PATH)


# ==========================================
# CLASS-LEVEL ANALYSIS
# ==========================================

focal_classes = focal_report_df.loc[
    class_names
].copy()

weakest = focal_classes.sort_values(
    "f1-score"
).head(15)

strongest = focal_classes.sort_values(
    "f1-score",
    ascending=False
).head(15)


print("\n===== 15 LOWEST F1 CLASSES =====")

print(
    weakest[
        ["precision", "recall", "f1-score", "support"]
    ].round(4).to_string()
)


print("\n===== 15 HIGHEST F1 CLASSES =====")

print(
    strongest[
        ["precision", "recall", "f1-score", "support"]
    ].round(4).to_string()
)


# ==========================================
# ZERO-F1 COUNT
# ==========================================

zero_f1 = focal_classes[
    focal_classes["f1-score"] == 0
]

print("\n===== ZERO-F1 CLASSES =====")
print("Number:", len(zero_f1))

for cls in zero_f1.index:
    print(
        cls,
        "| test images:",
        int(zero_f1.loc[cls, "support"])
    )


print("\n✅ FOCAL ERROR ANALYSIS COMPLETE")

Generating focal-model predictions...
143/143 ━━━━━━━━━━━━━━━━━━━━ 333s 2s/step

===== FOCAL MODEL OVERALL =====
Accuracy     : 28.95%
Macro F1     : 0.2722
Weighted F1  : 0.2767
Macro Recall : 0.2792

Saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick114_b2_focal_classification_report.csv

===== 15 LOWEST F1 CLASSES =====
                                    precision  recall  f1-score  support
calcinosis cutis                       0.0000  0.0000    0.0000      9.0
behcets disease                        0.0000  0.0000    0.0000     10.0
erythema elevatum diutinum             0.0000  0.0000    0.0000      4.0
disseminated actinic porokeratosis     0.0000  0.0000    0.0000      4.0
lichen simplex                         0.0000  0.0000    0.0000     13.0
livedo reticularis                     0.0000  0.0000    0.0000      9.0
langerhans cell histiocytosis          0.0000  0.0000    0.0000      7.0
neurodermatitis                        0.0000  0.0000    0.0000     11.0
ne

In [ ]:
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

comparison = pd.DataFrame([
    {
        "model": "EfficientNetB0 Stage2",
        "top1": 24.96,
        "top3": 42.59,
        "top5": 51.58,
        "macro_f1": 0.2332,
        "weighted_f1": 0.2392,
        "macro_recall": 0.2762
    },
    {
        "model": "EfficientNetB2 Continued",
        "top1": 27.68,
        "top3": 44.56,
        "top5": 54.34,
        "macro_f1": 0.2618,
        "weighted_f1": 0.2680,
        "macro_recall": 0.3026
    },
    {
        "model": "EfficientNetB2 + Focal Loss",
        "top1": 28.95,
        "top3": 47.06,
        "top5": 56.67,
        "macro_f1": 0.2722,
        "weighted_f1": 0.2767,
        "macro_recall": 0.2792
    }
])

SAVE_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_model_comparison.csv"
)

comparison.to_csv(SAVE_PATH, index=False)

print(comparison.to_string(index=False))
print("\nSaved:", SAVE_PATH)
print("\n✅ Experiment comparison preserved.")

                      model  top1  top3  top5  macro_f1  weighted_f1  macro_recall
      EfficientNetB0 Stage2 24.96 42.59 51.58    0.2332       0.2392        0.2762
   EfficientNetB2 Continued 27.68 44.56 54.34    0.2618       0.2680        0.3026
EfficientNetB2 + Focal Loss 28.95 47.06 56.67    0.2722       0.2767        0.2792

Saved: /content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick114_model_comparison.csv

✅ Experiment comparison preserved.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

files_to_check = [
    "dermawise_fitzpatrick114_b2_continued_best.keras",
    "dermawise_fitzpatrick114_b2_focal_best.keras",
    "fitzpatrick114_b2_focal_classification_report.csv",
    "fitzpatrick114_model_comparison.csv",
    "fitzpatrick114_train.csv",
    "fitzpatrick114_val.csv",
    "fitzpatrick114_test.csv",
]

print("\n===== DERMAWISE RECOVERY CHECK =====")

for filename in files_to_check:
    path = os.path.join(WORK_DIR, filename)

    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 ** 2)
        print(f"✅ {filename} | {size:.2f} MB")
    else:
        print(f"❌ MISSING: {filename}")

print("\nRecovery check complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== DERMAWISE RECOVERY CHECK =====
✅ dermawise_fitzpatrick114_b2_continued_best.keras | 60.15 MB
✅ dermawise_fitzpatrick114_b2_focal_best.keras | 64.35 MB
✅ fitzpatrick114_b2_focal_classification_report.csv | 0.01 MB
✅ fitzpatrick114_model_comparison.csv | 0.00 MB
✅ fitzpatrick114_train.csv | 3.84 MB
✅ fitzpatrick114_val.csv | 0.55 MB
✅ fitzpatrick114_test.csv | 1.09 MB

Recovery check complete.


In [ ]:
import os
import tensorflow as tf

print("===== RUNTIME CHECK =====")

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

LOCAL_TRAIN = "/content/DermaWise_Train_Images"
LOCAL_VAL   = "/content/DermaWise_Val_Images"
LOCAL_TEST  = "/content/DermaWise_Test_Images"

print("\nLocal image folders:")
print("Train:", os.path.exists(LOCAL_TRAIN))
print("Val  :", os.path.exists(LOCAL_VAL))
print("Test :", os.path.exists(LOCAL_TEST))

print("\n===== SAVED FOCAL MODEL =====")

FOCAL_MODEL = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_focal_best.keras"
)

print("Exists:", os.path.exists(FOCAL_MODEL))

if os.path.exists(FOCAL_MODEL):
    print(
        "Size:",
        round(os.path.getsize(FOCAL_MODEL) / (1024**2), 2),
        "MB"
    )

print("\n✅ Runtime check complete.")

===== RUNTIME CHECK =====
TensorFlow: 2.20.0
GPU: []

Local image folders:
Train: False
Val  : False
Test : False

===== SAVED FOCAL MODEL =====
Exists: True
Size: 64.35 MB

✅ Runtime check complete.


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TRAIN_CSV = os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
VAL_CSV   = os.path.join(WORK_DIR, "fitzpatrick114_val.csv")
TEST_CSV  = os.path.join(WORK_DIR, "fitzpatrick114_test.csv")

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("===== OFFICIAL PARTITIONS =====")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nClasses:")
print("Train:", train_df["label"].nunique())
print("Val  :", val_df["label"].nunique())
print("Test :", test_df["label"].nunique())

# Original extracted Fitzpatrick17k-C images on Drive
DRIVE_IMAGE_ROOT = os.path.join(
    WORK_DIR,
    "Fitzpatrick17k_C",
    "Categorized_AbbrvName"
)

print("\n===== IMAGE SOURCE CHECK =====")
print("Drive image root:")
print(DRIVE_IMAGE_ROOT)

print("\nExists:", os.path.isdir(DRIVE_IMAGE_ROOT))

if os.path.isdir(DRIVE_IMAGE_ROOT):
    image_count = 0

    for root, dirs, files in os.walk(DRIVE_IMAGE_ROOT):
        image_count += sum(
            f.lower().endswith((".jpg", ".jpeg", ".png"))
            for f in files
        )

    print("Images found:", image_count)

print("\nFirst manifest image path:")
print(train_df["image_path"].iloc[0])

print("\n✅ Manifest/source check complete.")

===== OFFICIAL PARTITIONS =====
Train: 7975
Val  : 1139
Test : 2280

Classes:
Train: 114
Val  : 114
Test : 114

===== IMAGE SOURCE CHECK =====
Drive image root:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C/Categorized_AbbrvName

Exists: True
Images found: 16577

First manifest image path:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C/Categorized_AbbrvName/ne-de_361/ne-de_f2_61_3a96728f.jpg

✅ Manifest/source check complete.


In [ ]:
import os
import shutil
from tqdm.auto import tqdm

LOCAL_TRAIN = "/content/DermaWise_Train_Images"
LOCAL_VAL   = "/content/DermaWise_Val_Images"

os.makedirs(LOCAL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_VAL, exist_ok=True)


def copy_partition(df, destination, name):

    copied = 0
    existing = 0
    missing = 0

    print(f"\n===== COPYING {name} =====")

    for source in tqdm(df["image_path"].astype(str)):

        if not os.path.isfile(source):
            missing += 1
            continue

        # Preserve class-folder name
        class_folder = os.path.basename(
            os.path.dirname(source)
        )

        target_folder = os.path.join(
            destination,
            class_folder
        )

        os.makedirs(target_folder, exist_ok=True)

        target = os.path.join(
            target_folder,
            os.path.basename(source)
        )

        if os.path.isfile(target):
            existing += 1
        else:
            shutil.copy2(source, target)
            copied += 1

    print(f"\n{name} complete")
    print("Copied :", copied)
    print("Already:", existing)
    print("Missing:", missing)


copy_partition(
    train_df,
    LOCAL_TRAIN,
    "TRAIN"
)

copy_partition(
    val_df,
    LOCAL_VAL,
    "VALIDATION"
)


print("\n======================================")
print("✅ LOCAL DATA PREPARATION COMPLETE")
print("======================================")


===== COPYING TRAIN =====


  0%|          | 0/7975 [00:00<?, ?it/s]


TRAIN complete
Copied : 7975
Already: 0
Missing: 0

===== COPYING VALIDATION =====


  0%|          | 0/1139 [00:00<?, ?it/s]


VALIDATION complete
Copied : 1139
Already: 0
Missing: 0

✅ LOCAL DATA PREPARATION COMPLETE


In [ ]:
import os
import pandas as pd
import tensorflow as tf

IMG_SIZE = 260
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# -------------------------------------------------
# 1. Fixed 114-class mapping
# -------------------------------------------------

class_names = sorted(train_df["label"].unique().tolist())
class_to_index = {
    name: i for i, name in enumerate(class_names)
}

print("Number of classes:", len(class_names))

# -------------------------------------------------
# 2. Convert Drive paths -> local paths
# -------------------------------------------------

def make_local_path(original_path, local_root):
    class_folder = os.path.basename(
        os.path.dirname(original_path)
    )

    filename = os.path.basename(original_path)

    return os.path.join(
        local_root,
        class_folder,
        filename
    )


train_df["local_path"] = train_df["image_path"].apply(
    lambda x: make_local_path(x, LOCAL_TRAIN)
)

val_df["local_path"] = val_df["image_path"].apply(
    lambda x: make_local_path(x, LOCAL_VAL)
)

train_df["class_id"] = train_df["label"].map(class_to_index)
val_df["class_id"] = val_df["label"].map(class_to_index)

# -------------------------------------------------
# 3. Verify files
# -------------------------------------------------

train_missing = sum(
    not os.path.isfile(p)
    for p in train_df["local_path"]
)

val_missing = sum(
    not os.path.isfile(p)
    for p in val_df["local_path"]
)

print("\n===== FILE CHECK =====")
print("Train images :", len(train_df))
print("Train missing:", train_missing)

print("Val images   :", len(val_df))
print("Val missing  :", val_missing)

# -------------------------------------------------
# 4. Image loader
# -------------------------------------------------

def load_image(path, label):

    image = tf.io.read_file(path)
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    # IMPORTANT:
    # Keep 0-255 range because EfficientNetB2 handles
    # preprocessing internally.
    image = tf.cast(image, tf.float32)

    return image, label


# -------------------------------------------------
# 5. Build datasets
# -------------------------------------------------

train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["local_path"].values,
        train_df["class_id"].values
    )
)

train_ds = (
    train_ds
    .shuffle(
        len(train_df),
        reshuffle_each_iteration=True
    )
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["local_path"].values,
        val_df["class_id"].values
    )
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# -------------------------------------------------
# 6. Final sanity check
# -------------------------------------------------

images, labels = next(iter(train_ds))

print("\n===== DATASET CHECK =====")
print("Image batch:", images.shape)
print("Label batch:", labels.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("\nFirst 5 classes:")
print(class_names[:5])

print("\n✅ 114-class datasets ready.")

Number of classes: 114

===== FILE CHECK =====
Train images : 7975
Train missing: 0
Val images   : 1139
Val missing  : 0

===== DATASET CHECK =====
Image batch: (16, 260, 260, 3)
Label batch: (16,)
Pixel range: 0.0 to 255.0

First 5 classes:
['acanthosis nigricans', 'acne', 'acne vulgaris', 'acquired autoimmune bullous diseaseherpes gestationis', 'acrodermatitis enteropathica']

✅ 114-class datasets ready.


In [ ]:
import os
import numpy as np
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

BASELINE_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_continued_best.keras"
)

FOCAL_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_b2_focal_best.keras"
)

print("===== MODEL CHECK =====")
print("Baseline:", os.path.exists(BASELINE_PATH))
print("Focal   :", os.path.exists(FOCAL_PATH))

# -------------------------------------------------
# Focal loss definition needed to load focal model
# -------------------------------------------------

@tf.keras.utils.register_keras_serializable()
def sparse_categorical_focal_loss(y_true, y_pred, gamma=2.0):

    y_true = tf.cast(y_true, tf.int32)
    y_true = tf.reshape(y_true, [-1])

    y_pred = tf.clip_by_value(
        y_pred,
        tf.keras.backend.epsilon(),
        1.0 - tf.keras.backend.epsilon()
    )

    indices = tf.stack(
        [tf.range(tf.shape(y_true)[0]), y_true],
        axis=1
    )

    p_t = tf.gather_nd(y_pred, indices)

    loss = (
        tf.pow(1.0 - p_t, gamma)
        * (-tf.math.log(p_t))
    )

    return tf.reduce_mean(loss)


print("\nLoading baseline model...")

baseline_model = tf.keras.models.load_model(
    BASELINE_PATH,
    compile=False
)

print("✅ Baseline loaded")

print("\nLoading focal model...")

focal_model = tf.keras.models.load_model(
    FOCAL_PATH,
    custom_objects={
        "sparse_categorical_focal_loss":
            sparse_categorical_focal_loss
    },
    compile=False
)

print("✅ Focal loaded")

# -------------------------------------------------
# Get validation labels
# -------------------------------------------------

y_true = np.concatenate([
    y.numpy()
    for _, y in val_ds
])

print("\nValidation samples:", len(y_true))

# -------------------------------------------------
# Predictions
# -------------------------------------------------

print("\nPredicting with baseline B2...")

baseline_probs = baseline_model.predict(
    val_ds,
    verbose=1
)

print("\nPredicting with focal B2...")

focal_probs = focal_model.predict(
    val_ds,
    verbose=1
)

print("\nPrediction shapes:")
print("Baseline:", baseline_probs.shape)
print("Focal   :", focal_probs.shape)

# -------------------------------------------------
# Equal 50/50 ensemble
# -------------------------------------------------

ensemble_probs = (
    0.50 * baseline_probs +
    0.50 * focal_probs
)


# -------------------------------------------------
# Metrics
# -------------------------------------------------

def calculate_metrics(probs, labels):

    top1_pred = np.argmax(probs, axis=1)

    top1 = np.mean(
        top1_pred == labels
    )

    top3_idx = np.argsort(
        probs,
        axis=1
    )[:, -3:]

    top3 = np.mean([
        labels[i] in top3_idx[i]
        for i in range(len(labels))
    ])

    top5_idx = np.argsort(
        probs,
        axis=1
    )[:, -5:]

    top5 = np.mean([
        labels[i] in top5_idx[i]
        for i in range(len(labels))
    ])

    return top1, top3, top5


baseline_metrics = calculate_metrics(
    baseline_probs,
    y_true
)

focal_metrics = calculate_metrics(
    focal_probs,
    y_true
)

ensemble_metrics = calculate_metrics(
    ensemble_probs,
    y_true
)


print("\n========================================")
print("       VALIDATION COMPARISON")
print("========================================")

print(
    f"Baseline B2 : "
    f"Top1 {baseline_metrics[0]*100:.2f}% | "
    f"Top3 {baseline_metrics[1]*100:.2f}% | "
    f"Top5 {baseline_metrics[2]*100:.2f}%"
)

print(
    f"Focal B2    : "
    f"Top1 {focal_metrics[0]*100:.2f}% | "
    f"Top3 {focal_metrics[1]*100:.2f}% | "
    f"Top5 {focal_metrics[2]*100:.2f}%"
)

print(
    f"50/50 Ensemble: "
    f"Top1 {ensemble_metrics[0]*100:.2f}% | "
    f"Top3 {ensemble_metrics[1]*100:.2f}% | "
    f"Top5 {ensemble_metrics[2]*100:.2f}%"
)

print("========================================")

===== MODEL CHECK =====
Baseline: True
Focal   : True

Loading baseline model...
✅ Baseline loaded

Loading focal model...
✅ Focal loaded

Validation samples: 1139

Predicting with baseline B2...
72/72 ━━━━━━━━━━━━━━━━━━━━ 165s 2s/step

Predicting with focal B2...
72/72 ━━━━━━━━━━━━━━━━━━━━ 155s 2s/step

Prediction shapes:
Baseline: (1139, 114)
Focal   : (1139, 114)

       VALIDATION COMPARISON
Baseline B2 : Top1 28.27% | Top3 45.57% | Top5 53.56%
Focal B2    : Top1 31.08% | Top3 48.02% | Top5 55.84%
50/50 Ensemble: Top1 29.94% | Top3 46.36% | Top5 55.40%


In [ ]:
import tensorflow as tf

NUM_CLASSES = 114
IMG_SIZE_CNX = 224

tf.keras.backend.clear_session()

augmentation_cnx = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="augmentation")

print("Loading ConvNeXtTiny...")

backbone_cnx = tf.keras.applications.ConvNeXtTiny(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE_CNX, IMG_SIZE_CNX, 3),
    include_preprocessing=True
)

backbone_cnx.trainable = False

inputs = tf.keras.Input(
    shape=(IMG_SIZE_CNX, IMG_SIZE_CNX, 3)
)

x = augmentation_cnx(inputs)

x = backbone_cnx(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.35)(x)

outputs = tf.keras.layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

convnext_model = tf.keras.Model(
    inputs,
    outputs,
    name="DermaWise_Fitzpatrick114_ConvNeXtTiny"
)

convnext_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

print("\n===== CONVNEXT CHECK =====")
print("Model:", convnext_model.name)
print("Input:", convnext_model.input_shape)
print("Output:", convnext_model.output_shape)

print(
    "Total parameters:",
    f"{convnext_model.count_params():,}"
)

print(
    "Trainable parameters:",
    f"{sum(tf.keras.backend.count_params(w) for w in convnext_model.trainable_weights):,}"
)

print("\n✅ ConvNeXtTiny initialized successfully.")

Loading ConvNeXtTiny...
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

===== CONVNEXT CHECK =====
Model: DermaWise_Fitzpatrick114_ConvNeXtTiny
Input: (None, 224, 224, 3)
Output: (None, 114)
Total parameters: 27,907,794
Trainable parameters: 87,666

✅ ConvNeXtTiny initialized successfully.


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE_CNX = 224
BATCH_SIZE_CNX = 16
AUTOTUNE = tf.data.AUTOTUNE

# -----------------------------------------
# Image loader for ConvNeXt
# -----------------------------------------

def load_image_cnx(path, label):
    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_CNX, IMG_SIZE_CNX]
    )

    # ConvNeXt include_preprocessing=True
    # expects images in the 0-255 range.
    image = tf.cast(image, tf.float32)

    return image, label


# -----------------------------------------
# Training dataset
# -----------------------------------------

train_ds_cnx = tf.data.Dataset.from_tensor_slices(
    (
        train_df["local_path"].values,
        train_df["class_id"].values
    )
)

train_ds_cnx = (
    train_ds_cnx
    .shuffle(
        len(train_df),
        reshuffle_each_iteration=True
    )
    .map(load_image_cnx, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE_CNX)
    .prefetch(AUTOTUNE)
)


# -----------------------------------------
# Validation dataset
# -----------------------------------------

val_ds_cnx = tf.data.Dataset.from_tensor_slices(
    (
        val_df["local_path"].values,
        val_df["class_id"].values
    )
)

val_ds_cnx = (
    val_ds_cnx
    .map(load_image_cnx, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE_CNX)
    .prefetch(AUTOTUNE)
)


# -----------------------------------------
# Balanced class weights
# -----------------------------------------

train_labels = train_df["class_id"].values

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)

class_weights_cnx = {
    i: float(weights[i])
    for i in range(NUM_CLASSES)
}


# -----------------------------------------
# Sanity check
# -----------------------------------------

images, labels = next(iter(train_ds_cnx))

print("===== CONVNEXT DATA CHECK =====")

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Classes:", NUM_CLASSES)

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("\nClass weights:", len(class_weights_cnx))

print(
    "Weight range:",
    round(min(class_weights_cnx.values()), 4),
    "to",
    round(max(class_weights_cnx.values()), 4)
)

print("\n✅ ConvNeXt datasets ready.")

===== CONVNEXT DATA CHECK =====
Train: 7975
Val  : 1139
Classes: 114

Batch image shape: (16, 224, 224, 3)
Batch label shape: (16,)
Pixel range: 0.0 to 255.0

Class weights: 114
Weight range: 0.187 to 4.9969

✅ ConvNeXt datasets ready.


In [ ]:
import gc
import tensorflow as tf

try:
    del convnext_model
except:
    pass

try:
    del backbone_cnx
except:
    pass

gc.collect()
tf.keras.backend.clear_session()

print("✅ Old ConvNeXt execution cleared.")
print("GPU:", tf.config.list_physical_devices("GPU"))

✅ Old ConvNeXt execution cleared.
GPU: []


In [ ]:
import tensorflow as tf
import time

IMG_SIZE_CNX = 224
BATCH_SIZE_CNX = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image_cnx_fast(path, label):
    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_CNX, IMG_SIZE_CNX],
        antialias=False
    )

    image = tf.cast(image, tf.float32)

    return image, label


# TRAIN
train_ds_cnx_fast = tf.data.Dataset.from_tensor_slices(
    (
        train_df["local_path"].values,
        train_df["class_id"].values
    )
)

train_ds_cnx_fast = (
    train_ds_cnx_fast
    .map(
        load_image_cnx_fast,
        num_parallel_calls=AUTOTUNE
    )
    .cache()  # decode/resize only once
    .shuffle(
        len(train_df),
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE_CNX)
    .prefetch(AUTOTUNE)
)


# VALIDATION
val_ds_cnx_fast = tf.data.Dataset.from_tensor_slices(
    (
        val_df["local_path"].values,
        val_df["class_id"].values
    )
)

val_ds_cnx_fast = (
    val_ds_cnx_fast
    .map(
        load_image_cnx_fast,
        num_parallel_calls=AUTOTUNE
    )
    .cache()
    .batch(BATCH_SIZE_CNX)
    .prefetch(AUTOTUNE)
)


print("Warming cache...")

start = time.time()

for _ in train_ds_cnx_fast:
    pass

for _ in val_ds_cnx_fast:
    pass

minutes = (time.time() - start) / 60

print("\n===== FAST DATASET READY =====")
print("Train images:", len(train_df))
print("Val images  :", len(val_df))
print("Batch size  :", BATCH_SIZE_CNX)
print("Train steps :", len(train_ds_cnx_fast))
print("Val steps   :", len(val_ds_cnx_fast))
print(f"Cache warm-up: {minutes:.2f} minutes")

images, labels = next(iter(train_ds_cnx_fast))

print("Batch:", images.shape)
print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("\n✅ Cached datasets ready.")

Warming cache...

===== FAST DATASET READY =====
Train images: 7975
Val images  : 1139
Batch size  : 32
Train steps : 250
Val steps   : 36
Cache warm-up: 1.44 minutes
Batch: (32, 224, 224, 3)
Pixel range: 0.0 to 255.0

✅ Cached datasets ready.


In [ ]:
import tensorflow as tf
import time
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# Load the successfully saved Epoch-1 model
CNX_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_convnext_stage1_best.keras"
)

print("Loading saved ConvNeXt checkpoint...")

convnext_model = tf.keras.models.load_model(
    CNX_STAGE1_PATH,
    compile=False
)

convnext_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3, name="top3"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5, name="top5"
        )
    ]
)

print("✅ Model loaded")

# ------------------------------------------------
# GPU check
# ------------------------------------------------

print("\nGPU:")
print(tf.config.list_physical_devices("GPU"))

# ------------------------------------------------
# Benchmark ONLY 10 batches
# ------------------------------------------------

benchmark_ds = train_ds_cnx_fast.take(10)

print("\nRunning 10-batch speed test...")

start = time.time()

result = convnext_model.evaluate(
    benchmark_ds,
    verbose=1,
    return_dict=True
)

elapsed = time.time() - start

print("\n==============================")
print("SPEED TEST")
print("==============================")
print(f"10 batches: {elapsed:.2f} seconds")
print(f"Average: {elapsed/10:.2f} sec/batch")

estimated_epoch = (
    (elapsed / 10) *
    len(train_ds_cnx_fast) / 60
)

print(
    f"Estimated training-pass time: "
    f"{estimated_epoch:.1f} minutes"
)

print("==============================")

Loading saved ConvNeXt checkpoint...
✅ Model loaded

GPU:
[]

Running 10-batch speed test...
10/10 ━━━━━━━━━━━━━━━━━━━━ 518s 49s/step - accuracy: 0.2125 - loss: 3.6910 - top3: 0.3562 - top5: 0.4156

SPEED TEST
10 batches: 577.63 seconds
Average: 57.76 sec/batch
Estimated training-pass time: 240.7 minutes


In [ ]:
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices("GPU"))

if tf.config.list_physical_devices("GPU"):
    print("✅ GPU READY")
else:
    print("❌ GPU NOT AVAILABLE")

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU READY


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import pandas as pd
from tqdm.auto import tqdm

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# Load official manifests
train_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
)

val_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_val.csv")
)

# Fixed 114-class mapping
class_names = sorted(train_df["label"].unique())
class_to_index = {
    name: i for i, name in enumerate(class_names)
}

train_df["class_id"] = train_df["label"].map(class_to_index)
val_df["class_id"] = val_df["label"].map(class_to_index)

LOCAL_TRAIN = "/content/DermaWise_Train_Images"
LOCAL_VAL = "/content/DermaWise_Val_Images"

os.makedirs(LOCAL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_VAL, exist_ok=True)


def restore_partition(df, destination, name):

    copied = 0
    missing = 0

    print(f"\nRestoring {name}...")

    for source in tqdm(df["image_path"].astype(str)):

        if not os.path.isfile(source):
            missing += 1
            continue

        class_folder = os.path.basename(
            os.path.dirname(source)
        )

        target_folder = os.path.join(
            destination,
            class_folder
        )

        os.makedirs(target_folder, exist_ok=True)

        target = os.path.join(
            target_folder,
            os.path.basename(source)
        )

        if not os.path.isfile(target):
            shutil.copy2(source, target)
            copied += 1

    print(f"{name} copied : {copied}")
    print(f"{name} missing: {missing}")


restore_partition(
    train_df,
    LOCAL_TRAIN,
    "TRAIN"
)

restore_partition(
    val_df,
    LOCAL_VAL,
    "VALIDATION"
)


# Rebuild local paths
def local_path(original, root):

    return os.path.join(
        root,
        os.path.basename(os.path.dirname(original)),
        os.path.basename(original)
    )


train_df["local_path"] = train_df["image_path"].apply(
    lambda x: local_path(x, LOCAL_TRAIN)
)

val_df["local_path"] = val_df["image_path"].apply(
    lambda x: local_path(x, LOCAL_VAL)
)


print("\n==============================")
print("RESTORE COMPLETE")
print("==============================")

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Classes:", len(class_names))

print(
    "Train missing:",
    sum(not os.path.isfile(x)
        for x in train_df["local_path"])
)

print(
    "Val missing:",
    sum(not os.path.isfile(x)
        for x in val_df["local_path"])
)

print("\n✅ Ready for GPU dataset setup.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Restoring TRAIN...


  0%|          | 0/7975 [00:00<?, ?it/s]

TRAIN copied : 7975
TRAIN missing: 0

Restoring VALIDATION...


  0%|          | 0/1139 [00:00<?, ?it/s]

VALIDATION copied : 1139
VALIDATION missing: 0

RESTORE COMPLETE
Train: 7975
Val  : 1139
Classes: 114
Train missing: 0
Val missing: 0

✅ Ready for GPU dataset setup.


In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE_CNX = 224
BATCH_SIZE_CNX = 32
AUTOTUNE = tf.data.AUTOTUNE

# Confirm GPU before doing anything
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)

if not gpus:
    raise RuntimeError("STOP — GPU is not available.")

# ---------------------------------------
# Image loader
# ---------------------------------------

def load_image_cnx(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_CNX, IMG_SIZE_CNX],
        antialias=False
    )

    image = tf.cast(image, tf.float32)

    return image, label


# ---------------------------------------
# TRAIN
# ---------------------------------------

train_ds_cnx_fast = tf.data.Dataset.from_tensor_slices(
    (
        train_df["local_path"].values,
        train_df["class_id"].values
    )
)

train_ds_cnx_fast = (
    train_ds_cnx_fast
    .map(load_image_cnx, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(
        len(train_df),
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE_CNX)
    .prefetch(AUTOTUNE)
)


# ---------------------------------------
# VALIDATION
# ---------------------------------------

val_ds_cnx_fast = tf.data.Dataset.from_tensor_slices(
    (
        val_df["local_path"].values,
        val_df["class_id"].values
    )
)

val_ds_cnx_fast = (
    val_ds_cnx_fast
    .map(load_image_cnx, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE_CNX)
    .prefetch(AUTOTUNE)
)


# ---------------------------------------
# Class weights
# ---------------------------------------

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(114),
    y=train_df["class_id"].values
)

class_weights_cnx = {
    i: float(weights[i])
    for i in range(114)
}


# ---------------------------------------
# Check
# ---------------------------------------

images, labels = next(iter(train_ds_cnx_fast))

print("\n===== GPU DATASET CHECK =====")
print("Train images:", len(train_df))
print("Val images  :", len(val_df))
print("Train steps :", len(train_ds_cnx_fast))
print("Val steps   :", len(val_ds_cnx_fast))
print("Batch shape :", images.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("\n✅ GPU datasets ready.")

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

===== GPU DATASET CHECK =====
Train images: 7975
Val images  : 1139
Train steps : 250
Val steps   : 36
Batch shape : (32, 224, 224, 3)
Pixel range: 0.0 to 255.0

✅ GPU datasets ready.


In [ ]:
import tensorflow as tf
import time
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

CNX_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_convnext_stage1_best.keras"
)

print("===== GPU CHECK =====")
print(tf.config.list_physical_devices("GPU"))

# Load the saved Epoch-1 ConvNeXt model
print("\nLoading saved Epoch-1 ConvNeXt model...")

convnext_model = tf.keras.models.load_model(
    CNX_STAGE1_PATH,
    compile=False
)

convnext_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

print("✅ Model loaded")

# Warm up GPU first
print("\nWarming up GPU...")

sample_images, _ = next(iter(train_ds_cnx_fast))

_ = convnext_model(
    sample_images,
    training=False
)

print("✅ GPU warm-up complete")

# --------------------------------------------
# Benchmark 10 batches
# --------------------------------------------

benchmark_ds = train_ds_cnx_fast.take(10)

print("\nRunning 10-batch benchmark...")

start = time.time()

results = convnext_model.evaluate(
    benchmark_ds,
    verbose=1,
    return_dict=True
)

elapsed = time.time() - start

seconds_per_batch = elapsed / 10
estimated_minutes = (
    seconds_per_batch *
    len(train_ds_cnx_fast)
) / 60

print("\n================================")
print("GPU SPEED TEST")
print("================================")

print(f"10 batches: {elapsed:.2f} seconds")
print(f"Average: {seconds_per_batch:.2f} sec/batch")
print(
    f"Estimated full pass: "
    f"{estimated_minutes:.1f} minutes"
)

print("================================")

===== GPU CHECK =====
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loading saved Epoch-1 ConvNeXt model...
✅ Model loaded

Warming up GPU...


✅ GPU warm-up complete

Running 10-batch benchmark...
10/10 ━━━━━━━━━━━━━━━━━━━━ 10s 314ms/step - accuracy: 0.1594 - loss: 3.8776 - top3: 0.2844 - top5: 0.3438

GPU SPEED TEST
10 batches: 10.02 seconds
Average: 1.00 sec/batch
Estimated full pass: 4.2 minutes


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

CNX_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_convnext_stage1_best.keras"
)

# The loaded checkpoint contains the trained Epoch-1 weights.
# Keep the ConvNeXt backbone frozen for Stage 1.
for layer in convnext_model.layers:
    if isinstance(layer, tf.keras.Model):
        layer.trainable = False

convnext_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CNX_STAGE1_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("===== CONVNEXT GPU CONTINUATION =====")
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Starting from saved Epoch-1 weights...")
print()

history_cnx = convnext_model.fit(
    train_ds_cnx_fast,
    validation_data=val_ds_cnx_fast,
    initial_epoch=1,
    epochs=15,
    class_weight=class_weights_cnx,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ ConvNeXt Stage 1 finished.")
print("Best checkpoint:", CNX_STAGE1_PATH)

===== CONVNEXT GPU CONTINUATION =====
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Starting from saved Epoch-1 weights...

Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 356ms/step - accuracy: 0.1375 - loss: 3.8236 - top3: 0.2739 - top5: 0.3547
Epoch 2: val_loss improved from None to 3.82052, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_convnext_stage1_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_convnext_stage1_best.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 134s 461ms/step - accuracy: 0.1421 - loss: 3.7753 - top3: 0.2769 - top5: 0.3582 - val_accuracy: 0.1370 - val_loss: 3.8205 - val_top3: 0.2853 - val_top5: 0.3600 - learning_rate: 0.0010
Epoch 3/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.1817 - loss: 3.4290 - top3: 0.3254 - top5: 0.4203
Epoch 3: val_loss improved from 3.82052 to 3.77614, saving model to /content/drive/MyDrive/DermaWise/

In [ ]:
import numpy as np

h = history_cnx.history

best_idx = int(np.argmin(h["val_loss"]))

print("===== CONVNEXT STAGE-1 BEST EPOCH =====")
print("Best epoch :", best_idx + 2)  # continuation started at epoch 2
print(f"Val loss   : {h['val_loss'][best_idx]:.4f}")
print(f"Val Top-1  : {h['val_accuracy'][best_idx]*100:.2f}%")
print(f"Val Top-3  : {h['val_top3'][best_idx]*100:.2f}%")
print(f"Val Top-5  : {h['val_top5'][best_idx]*100:.2f}%")

print("\n===== CURRENT TARGET TO BEAT =====")
print("Focal B2 Top-1 : 31.08%")
print("Focal B2 Top-3 : 48.02%")
print("Focal B2 Top-5 : 55.84%")

===== CONVNEXT STAGE-1 BEST EPOCH =====
Best epoch : 14
Val loss   : 3.4421
Val Top-1  : 22.12%
Val Top-3  : 38.81%
Val Top-5  : 46.71%

===== CURRENT TARGET TO BEAT =====
Focal B2 Top-1 : 31.08%
Focal B2 Top-3 : 48.02%
Focal B2 Top-5 : 55.84%


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_convnext_stage1_best.keras"
)

STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_convnext_finetuned_best.keras"
)

# --------------------------------------------------
# 1. Load BEST Stage-1 checkpoint
# --------------------------------------------------

tf.keras.backend.clear_session()

model = tf.keras.models.load_model(
    STAGE1_PATH,
    compile=False
)

print("✅ Best Stage-1 model loaded")

# --------------------------------------------------
# 2. Find ConvNeXt backbone
# --------------------------------------------------

backbone = None

for layer in model.layers:
    if isinstance(layer, tf.keras.Model):
        if "convnext" in layer.name.lower():
            backbone = layer
            break

if backbone is None:
    raise RuntimeError("ConvNeXt backbone not found.")

print("Backbone:", backbone.name)

# --------------------------------------------------
# 3. Fine-tune last ~40 eligible layers
# --------------------------------------------------

backbone.trainable = True

# First freeze everything
for layer in backbone.layers:
    layer.trainable = False

eligible_layers = [
    layer for layer in backbone.layers
    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    )
]

# Unfreeze last 40 non-BN layers
for layer in eligible_layers[-40:]:
    layer.trainable = True

trainable_backbone_layers = sum(
    layer.trainable for layer in backbone.layers
)

print(
    "Trainable backbone layers:",
    trainable_backbone_layers
)

print(
    "Total trainable parameters:",
    f"{sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}"
)

# --------------------------------------------------
# 4. Compile with SMALL learning rate
# --------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss="sparse_categorical_crossentropy",

    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(
            name="accuracy"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=3,
            name="top3"
        ),

        tf.keras.metrics.SparseTopKCategoricalAccuracy(
            k=5,
            name="top5"
        )
    ]
)

# --------------------------------------------------
# 5. Callbacks
# --------------------------------------------------

callbacks_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(
        STAGE2_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

# --------------------------------------------------
# 6. GPU safety check
# --------------------------------------------------

gpus = tf.config.list_physical_devices("GPU")

print("\nGPU:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU NOT AVAILABLE — DO NOT TRAIN."
    )

# --------------------------------------------------
# 7. Fine-tune
# --------------------------------------------------

print("\n===== CONVNEXT STAGE 2 =====")
print("Starting fine-tuning...\n")

history_cnx_stage2 = model.fit(
    train_ds_cnx_fast,
    validation_data=val_ds_cnx_fast,

    epochs=15,

    class_weight=class_weights_cnx,

    callbacks=callbacks_stage2,

    verbose=1
)

print("\n====================================")
print("✅ CONVNEXT STAGE 2 COMPLETE")
print("====================================")

print("Best checkpoint:")
print(STAGE2_PATH)

✅ Best Stage-1 model loaded
Backbone: convnext_tiny
Trainable backbone layers: 40
Total trainable parameters: 18,554,226

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

===== CONVNEXT STAGE 2 =====
Starting fine-tuning...

Epoch 1/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 736ms/step - accuracy: 0.3480 - loss: 2.2610 - top3: 0.5353 - top5: 0.6304
Epoch 1: val_loss improved from None to 3.41113, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_convnext_finetuned_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_convnext_finetuned_best.keras
250/250 ━━━━━━━━━━━━━━━━━━━━ 233s 824ms/step - accuracy: 0.3594 - loss: 2.2589 - top3: 0.5475 - top5: 0.6426 - val_accuracy: 0.2353 - val_loss: 3.4111 - val_top3: 0.4004 - val_top5: 0.4925 - learning_rate: 1.0000e-05
Epoch 2/15
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 723ms/step - accuracy: 0.3646 - loss: 2.1434 - top3: 0.5590 - top5:

In [ ]:
import numpy as np

h = history_cnx_stage2.history

best_idx = int(np.argmin(h["val_loss"]))

print("===== CONVNEXT STAGE-2 BEST RESULT =====")
print("Best epoch :", best_idx + 1)
print(f"Val loss   : {h['val_loss'][best_idx]:.4f}")
print(f"Val Top-1  : {h['val_accuracy'][best_idx]*100:.2f}%")
print(f"Val Top-3  : {h['val_top3'][best_idx]*100:.2f}%")
print(f"Val Top-5  : {h['val_top5'][best_idx]*100:.2f}%")

print("\n===== COMPARISON =====")
print("ConvNeXt Stage 1 : 22.12% / 38.81% / 46.71%")
print("B2 Focal         : 31.08% / 48.02% / 55.84%")

===== CONVNEXT STAGE-2 BEST RESULT =====
Best epoch : 15
Val loss   : 3.2204
Val Top-1  : 27.48%
Val Top-3  : 43.99%
Val Top-5  : 52.41%

===== COMPARISON =====
ConvNeXt Stage 1 : 22.12% / 38.81% / 46.71%
B2 Focal         : 31.08% / 48.02% / 55.84%


In [ ]:
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

comparison = pd.DataFrame([
    {
        "Model": "EfficientNetB0 Stage2",
        "Val_Top1": None,
        "Test_Top1": 24.96,
        "Test_Top3": 42.59,
        "Test_Top5": 51.58
    },
    {
        "Model": "EfficientNetB2 Continued",
        "Val_Top1": 28.27,
        "Test_Top1": 27.68,
        "Test_Top3": 44.56,
        "Test_Top5": 54.34
    },
    {
        "Model": "EfficientNetB2 Focal",
        "Val_Top1": 31.08,
        "Test_Top1": 28.95,
        "Test_Top3": 47.06,
        "Test_Top5": 56.67
    },
    {
        "Model": "ConvNeXtTiny Stage2",
        "Val_Top1": 27.48,
        "Test_Top1": None,
        "Test_Top3": None,
        "Test_Top5": None
    }
])

path = os.path.join(
    WORK_DIR,
    "fitzpatrick114_architecture_comparison.csv"
)

comparison.to_csv(path, index=False)

print(comparison.to_string(index=False))
print("\n✅ Saved:", path)

                   Model  Val_Top1  Test_Top1  Test_Top3  Test_Top5
   EfficientNetB0 Stage2       NaN      24.96      42.59      51.58
EfficientNetB2 Continued     28.27      27.68      44.56      54.34
    EfficientNetB2 Focal     31.08      28.95      47.06      56.67
     ConvNeXtTiny Stage2     27.48        NaN        NaN        NaN

✅ Saved: /content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick114_architecture_comparison.csv


In [ ]:
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

META_PATH = os.path.join(
    WORK_DIR,
    "Fitzpatrick17k-C.csv"
)

meta = pd.read_csv(META_PATH)

print("===== FITZPATRICK17K-C HIERARCHY CHECK =====")

print("Shape:", meta.shape)

print("\nAvailable columns:")
print(meta.columns.tolist())

for column in [
    "three_partition_label",
    "nine_partition_label",
    "label"
]:
    print(f"\n===== {column} =====")
    print("Unique:", meta[column].nunique())
    print(meta[column].value_counts(dropna=False).to_string())

print("\n===== LABEL → GROUP EXAMPLES =====")

mapping = (
    meta[
        [
            "label",
            "three_partition_label",
            "nine_partition_label"
        ]
    ]
    .drop_duplicates()
    .sort_values([
        "nine_partition_label",
        "label"
    ])
)

print(mapping.head(40).to_string(index=False))

print("\nUnique label/group mappings:", len(mapping))

===== FITZPATRICK17K-C HIERARCHY CHECK =====
Shape: (11394, 15)

Available columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name']

===== three_partition_label =====
Unique: 3
three_partition_label
non-neoplastic    8204
malignant         1683
benign            1507

===== nine_partition_label =====
Unique: 9
nine_partition_label
inflammatory                    7495
malignant epidermal             1013
benign dermal                    760
genodermatoses                   709
benign epidermal                 590
malignant melanoma               429
benign melanocyte                157
malignant cutaneous lymphoma     143
malignant dermal                  98

===== label =====
Unique: 114
label
psoriasis                                                534
basal cell carcinoma                                     412
lichen planus           

In [ ]:
# Check whether every disease maps consistently to one
# 3-way and one 9-way parent category.

hierarchy_check = (
    meta.groupby("label")
    .agg(
        three_groups=("three_partition_label", "nunique"),
        nine_groups=("nine_partition_label", "nunique"),
        samples=("label", "size")
    )
    .reset_index()
)

problematic = hierarchy_check[
    (hierarchy_check["three_groups"] != 1) |
    (hierarchy_check["nine_groups"] != 1)
]

print("===== HIERARCHY VALIDATION =====")
print("Total disease labels:", len(hierarchy_check))

print(
    "Diseases with exactly one 3-way group:",
    (hierarchy_check["three_groups"] == 1).sum()
)

print(
    "Diseases with exactly one 9-way group:",
    (hierarchy_check["nine_groups"] == 1).sum()
)

print(
    "Problematic mappings:",
    len(problematic)
)

if len(problematic) > 0:
    print("\nProblematic diseases:")
    print(problematic.to_string(index=False))
else:
    print("\n✅ Every disease maps consistently to the hierarchy.")


print("\n===== 9-WAY GROUP → NUMBER OF DISEASES =====")

nine_mapping = (
    meta[["label", "nine_partition_label"]]
    .drop_duplicates()
)

group_summary = (
    nine_mapping
    .groupby("nine_partition_label")["label"]
    .nunique()
    .sort_values(ascending=False)
)

print(group_summary.to_string())


print("\n===== 3-WAY GROUP → NUMBER OF DISEASES =====")

three_mapping = (
    meta[["label", "three_partition_label"]]
    .drop_duplicates()
)

print(
    three_mapping
    .groupby("three_partition_label")["label"]
    .nunique()
    .sort_values(ascending=False)
    .to_string()
)

===== HIERARCHY VALIDATION =====
Total disease labels: 114
Diseases with exactly one 3-way group: 114
Diseases with exactly one 9-way group: 114
Problematic mappings: 0

✅ Every disease maps consistently to the hierarchy.

===== 9-WAY GROUP → NUMBER OF DISEASES =====
nine_partition_label
inflammatory                    69
benign dermal                   12
genodermatoses                  10
benign epidermal                 9
malignant epidermal              5
malignant melanoma               4
benign melanocyte                3
malignant dermal                 1
malignant cutaneous lymphoma     1

===== 3-WAY GROUP → NUMBER OF DISEASES =====
three_partition_label
non-neoplastic    79
benign            24
malignant         11


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# --------------------------------------------------
# 1. Create disease -> hierarchy mapping
# --------------------------------------------------

hierarchy_map = (
    meta[
        [
            "label",
            "three_partition_label",
            "nine_partition_label"
        ]
    ]
    .drop_duplicates(subset=["label"])
    .copy()
)

print("Disease mappings:", len(hierarchy_map))

# --------------------------------------------------
# 2. Add hierarchy to existing official splits
# --------------------------------------------------

# Remove old hierarchy columns if this cell is rerun
for df in [train_df, val_df]:
    for col in [
        "three_partition_label",
        "nine_partition_label"
    ]:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

train_h = train_df.merge(
    hierarchy_map,
    on="label",
    how="left"
)

val_h = val_df.merge(
    hierarchy_map,
    on="label",
    how="left"
)

# --------------------------------------------------
# 3. Verify nothing was lost
# --------------------------------------------------

print("\n===== HIERARCHICAL SPLITS =====")
print("Train:", len(train_h))
print("Val  :", len(val_h))

print("\nMissing hierarchy labels:")

print(
    "Train 3-way:",
    train_h["three_partition_label"].isna().sum()
)

print(
    "Train 9-way:",
    train_h["nine_partition_label"].isna().sum()
)

print(
    "Val 3-way:",
    val_h["three_partition_label"].isna().sum()
)

print(
    "Val 9-way:",
    val_h["nine_partition_label"].isna().sum()
)

# --------------------------------------------------
# 4. Create fixed parent-class mappings
# --------------------------------------------------

three_classes = sorted(
    train_h["three_partition_label"].unique()
)

nine_classes = sorted(
    train_h["nine_partition_label"].unique()
)

three_to_index = {
    name: i
    for i, name in enumerate(three_classes)
}

nine_to_index = {
    name: i
    for i, name in enumerate(nine_classes)
}

train_h["three_id"] = (
    train_h["three_partition_label"]
    .map(three_to_index)
)

val_h["three_id"] = (
    val_h["three_partition_label"]
    .map(three_to_index)
)

train_h["nine_id"] = (
    train_h["nine_partition_label"]
    .map(nine_to_index)
)

val_h["nine_id"] = (
    val_h["nine_partition_label"]
    .map(nine_to_index)
)

# --------------------------------------------------
# 5. Display class distributions
# --------------------------------------------------

print("\n===== 3-WAY CLASSES =====")

for name in three_classes:

    train_count = (
        train_h["three_partition_label"] == name
    ).sum()

    val_count = (
        val_h["three_partition_label"] == name
    ).sum()

    print(
        f"{three_to_index[name]:2d} | "
        f"{name:35s} | "
        f"Train {train_count:4d} | "
        f"Val {val_count:4d}"
    )


print("\n===== 9-WAY CLASSES =====")

for name in nine_classes:

    train_count = (
        train_h["nine_partition_label"] == name
    ).sum()

    val_count = (
        val_h["nine_partition_label"] == name
    ).sum()

    print(
        f"{nine_to_index[name]:2d} | "
        f"{name:40s} | "
        f"Train {train_count:4d} | "
        f"Val {val_count:4d}"
    )

# --------------------------------------------------
# 6. Save manifests
# --------------------------------------------------

TRAIN_H_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_hierarchical_train.csv"
)

VAL_H_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_hierarchical_val.csv"
)

train_h.to_csv(TRAIN_H_PATH, index=False)
val_h.to_csv(VAL_H_PATH, index=False)

print("\n====================================")
print("✅ HIERARCHICAL MANIFESTS READY")
print("====================================")

print("3-way classes:", len(three_classes))
print("9-way classes:", len(nine_classes))
print("Disease classes:", train_h["label"].nunique())

print("\nSaved:")
print(TRAIN_H_PATH)
print(VAL_H_PATH)

Disease mappings: 114

===== HIERARCHICAL SPLITS =====
Train: 7975
Val  : 1139

Missing hierarchy labels:
Train 3-way: 0
Train 9-way: 0
Val 3-way: 0
Val 9-way: 0

===== 3-WAY CLASSES =====
 0 | benign                              | Train 1056 | Val  151
 1 | malignant                           | Train 1177 | Val  168
 2 | non-neoplastic                      | Train 5742 | Val  820

===== 9-WAY CLASSES =====
 0 | benign dermal                            | Train  531 | Val   77
 1 | benign epidermal                         | Train  415 | Val   58
 2 | benign melanocyte                        | Train  110 | Val   16
 3 | genodermatoses                           | Train  496 | Val   71
 4 | inflammatory                             | Train 5246 | Val  749
 5 | malignant cutaneous lymphoma             | Train  100 | Val   14
 6 | malignant dermal                         | Train   69 | Val   10
 7 | malignant epidermal                      | Train  708 | Val  101
 8 | malignant melanoma      

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE_H = 260
BATCH_SIZE_H = 16
AUTOTUNE = tf.data.AUTOTUNE

# --------------------------------------------
# Image loader
# --------------------------------------------

def load_hierarchical_image(path, three_id, nine_id, disease_id):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE_H, IMG_SIZE_H],
        antialias=False
    )

    image = tf.cast(image, tf.float32)

    targets = {
        "three_output": three_id,
        "nine_output": nine_id,
        "disease_output": disease_id
    }

    return image, targets


# --------------------------------------------
# TRAIN dataset
# --------------------------------------------

train_ds_h = tf.data.Dataset.from_tensor_slices(
    (
        train_h["local_path"].values,
        train_h["three_id"].values,
        train_h["nine_id"].values,
        train_h["class_id"].values
    )
)

train_ds_h = (
    train_ds_h
    .shuffle(
        len(train_h),
        reshuffle_each_iteration=True
    )
    .map(
        load_hierarchical_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE_H)
    .prefetch(AUTOTUNE)
)


# --------------------------------------------
# VALIDATION dataset
# --------------------------------------------

val_ds_h = tf.data.Dataset.from_tensor_slices(
    (
        val_h["local_path"].values,
        val_h["three_id"].values,
        val_h["nine_id"].values,
        val_h["class_id"].values
    )
)

val_ds_h = (
    val_ds_h
    .map(
        load_hierarchical_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE_H)
    .prefetch(AUTOTUNE)
)


# --------------------------------------------
# Disease class weights
# --------------------------------------------

disease_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(114),
    y=train_h["class_id"].values
)

disease_weights = {
    i: float(w)
    for i, w in enumerate(disease_weights)
}


# --------------------------------------------
# Check one batch
# --------------------------------------------

images, targets = next(iter(train_ds_h))

print("===== HIERARCHICAL DATASET CHECK =====")

print("Images:", images.shape)

print(
    "3-way targets:",
    targets["three_output"].shape
)

print(
    "9-way targets:",
    targets["nine_output"].shape
)

print(
    "114-way targets:",
    targets["disease_output"].shape
)

print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print("\nClasses:")
print("3-way :", len(three_classes))
print("9-way :", len(nine_classes))
print("Disease:", len(class_names))

print("\nGPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "GPU unavailable — DO NOT START TRAINING."
    )

print("\n✅ Hierarchical datasets ready.")

===== HIERARCHICAL DATASET CHECK =====
Images: (16, 260, 260, 3)
3-way targets: (16,)
9-way targets: (16,)
114-way targets: (16,)
Pixel range: 0.0 to 255.0

Classes:
3-way : 3
9-way : 9
Disease: 114

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ Hierarchical datasets ready.


In [ ]:
import tensorflow as tf
import os

tf.keras.backend.clear_session()

IMG_SIZE_H = 260

# --------------------------------------------------
# 1. Data augmentation
# --------------------------------------------------

augmentation_h = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name="hierarchical_augmentation")


# --------------------------------------------------
# 2. EfficientNetB2 backbone
# --------------------------------------------------

backbone_h = tf.keras.applications.EfficientNetB2(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE_H, IMG_SIZE_H, 3)
)

# Stage 1 = frozen backbone
backbone_h.trainable = False


# --------------------------------------------------
# 3. Shared feature representation
# --------------------------------------------------

inputs = tf.keras.Input(
    shape=(IMG_SIZE_H, IMG_SIZE_H, 3),
    name="image"
)

x = augmentation_h(inputs)

x = backbone_h(
    x,
    training=False
)

x = tf.keras.layers.GlobalAveragePooling2D(
    name="shared_pool"
)(x)

x = tf.keras.layers.Dropout(
    0.35,
    name="shared_dropout"
)(x)


# --------------------------------------------------
# 4. Three related prediction heads
# --------------------------------------------------

three_output = tf.keras.layers.Dense(
    3,
    activation="softmax",
    name="three_output"
)(x)

nine_output = tf.keras.layers.Dense(
    9,
    activation="softmax",
    name="nine_output"
)(x)

disease_output = tf.keras.layers.Dense(
    114,
    activation="softmax",
    name="disease_output"
)(x)


hierarchical_model = tf.keras.Model(
    inputs=inputs,
    outputs={
        "three_output": three_output,
        "nine_output": nine_output,
        "disease_output": disease_output
    },
    name="DermaWise_Hierarchical_B2"
)


# --------------------------------------------------
# 5. Compile
#
# Disease prediction gets the dominant weight.
# Parent categories act as auxiliary supervision.
# --------------------------------------------------

hierarchical_model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss={
        "three_output":
            "sparse_categorical_crossentropy",

        "nine_output":
            "sparse_categorical_crossentropy",

        "disease_output":
            "sparse_categorical_crossentropy"
    },

    loss_weights={
        "three_output": 0.10,
        "nine_output": 0.20,
        "disease_output": 1.00
    },

    metrics={
        "three_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "nine_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "disease_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=3,
                name="top3"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=5,
                name="top5"
            )
        ]
    }
)


# --------------------------------------------------
# 6. Verification
# --------------------------------------------------

print("===== HIERARCHICAL MODEL =====")

print("Model:", hierarchical_model.name)
print("Backbone:", backbone_h.name)
print("Backbone trainable:", backbone_h.trainable)

print("\nOutputs:")
for name, output in zip(
    hierarchical_model.output_names,
    hierarchical_model.outputs
):
    print(name, output.shape)

print(
    "\nTotal parameters:",
    f"{hierarchical_model.count_params():,}"
)

print(
    "Trainable parameters:",
    f"{sum(tf.keras.backend.count_params(w) for w in hierarchical_model.trainable_weights):,}"
)

print("\nGPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "GPU unavailable — DO NOT TRAIN."
    )

print("\n✅ Hierarchical EfficientNetB2 ready.")

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
===== HIERARCHICAL MODEL =====
Model: DermaWise_Hierarchical_B2
Backbone: efficientnetb2
Backbone trainable: False

Outputs:
disease_output (None, 114)
nine_output (None, 9)
three_output (None, 3)

Total parameters: 7,946,103
Trainable parameters: 177,534

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

✅ Hierarchical EfficientNetB2 ready.


In [ ]:
import os
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

HIER_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_stage1_best.keras"
)

HIER_BACKUP_DIR = os.path.join(
    WORK_DIR,
    "hierarchical_b2_stage1_backup"
)

# --------------------------------------------------
# GPU safety check
# --------------------------------------------------

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU NOT AVAILABLE — DO NOT START TRAINING."
    )

# --------------------------------------------------
# Callbacks
# --------------------------------------------------

callbacks_h_stage1 = [

    tf.keras.callbacks.ModelCheckpoint(
        filepath=HIER_STAGE1_PATH,

        # IMPORTANT:
        # Select according to the actual 114-condition output
        monitor="val_disease_output_accuracy",

        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=HIER_BACKUP_DIR
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_disease_output_accuracy",
        mode="max",

        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_disease_output_loss",
        mode="min",

        factor=0.3,
        patience=2,

        min_lr=1e-7,
        verbose=1
    )
]

print("\n===== HIERARCHICAL B2 — STAGE 1 =====")
print("Backbone trainable:", backbone_h.trainable)
print("Epochs: maximum 15")
print("\nStarting training...\n")

history_h_stage1 = hierarchical_model.fit(
    train_ds_h,
    validation_data=val_ds_h,

    epochs=15,

    callbacks=callbacks_h_stage1,

    verbose=1
)

print("\n====================================")
print("✅ HIERARCHICAL STAGE 1 COMPLETE")
print("====================================")

print("Best checkpoint:")
print(HIER_STAGE1_PATH)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

===== HIERARCHICAL B2 — STAGE 1 =====
Backbone trainable: False
Epochs: maximum 15

Starting training...

Epoch 1/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - disease_output_accuracy: 0.0789 - disease_output_loss: 4.3289 - disease_output_top3: 0.1668 - disease_output_top5: 0.2309 - loss: 4.6647 - nine_output_accuracy: 0.6320 - nine_output_loss: 1.3029 - three_output_accuracy: 0.7187 - three_output_loss: 0.7513
Epoch 1: val_disease_output_accuracy improved from None to 0.16769, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_hierarchical_b2_stage1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawise_fitzpatrick114_hierarchical_b2_stage1_best.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 81s 128ms/step - disease_output_accuracy: 0.1083 - disease_output_loss: 4.0607 - disease_output_top3: 0.2219 - disease_output_top5: 0.3013 - loss: 4.

In [ ]:
import tensorflow as tf
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

HIER_STAGE1_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_stage1_best.keras"
)

print("Loading best hierarchical Stage-1 model...")

stage1_h_best = tf.keras.models.load_model(
    HIER_STAGE1_PATH,
    compile=False
)

# --------------------------------------------------
# Compile for clean evaluation
# --------------------------------------------------

stage1_h_best.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),

    loss={
        "three_output": "sparse_categorical_crossentropy",
        "nine_output": "sparse_categorical_crossentropy",
        "disease_output": "sparse_categorical_crossentropy"
    },

    loss_weights={
        "three_output": 0.10,
        "nine_output": 0.20,
        "disease_output": 1.00
    },

    metrics={
        "three_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "nine_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "disease_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=3,
                name="top3"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=5,
                name="top5"
            )
        ]
    }
)

print("\nEvaluating best Stage-1 checkpoint...\n")

results = stage1_h_best.evaluate(
    val_ds_h,
    verbose=1,
    return_dict=True
)

print("\n========================================")
print("HIERARCHICAL STAGE-1 VALIDATION RESULTS")
print("========================================")

for key, value in results.items():
    print(f"{key:35s}: {value:.4f}")

print("\n===== MAIN 114-CONDITION RESULTS =====")

print(
    "Top-1:",
    f"{results['disease_output_accuracy'] * 100:.2f}%"
)

print(
    "Top-3:",
    f"{results['disease_output_top3'] * 100:.2f}%"
)

print(
    "Top-5:",
    f"{results['disease_output_top5'] * 100:.2f}%"
)

print("\nCurrent model to beat:")
print("Focal B2 validation Top-1 : 31.08%")
print("Focal B2 validation Top-3 : 48.02%")
print("Focal B2 validation Top-5 : 55.84%")

Loading best hierarchical Stage-1 model...

Evaluating best Stage-1 checkpoint...

72/72 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step - disease_output_accuracy: 0.2871 - disease_output_loss: 3.1718 - disease_output_top3: 0.4486 - disease_output_top5: 0.5408 - loss: 3.4162 - nine_output_accuracy: 0.7103 - nine_output_loss: 0.9599 - three_output_accuracy: 0.7779 - three_output_loss: 0.5942

HIERARCHICAL STAGE-1 VALIDATION RESULTS
disease_output_accuracy            : 0.2871
disease_output_loss                : 3.1718
disease_output_top3                : 0.4486
disease_output_top5                : 0.5408
loss                               : 3.4162
nine_output_accuracy               : 0.7103
nine_output_loss                   : 0.9599
three_output_accuracy              : 0.7779
three_output_loss                  : 0.5942

===== MAIN 114-CONDITION RESULTS =====
Top-1: 28.71%
Top-3: 44.86%
Top-5: 54.08%

Current model to beat:
Focal B2 validation Top-1 : 31.08%
Focal B2 validation Top-3 : 48.02%
Focal 

In [ ]:
import tensorflow as tf
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

HIER_STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

HIER_STAGE2_BACKUP = os.path.join(
    WORK_DIR,
    "hierarchical_b2_stage2_backup"
)

# ============================================================
# 1. GPU CHECK
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

print("GPU:", gpus)

if not gpus:
    raise RuntimeError(
        "GPU NOT AVAILABLE — DO NOT START TRAINING."
    )


# ============================================================
# 2. LOAD BEST STAGE-1 MODEL
# ============================================================

print("\nLoading best hierarchical Stage-1 model...")

hierarchical_stage2 = tf.keras.models.load_model(
    HIER_STAGE1_PATH,
    compile=False
)

print("Loaded:", hierarchical_stage2.name)


# ============================================================
# 3. FIND EFFICIENTNETB2 BACKBONE
# ============================================================

backbone_stage2 = None

for layer in hierarchical_stage2.layers:

    if isinstance(layer, tf.keras.Model):

        if "efficientnet" in layer.name.lower():

            backbone_stage2 = layer
            break


if backbone_stage2 is None:
    raise RuntimeError(
        "Could not locate EfficientNetB2 backbone."
    )

print("Backbone found:", backbone_stage2.name)


# ============================================================
# 4. FREEZE EVERYTHING FIRST
# ============================================================

backbone_stage2.trainable = True

for layer in backbone_stage2.layers:
    layer.trainable = False


# ============================================================
# 5. UNFREEZE LAST ~40 NON-BATCHNORM LAYERS
# ============================================================

eligible_layers = [
    layer
    for layer in backbone_stage2.layers
    if not isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    )
]

for layer in eligible_layers[-40:]:
    layer.trainable = True


# Explicitly keep BatchNorm frozen
for layer in backbone_stage2.layers:

    if isinstance(
        layer,
        tf.keras.layers.BatchNormalization
    ):
        layer.trainable = False


trainable_backbone_layers = [
    layer.name
    for layer in backbone_stage2.layers
    if layer.trainable
]


print(
    "\nTrainable backbone layers:",
    len(trainable_backbone_layers)
)

print("\nLast trainable layers:")

for name in trainable_backbone_layers[-10:]:
    print(" •", name)


# ============================================================
# 6. COMPILE WITH LOW LEARNING RATE
# ============================================================

hierarchical_stage2.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),

    loss={
        "three_output":
            "sparse_categorical_crossentropy",

        "nine_output":
            "sparse_categorical_crossentropy",

        "disease_output":
            "sparse_categorical_crossentropy"
    },

    loss_weights={
        "three_output": 0.10,
        "nine_output": 0.20,
        "disease_output": 1.00
    },

    metrics={

        "three_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "nine_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "disease_output": [

            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=3,
                name="top3"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=5,
                name="top5"
            )
        ]
    }
)


# ============================================================
# 7. CALLBACKS
# ============================================================

callbacks_h_stage2 = [

    tf.keras.callbacks.ModelCheckpoint(

        HIER_STAGE2_PATH,

        monitor="val_disease_output_accuracy",

        mode="max",

        save_best_only=True,

        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=HIER_STAGE2_BACKUP
    ),

    tf.keras.callbacks.EarlyStopping(

        monitor="val_disease_output_accuracy",

        mode="max",

        patience=4,

        restore_best_weights=True,

        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(

        monitor="val_disease_output_loss",

        mode="min",

        factor=0.3,

        patience=2,

        min_lr=1e-7,

        verbose=1
    )
]


# ============================================================
# 8. TRAIN
# ============================================================

print("\n==========================================")
print("HIERARCHICAL B2 — STAGE 2 FINE-TUNING")
print("==========================================")

print("Learning rate: 1e-5")
print("Maximum epochs: 15")
print("Backbone layers unfrozen:", len(trainable_backbone_layers))

print("\nStarting...\n")


history_h_stage2 = hierarchical_stage2.fit(

    train_ds_h,

    validation_data=val_ds_h,

    epochs=15,

    callbacks=callbacks_h_stage2,

    verbose=1
)


print("\n==========================================")
print("✅ HIERARCHICAL STAGE 2 COMPLETE")
print("==========================================")

print("\nBest model saved to:")
print(HIER_STAGE2_PATH)

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Loading best hierarchical Stage-1 model...
Loaded: DermaWise_Hierarchical_B2
Backbone found: efficientnetb2

Trainable backbone layers: 40

Last trainable layers:
 • block7b_se_squeeze
 • block7b_se_reshape
 • block7b_se_reduce
 • block7b_se_expand
 • block7b_se_excite
 • block7b_project_conv
 • block7b_drop
 • block7b_add
 • top_conv
 • top_activation

HIERARCHICAL B2 — STAGE 2 FINE-TUNING
Learning rate: 1e-5
Maximum epochs: 15
Backbone layers unfrozen: 40

Starting...

Epoch 1/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - disease_output_accuracy: 0.4385 - disease_output_loss: 2.1987 - disease_output_top3: 0.6418 - disease_output_top5: 0.7362 - loss: 2.4419 - nine_output_accuracy: 0.7087 - nine_output_loss: 0.9101 - three_output_accuracy: 0.7677 - three_output_loss: 0.6122
Epoch 1: val_disease_output_accuracy improved from None to 0.29236, saving model to /content/drive/MyDrive/DermaWise/Fitzpatrick114/dermawis

In [ ]:
import tensorflow as tf
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

HIER_STAGE2_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

# --------------------------------------------------
# GPU check
# --------------------------------------------------

assert tf.config.list_physical_devices("GPU"), \
    "GPU NOT AVAILABLE"

# --------------------------------------------------
# Load BEST Stage-2 checkpoint
# --------------------------------------------------

print("Loading best hierarchical Stage-2 model...")

best_hierarchical = tf.keras.models.load_model(
    HIER_STAGE2_PATH,
    compile=False
)

# --------------------------------------------------
# Compile for evaluation
# --------------------------------------------------

best_hierarchical.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),

    loss={
        "three_output": "sparse_categorical_crossentropy",
        "nine_output": "sparse_categorical_crossentropy",
        "disease_output": "sparse_categorical_crossentropy"
    },

    loss_weights={
        "three_output": 0.10,
        "nine_output": 0.20,
        "disease_output": 1.00
    },

    metrics={
        "three_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "nine_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],

        "disease_output": [
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=3,
                name="top3"
            ),

            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=5,
                name="top5"
            )
        ]
    }
)

# --------------------------------------------------
# Validation evaluation
# --------------------------------------------------

print("\nEvaluating best Stage-2 checkpoint...\n")

results_h2 = best_hierarchical.evaluate(
    val_ds_h,
    verbose=1,
    return_dict=True
)

top1 = results_h2["disease_output_accuracy"] * 100
top3 = results_h2["disease_output_top3"] * 100
top5 = results_h2["disease_output_top5"] * 100

print("\n========================================")
print("HIERARCHICAL STAGE-2 VALIDATION")
print("========================================")

print(f"114-class Top-1 : {top1:.2f}%")
print(f"114-class Top-3 : {top3:.2f}%")
print(f"114-class Top-5 : {top5:.2f}%")

print("\n===== COMPARISON =====")
print("Hierarchical Stage 1 : 28.71 / 44.86 / 54.08")
print(f"Hierarchical Stage 2 : {top1:.2f} / {top3:.2f} / {top5:.2f}")
print("Focal B2 benchmark   : 31.08 / 48.02 / 55.84")

print("\nAuxiliary heads:")
print(
    "3-way accuracy:",
    f"{results_h2['three_output_accuracy'] * 100:.2f}%"
)
print(
    "9-way accuracy:",
    f"{results_h2['nine_output_accuracy'] * 100:.2f}%"
)

Loading best hierarchical Stage-2 model...

Evaluating best Stage-2 checkpoint...

72/72 ━━━━━━━━━━━━━━━━━━━━ 13s 87ms/step - disease_output_accuracy: 0.3257 - disease_output_loss: 3.0897 - disease_output_top3: 0.4917 - disease_output_top5: 0.5768 - loss: 3.3266 - nine_output_accuracy: 0.7173 - nine_output_loss: 0.9352 - three_output_accuracy: 0.7867 - three_output_loss: 0.5743

HIERARCHICAL STAGE-2 VALIDATION
114-class Top-1 : 32.57%
114-class Top-3 : 49.17%
114-class Top-5 : 57.68%

===== COMPARISON =====
Hierarchical Stage 1 : 28.71 / 44.86 / 54.08
Hierarchical Stage 2 : 32.57 / 49.17 / 57.68
Focal B2 benchmark   : 31.08 / 48.02 / 55.84

Auxiliary heads:
3-way accuracy: 78.67%
9-way accuracy: 71.73%


In [ ]:
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TEST_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_test.csv"
)

# Load the already-saved official test manifest
test_df = pd.read_csv(TEST_CSV)

print("===== TEST DATAFRAME RESTORED =====")
print("Shape:", test_df.shape)
print("Images:", len(test_df))
print("\nColumns:")
print(test_df.columns.tolist())

# Check important columns
required = ["label", "image_path"]

print("\nRequired columns:")
for col in required:
    print(
        col,
        "✅" if col in test_df.columns else "❌"
    )

print("\nFirst row:")
print(test_df.iloc[0])

===== TEST DATAFRAME RESTORED =====
Shape: (2280, 17)
Images: 2280

Columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name', 'image_path', 'exists']

Required columns:
label ✅
image_path ✅

First row:
filepath                                       pa_59/pa_f4_17_4f4f5d2d.jpg
filename                                                 pa_f4_17_4f4f5d2d
diag                                                                    pa
fst                                                                      4
partition                                                             test
md5hash                                   4f4f5d2d87e965726350acd6b690eb0a
fitzpatrick                                                              4
label                                                           paronychia
nine_partition_label                            

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("CPU:", tf.config.list_physical_devices("CPU"))

print("\n✅ CPU runtime ready")

TensorFlow: 2.20.0
GPU: []
CPU: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

✅ CPU runtime ready


In [ ]:
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount("/content/drive")

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

# Paths
TEST_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_test.csv"
)

MODEL_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

# Load official Fitzpatrick17k-C test manifest
test_df = pd.read_csv(TEST_CSV)

print("\n===== RESTORE CHECK =====")
print("Test images:", len(test_df))
print("Test CSV exists:", os.path.exists(TEST_CSV))
print("Model exists:", os.path.exists(MODEL_PATH))

print("\nColumns:")
print(test_df.columns.tolist())

print("\nPartition:")
print(test_df["partition"].value_counts())

print("\nUnique diseases:", test_df["label"].nunique())

print("\n===== READY =====")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== RESTORE CHECK =====
Test images: 2280
Test CSV exists: True
Model exists: True

Columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name', 'image_path', 'exists']

Partition:
partition
test    2280
Name: count, dtype: int64

Unique diseases: 114

===== READY =====


In [ ]:
import os
import shutil
import time

LOCAL_TEST_DIR = "/content/DermaWise_Test_Images"

os.makedirs(
    LOCAL_TEST_DIR,
    exist_ok=True
)

print("===== COPYING TEST IMAGES TO LOCAL STORAGE =====")
print("Images to copy:", len(test_df))

start_time = time.time()

local_paths = []
missing = 0

for i, row in test_df.iterrows():

    source = row["image_path"]

    # Unique filename prevents collisions
    filename = f"{i:04d}_{os.path.basename(source)}"

    destination = os.path.join(
        LOCAL_TEST_DIR,
        filename
    )

    if os.path.exists(source):

        if not os.path.exists(destination):
            shutil.copy2(
                source,
                destination
            )

        local_paths.append(destination)

    else:
        local_paths.append(None)
        missing += 1

    # Progress
    if (i + 1) % 250 == 0:
        print(
            f"Copied/checked {i + 1}/{len(test_df)}"
        )


elapsed = time.time() - start_time

# Add local paths to dataframe
test_df["local_path"] = local_paths


print("\n====================================")
print("LOCAL TEST COPY COMPLETE")
print("====================================")

print("Expected images :", len(test_df))

print(
    "Local images    :",
    sum(
        path is not None
        and os.path.exists(path)
        for path in local_paths
    )
)

print("Missing images  :", missing)

print(
    "Time:",
    f"{elapsed / 60:.2f} minutes"
)

print("\nLocal folder:")
print(LOCAL_TEST_DIR)

===== COPYING TEST IMAGES TO LOCAL STORAGE =====
Images to copy: 2280
Copied/checked 250/2280
Copied/checked 500/2280
Copied/checked 750/2280
Copied/checked 1000/2280
Copied/checked 1250/2280
Copied/checked 1500/2280
Copied/checked 1750/2280
Copied/checked 2000/2280
Copied/checked 2250/2280

LOCAL TEST COPY COMPLETE
Expected images : 2280
Local images    : 2280
Missing images  : 0
Time: 10.10 minutes

Local folder:
/content/DermaWise_Test_Images


In [ ]:
import tensorflow as tf
import pandas as pd
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

MODEL_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

TRAIN_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

# --------------------------------------------------
# 1. Restore training metadata
# --------------------------------------------------

train_restore = pd.read_csv(TRAIN_CSV)

# Exact 114-class mapping used throughout our experiments
class_names = sorted(
    train_restore["label"].dropna().unique()
)

class_to_index = {
    name: i
    for i, name in enumerate(class_names)
}

# Restore hierarchy mappings
three_classes = sorted(
    train_restore["three_partition_label"]
    .dropna()
    .unique()
)

nine_classes = sorted(
    train_restore["nine_partition_label"]
    .dropna()
    .unique()
)

three_to_index = {
    name: i
    for i, name in enumerate(three_classes)
}

nine_to_index = {
    name: i
    for i, name in enumerate(nine_classes)
}

# --------------------------------------------------
# 2. Add IDs to test dataframe
# --------------------------------------------------

test_df["class_id"] = (
    test_df["label"]
    .map(class_to_index)
)

test_df["three_id"] = (
    test_df["three_partition_label"]
    .map(three_to_index)
)

test_df["nine_id"] = (
    test_df["nine_partition_label"]
    .map(nine_to_index)
)

print("===== MAPPING CHECK =====")
print("Disease classes:", len(class_names))
print("3-way classes:", len(three_classes))
print("9-way classes:", len(nine_classes))

print("\nMissing IDs:")
print("Disease:", test_df["class_id"].isna().sum())
print("3-way:", test_df["three_id"].isna().sum())
print("9-way:", test_df["nine_id"].isna().sum())

assert len(class_names) == 114
assert test_df["class_id"].isna().sum() == 0
assert test_df["three_id"].isna().sum() == 0
assert test_df["nine_id"].isna().sum() == 0

# --------------------------------------------------
# 3. Load saved Stage-2 model
# --------------------------------------------------

print("\nLoading hierarchical model...")
print("CPU loading may take a little time.")

best_hierarchical = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

print("\n===== MODEL RESTORED =====")
print("Model:", best_hierarchical.name)
print("Outputs:", best_hierarchical.output_names)
print("Parameters:", f"{best_hierarchical.count_params():,}")

print("\n✅ READY FOR FINAL CPU INFERENCE")

===== MAPPING CHECK =====
Disease classes: 114
3-way classes: 3
9-way classes: 9

Missing IDs:
Disease: 0
3-way: 0
9-way: 0

Loading hierarchical model...
CPU loading may take a little time.

===== MODEL RESTORED =====
Model: DermaWise_Hierarchical_B2
Outputs: ListWrapper(['disease_output', 'nine_output', 'three_output'])
Parameters: 7,946,103

✅ READY FOR FINAL CPU INFERENCE


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import time

from sklearn.metrics import (
    classification_report,
    accuracy_score
)

IMG_SIZE = 260
BATCH_SIZE = 16

# ============================================================
# 1. IMAGE LOADER
# ============================================================

def load_test_image(path, disease_id, three_id, nine_id):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE],
        antialias=False
    )

    image = tf.cast(image, tf.float32)

    return image, (
        tf.cast(disease_id, tf.int32),
        tf.cast(three_id, tf.int32),
        tf.cast(nine_id, tf.int32)
    )


# ============================================================
# 2. BUILD DATASET FROM LOCAL /content IMAGES
# ============================================================

test_ds_cpu = tf.data.Dataset.from_tensor_slices(
    (
        test_df["local_path"].astype(str).values,
        test_df["class_id"].astype("int32").values,
        test_df["three_id"].astype("int32").values,
        test_df["nine_id"].astype("int32").values
    )
)

test_ds_cpu = (
    test_ds_cpu
    .map(
        load_test_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)


# ============================================================
# 3. QUICK CHECK
# ============================================================

images_check, labels_check = next(iter(test_ds_cpu))

print("===== FINAL TEST DATASET =====")
print("Images:", images_check.shape)

print(
    "Pixel range:",
    float(tf.reduce_min(images_check)),
    "to",
    float(tf.reduce_max(images_check))
)

print("Test images:", len(test_df))
print("Expected batches:", int(np.ceil(len(test_df) / BATCH_SIZE)))


# ============================================================
# 4. SINGLE INFERENCE PASS
# ============================================================

print("\n========================================")
print("STARTING SINGLE-PASS CPU INFERENCE")
print("========================================")
print("CPU evaluation can take some time.")
print("Do not interrupt while batches are progressing.\n")

start = time.time()

prediction_output = best_hierarchical.predict(
    test_ds_cpu.map(
        lambda images, labels: images
    ),
    verbose=1
)

elapsed = time.time() - start

print(
    "\nInference time:",
    f"{elapsed / 60:.2f} minutes"
)


# ============================================================
# 5. EXTRACT THREE MODEL OUTPUTS
# ============================================================

if isinstance(prediction_output, dict):

    disease_probs = prediction_output["disease_output"]
    three_probs = prediction_output["three_output"]
    nine_probs = prediction_output["nine_output"]

else:

    output_map = dict(
        zip(
            best_hierarchical.output_names,
            prediction_output
        )
    )

    disease_probs = output_map["disease_output"]
    three_probs = output_map["three_output"]
    nine_probs = output_map["nine_output"]


print("\nPrediction shapes:")
print("Disease:", disease_probs.shape)
print("3-way:", three_probs.shape)
print("9-way:", nine_probs.shape)


# ============================================================
# 6. TRUE LABELS
# ============================================================

y_true = test_df["class_id"].astype(int).values
y_three_true = test_df["three_id"].astype(int).values
y_nine_true = test_df["nine_id"].astype(int).values


# ============================================================
# 7. TOP-1
# ============================================================

y_pred = np.argmax(
    disease_probs,
    axis=1
)

top1 = np.mean(
    y_pred == y_true
)


# ============================================================
# 8. TOP-3 AND TOP-5
# ============================================================

top3_indices = np.argsort(
    disease_probs,
    axis=1
)[:, -3:]

top5_indices = np.argsort(
    disease_probs,
    axis=1
)[:, -5:]

top3 = np.mean([
    true in preds
    for true, preds
    in zip(y_true, top3_indices)
])

top5 = np.mean([
    true in preds
    for true, preds
    in zip(y_true, top5_indices)
])


# ============================================================
# 9. AUXILIARY HEAD ACCURACY
# ============================================================

three_pred = np.argmax(
    three_probs,
    axis=1
)

nine_pred = np.argmax(
    nine_probs,
    axis=1
)

three_accuracy = np.mean(
    three_pred == y_three_true
)

nine_accuracy = np.mean(
    nine_pred == y_nine_true
)


# ============================================================
# 10. DETAILED 114-CLASS REPORT
# ============================================================

report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(114),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

macro_f1 = report["macro avg"]["f1-score"]
weighted_f1 = report["weighted avg"]["f1-score"]
macro_recall = report["macro avg"]["recall"]

class_report = report_df.loc[class_names]

zero_f1 = class_report[
    class_report["f1-score"] == 0
]


# ============================================================
# 11. SAVE RESULTS
# ============================================================

REPORT_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_hierarchical_final_classification_report.csv"
)

report_df.to_csv(
    REPORT_PATH,
    index=True
)

PREDICTION_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_hierarchical_test_predictions.csv"
)

prediction_df = test_df[
    [
        "label",
        "three_partition_label",
        "nine_partition_label"
    ]
].copy()

prediction_df["predicted_label"] = [
    class_names[i]
    for i in y_pred
]

prediction_df["confidence"] = np.max(
    disease_probs,
    axis=1
)

prediction_df.to_csv(
    PREDICTION_PATH,
    index=False
)


# ============================================================
# 12. FINAL RESULTS
# ============================================================

print("\n========================================")
print("FINAL HIERARCHICAL TEST SUMMARY")
print("========================================")

print(f"114-class Top-1 : {top1 * 100:.2f}%")
print(f"114-class Top-3 : {top3 * 100:.2f}%")
print(f"114-class Top-5 : {top5 * 100:.2f}%")

print(f"\nMacro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")
print(f"Macro Recall    : {macro_recall:.4f}")

print(
    f"\n3-way Accuracy  : "
    f"{three_accuracy * 100:.2f}%"
)

print(
    f"9-way Accuracy  : "
    f"{nine_accuracy * 100:.2f}%"
)

print(
    "\nZero-F1 classes:",
    len(zero_f1)
)

print("\n===== PREVIOUS FOCAL B2 =====")
print("Top-1          : 28.95%")
print("Top-3          : 47.06%")
print("Top-5          : 56.67%")
print("Macro F1       : 0.2722")
print("Weighted F1    : 0.2767")
print("Macro Recall   : 0.2792")
print("Zero-F1 classes: 10")

print("\nSaved report:")
print(REPORT_PATH)

print("\nSaved predictions:")
print(PREDICTION_PATH)

===== FINAL TEST DATASET =====
Images: (16, 260, 260, 3)
Pixel range: 0.0 to 255.0
Test images: 2280
Expected batches: 143

STARTING SINGLE-PASS CPU INFERENCE
CPU evaluation can take some time.
Do not interrupt while batches are progressing.

143/143 ━━━━━━━━━━━━━━━━━━━━ 376s 3s/step

Inference time: 6.27 minutes

Prediction shapes:
Disease: (2280, 114)
3-way: (2280, 3)
9-way: (2280, 9)

FINAL HIERARCHICAL TEST SUMMARY
114-class Top-1 : 30.44%
114-class Top-3 : 49.30%
114-class Top-5 : 57.89%

Macro F1        : 0.2784
Weighted F1     : 0.2882
Macro Recall    : 0.2785

3-way Accuracy  : 78.64%
9-way Accuracy  : 72.06%

Zero-F1 classes: 11

===== PREVIOUS FOCAL B2 =====
Top-1          : 28.95%
Top-3          : 47.06%
Top-5          : 56.67%
Macro F1       : 0.2722
Weighted F1    : 0.2767
Macro Recall   : 0.2792
Zero-F1 classes: 10

Saved report:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick114_hierarchical_final_classification_report.csv

Saved predictions:
/content/drive/M

In [ ]:
import os
import zipfile
from collections import Counter

AUTHORIZED_ZIP = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/fitzpatrick17k.zip"
)

print("===== AUTHORIZED FITZPATRICK17K CHECK =====")
print("ZIP exists:", os.path.exists(AUTHORIZED_ZIP))

if not os.path.exists(AUTHORIZED_ZIP):
    raise FileNotFoundError(AUTHORIZED_ZIP)

size_gb = os.path.getsize(AUTHORIZED_ZIP) / (1024 ** 3)
print(f"ZIP size: {size_gb:.2f} GB")

with zipfile.ZipFile(AUTHORIZED_ZIP, "r") as z:

    names = z.namelist()

    image_ext = (
        ".jpg", ".jpeg", ".png",
        ".bmp", ".webp"
    )

    images = [
        x for x in names
        if x.lower().endswith(image_ext)
    ]

    print("ZIP opened successfully: ✅")
    print("Total ZIP entries:", len(names))
    print("Image files:", len(images))

    print("\n===== TOP-LEVEL CONTENT =====")

    top = Counter(
        x.split("/")[0]
        for x in names
        if x
    )

    for name, count in top.most_common(30):
        print(name, ":", count)

    print("\n===== FIRST 30 ENTRIES =====")

    for x in names[:30]:
        print(x)

    print("\n===== SAMPLE IMAGE PATHS =====")

    for x in images[:20]:
        print(x)

print("\n✅ Inspection finished. ZIP was NOT extracted.")

===== AUTHORIZED FITZPATRICK17K CHECK =====
ZIP exists: True
ZIP size: 1.32 GB
ZIP opened successfully: ✅
Total ZIP entries: 16579
Image files: 16577

===== TOP-LEVEL CONTENT =====
data : 16579

===== FIRST 30 ENTRIES =====
data/
data/finalfitz17k/
data/finalfitz17k/8da0ad261c76d6ed8587ad2275664650.jpg
data/finalfitz17k/ae608b81693c2a1b5c2d9c7c471b0a4a.jpg
data/finalfitz17k/0ca870413a13e07dc10b44cb327e1289.jpg
data/finalfitz17k/c027d6925af7aa63d143b0c6da90de6e.jpg
data/finalfitz17k/08feca025dfb53bff8826d9c78c5fde8.jpg
data/finalfitz17k/9a7c1451aa6d667b5b4dc11e243f40f5.jpg
data/finalfitz17k/052963515e6a99d2a7f21d08926e2ca2.jpg
data/finalfitz17k/9a38dd9a7a532715e7ec5937763f359e.jpg
data/finalfitz17k/eb9acf3ecf71fd86dca542b99a2d0f53.jpg
data/finalfitz17k/a5a355561aa62dd374d3bd2e52926d66.jpg
data/finalfitz17k/e4694223d60c3b4763496396b4d78ede.jpg
data/finalfitz17k/ee1f23c38e96589ac0c642b44e76f52e.jpg
data/finalfitz17k/aa1ee4a91d7437fc05eecfa1b6f5f3b8.jpg
data/finalfitz17k/66f84860d3920287b8

In [ ]:
import pandas as pd
import zipfile
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

AUTHORIZED_ZIP = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/fitzpatrick17k.zip"
)

META_PATH = os.path.join(
    WORK_DIR,
    "Fitzpatrick17k-C.csv"
)

# --------------------------------------------------
# Corrected metadata
# --------------------------------------------------

meta_c = pd.read_csv(META_PATH)

corrected_hashes = set(
    meta_c["md5hash"]
    .dropna()
    .astype(str)
    .str.lower()
    .str.strip()
)

# --------------------------------------------------
# Authorized ZIP filenames
# --------------------------------------------------

with zipfile.ZipFile(AUTHORIZED_ZIP, "r") as z:

    authorized_images = [
        x for x in z.namelist()
        if x.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

authorized_hashes = set(
    os.path.splitext(
        os.path.basename(path)
    )[0].lower()
    for path in authorized_images
)

# --------------------------------------------------
# Compare
# --------------------------------------------------

overlap = authorized_hashes & corrected_hashes

authorized_only = (
    authorized_hashes - corrected_hashes
)

corrected_only = (
    corrected_hashes - authorized_hashes
)

print("===== DATASET OVERLAP CHECK =====")

print(
    "Authorized images:",
    len(authorized_hashes)
)

print(
    "Corrected metadata images:",
    len(corrected_hashes)
)

print(
    "Exact hash overlap:",
    len(overlap)
)

print(
    "Authorized images NOT in corrected metadata:",
    len(authorized_only)
)

print(
    "Corrected metadata images NOT in authorized ZIP:",
    len(corrected_only)
)

print(
    "\nCorrected metadata covered by authorized ZIP:",
    f"{len(overlap) / len(corrected_hashes) * 100:.2f}%"
)

print(
    "Authorized ZIP represented in corrected metadata:",
    f"{len(overlap) / len(authorized_hashes) * 100:.2f}%"
)

print("\nSample authorized-only hashes:")

for x in list(sorted(authorized_only))[:20]:
    print(x)

===== DATASET OVERLAP CHECK =====
Authorized images: 16577
Corrected metadata images: 11394
Exact hash overlap: 11394
Authorized images NOT in corrected metadata: 5183
Corrected metadata images NOT in authorized ZIP: 0

Corrected metadata covered by authorized ZIP: 100.00%
Authorized ZIP represented in corrected metadata: 68.73%

Sample authorized-only hashes:
000491af8dd4d739de520e8a68be7134
001c92cc05ea5be4afa0d0e1a0dbc834
002f4ec0b620339d58ccfa26c4268584
005b804472c7a27908f99e3d6d6cf91c
0066264501f73b1db09b3e2aa4dac0a0
00742b7d36ec4038d68e4956deb8fa83
00918f1c84f591a61d5e59600db86f05
0097275da3cb707415d13d2c59cf8c8c
00a5349478ef54f5445117b6c3221038
00a9084c6a733da061bd1a0d2c44db5c
00ab3a71e26ce200ded0aa6b60efda4e
00bd57ad3094d207c37ea8751f577a19
00bd904d29d626ead5f3642819c18918
00c3d96df4950c3842003743fac2a23b
00c5083c6fe716b4fc02a814856ec7e8
00c8a06d17cc1ab86d336168767c8278
00e2d3aa3bdad5ec550d3288543a9c2e
01011bb38480c92b637d4adc216122bd
01021bf7ee6a288a4eba76ed7a7bc9f0
010c97dc60

In [ ]:
import pandas as pd
import os

ORIGINAL_META = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/fitzpatrick17k.csv"
)

print("===== ORIGINAL METADATA CHECK =====")
print("Exists:", os.path.exists(ORIGINAL_META))

if not os.path.exists(ORIGINAL_META):
    raise FileNotFoundError(ORIGINAL_META)

original_meta = pd.read_csv(ORIGINAL_META)

print("Shape:", original_meta.shape)

print("\nColumns:")
print(original_meta.columns.tolist())

print("\nUnique labels:")

if "label" in original_meta.columns:
    print(original_meta["label"].nunique())

print("\nFirst row:")
print(original_meta.iloc[0])

===== ORIGINAL METADATA CHECK =====
Exists: True
Shape: (16577, 9)

Columns:
['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']

Unique labels:
114

First row:
md5hash                                   5e82a45bc5d78bd24ae9202d194423f8
fitzpatrick_scale                                                        3
fitzpatrick_centaur                                                      3
label                                      drug induced pigmentary changes
nine_partition_label                                          inflammatory
three_partition_label                                       non-neoplastic
qc                                                                     NaN
url                      https://www.dermaamin.com/site/images/clinical...
url_alphanum             httpwwwdermaamincomsiteimagesclinicalpicmminoc...
Name: 0, dtype: object


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Normalize hashes
# --------------------------------------------------

original_meta["md5hash_clean"] = (
    original_meta["md5hash"]
    .astype(str)
    .str.lower()
    .str.strip()
)

meta_c["md5hash_clean"] = (
    meta_c["md5hash"]
    .astype(str)
    .str.lower()
    .str.strip()
)

corrected_hash_set = set(
    meta_c["md5hash_clean"]
)

# --------------------------------------------------
# 2. Separate retained vs excluded records
# --------------------------------------------------

original_meta["in_corrected_C"] = (
    original_meta["md5hash_clean"]
    .isin(corrected_hash_set)
)

retained = original_meta[
    original_meta["in_corrected_C"]
].copy()

excluded = original_meta[
    ~original_meta["in_corrected_C"]
].copy()

print("===== ORIGINAL → CORRECTED COMPARISON =====")

print("Original records :", len(original_meta))
print("Retained in C    :", len(retained))
print("Excluded from C  :", len(excluded))

print(
    "Retained %       :",
    f"{len(retained) / len(original_meta) * 100:.2f}%"
)

print(
    "Excluded %       :",
    f"{len(excluded) / len(original_meta) * 100:.2f}%"
)


# --------------------------------------------------
# 3. How many conditions remain?
# --------------------------------------------------

print("\n===== CONDITION COVERAGE =====")

print(
    "Original conditions:",
    original_meta["label"].nunique()
)

print(
    "Retained conditions:",
    retained["label"].nunique()
)

print(
    "Excluded records span:",
    excluded["label"].nunique(),
    "conditions"
)


# --------------------------------------------------
# 4. Conditions with most exclusions
# --------------------------------------------------

original_counts = (
    original_meta["label"]
    .value_counts()
    .rename("original")
)

retained_counts = (
    retained["label"]
    .value_counts()
    .rename("retained")
)

excluded_counts = (
    excluded["label"]
    .value_counts()
    .rename("excluded")
)

comparison = pd.concat(
    [
        original_counts,
        retained_counts,
        excluded_counts
    ],
    axis=1
).fillna(0)

comparison = comparison.astype(int)

comparison["excluded_pct"] = (
    comparison["excluded"] /
    comparison["original"] * 100
)

comparison = comparison.sort_values(
    "excluded",
    ascending=False
)

print("\n===== TOP 25 CONDITIONS BY EXCLUDED IMAGES =====")

print(
    comparison.head(25).to_string()
)


# --------------------------------------------------
# 5. Quality-control field
# --------------------------------------------------

print("\n===== QC VALUES — EXCLUDED RECORDS =====")

print(
    excluded["qc"]
    .value_counts(dropna=False)
    .to_string()
)


print("\n===== QC VALUES — RETAINED RECORDS =====")

print(
    retained["qc"]
    .value_counts(dropna=False)
    .to_string()
)


# --------------------------------------------------
# 6. Fitzpatrick-scale distribution
# --------------------------------------------------

print("\n===== FITZPATRICK SCALE — EXCLUDED =====")

print(
    excluded["fitzpatrick_scale"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)


print("\n===== FITZPATRICK SCALE — RETAINED =====")

print(
    retained["fitzpatrick_scale"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)


# --------------------------------------------------
# 7. Save analysis
# --------------------------------------------------

EXCLUDED_PATH = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/"
    "fitzpatrick17k_original_excluded_from_C.csv"
)

COMPARISON_PATH = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/"
    "original_vs_corrected_condition_counts.csv"
)

excluded.to_csv(
    EXCLUDED_PATH,
    index=False
)

comparison.to_csv(
    COMPARISON_PATH
)

print("\n====================================")
print("✅ ANALYSIS COMPLETE")
print("====================================")

print("\nSaved excluded records:")
print(EXCLUDED_PATH)

print("\nSaved condition comparison:")
print(COMPARISON_PATH)

===== ORIGINAL → CORRECTED COMPARISON =====
Original records : 16577
Retained in C    : 11394
Excluded from C  : 5183
Retained %       : 68.73%
Excluded %       : 31.27%

===== CONDITION COVERAGE =====
Original conditions: 114
Retained conditions: 114
Excluded records span: 114 conditions

===== TOP 25 CONDITIONS BY EXCLUDED IMAGES =====
                                          original  retained  excluded  excluded_pct
label                                                                               
squamous cell carcinoma                        581       372       209     35.972461
folliculitis                                   342       182       160     46.783626
lupus erythematosus                            410       257       153     37.317073
nematode infection                             260       113       147     56.538462
scabies                                        339       203       136     40.117994
psoriasis                                      653       534     

In [ ]:
import os
import pandas as pd
import zipfile

# ============================================================
# PATHS
# ============================================================

BASE = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

AUTHORIZED_ZIP = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/fitzpatrick17k.zip"
)

ORIGINAL_META = os.path.join(
    BASE,
    "fitzpatrick17k.csv"
)

TRAIN_CSV = os.path.join(
    BASE,
    "fitzpatrick114_train.csv"
)

VAL_CSV = os.path.join(
    BASE,
    "fitzpatrick114_val.csv"
)

TEST_CSV = os.path.join(
    BASE,
    "fitzpatrick114_test.csv"
)

OUTPUT_CSV = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/"
    "dermawise_ssl_candidate_pool.csv"
)

# ============================================================
# LOAD DATA
# ============================================================

original = pd.read_csv(ORIGINAL_META)
train = pd.read_csv(TRAIN_CSV)
val = pd.read_csv(VAL_CSV)
test = pd.read_csv(TEST_CSV)

print("===== LOADED DATA =====")
print("Original :", len(original))
print("Train-C  :", len(train))
print("Val-C    :", len(val))
print("Test-C   :", len(test))

# ============================================================
# NORMALIZE HASHES
# ============================================================

def clean_hash(series):
    return (
        series.astype(str)
        .str.lower()
        .str.strip()
    )

original["hash"] = clean_hash(original["md5hash"])
train["hash"] = clean_hash(train["md5hash"])
val["hash"] = clean_hash(val["md5hash"])
test["hash"] = clean_hash(test["md5hash"])

train_hashes = set(train["hash"])
val_hashes = set(val["hash"])
test_hashes = set(test["hash"])

# ============================================================
# VERIFY OFFICIAL SPLITS ARE DISJOINT
# ============================================================

print("\n===== OFFICIAL SPLIT OVERLAP =====")

print(
    "Train ∩ Val :",
    len(train_hashes & val_hashes)
)

print(
    "Train ∩ Test:",
    len(train_hashes & test_hashes)
)

print(
    "Val ∩ Test  :",
    len(val_hashes & test_hashes)
)

# ============================================================
# VERIFY AUTHORIZED ZIP CONTENT
# ============================================================

with zipfile.ZipFile(AUTHORIZED_ZIP, "r") as z:

    zip_images = [
        x for x in z.namelist()
        if x.lower().endswith(
            (".jpg", ".jpeg", ".png")
        )
    ]

zip_hash_to_path = {
    os.path.splitext(
        os.path.basename(x)
    )[0].lower(): x
    for x in zip_images
}

authorized_hashes = set(
    zip_hash_to_path.keys()
)

print("\nAuthorized ZIP images:",
      len(authorized_hashes))

# ============================================================
# BUILD CANDIDATE SSL POOL
# ============================================================

# Absolutely exclude corrected validation/test images.
blocked_hashes = val_hashes | test_hashes

ssl = original[
    ~original["hash"].isin(blocked_hashes)
].copy()

# Must physically exist in authorized archive.
ssl = ssl[
    ssl["hash"].isin(authorized_hashes)
].copy()

# Add ZIP location.
ssl["zip_path"] = ssl["hash"].map(
    zip_hash_to_path
)

# Identify source.
ssl["source_type"] = "original_only"

ssl.loc[
    ssl["hash"].isin(train_hashes),
    "source_type"
] = "corrected_train"

# ============================================================
# SAFETY CHECKS
# ============================================================

ssl_hashes = set(ssl["hash"])

print("\n===== SSL CANDIDATE POOL =====")

print("Total SSL candidates:", len(ssl))

print(
    "Corrected training images:",
    (ssl["source_type"] ==
     "corrected_train").sum()
)

print(
    "Original-only images:",
    (ssl["source_type"] ==
     "original_only").sum()
)

print("\n===== LEAKAGE CHECK =====")

print(
    "SSL ∩ corrected validation:",
    len(ssl_hashes & val_hashes)
)

print(
    "SSL ∩ corrected test:",
    len(ssl_hashes & test_hashes)
)

assert len(ssl_hashes & val_hashes) == 0
assert len(ssl_hashes & test_hashes) == 0

assert ssl["zip_path"].notna().all()

# ============================================================
# SAVE
# ============================================================

ssl.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:")
print(OUTPUT_CSV)

print("\n====================================")
print("✅ SSL CANDIDATE POOL CREATED")
print("====================================")

===== LOADED DATA =====
Original : 16577
Train-C  : 7975
Val-C    : 1139
Test-C   : 2280

===== OFFICIAL SPLIT OVERLAP =====
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Authorized ZIP images: 16577

===== SSL CANDIDATE POOL =====
Total SSL candidates: 13158
Corrected training images: 7975
Original-only images: 5183

===== LEAKAGE CHECK =====
SSL ∩ corrected validation: 0
SSL ∩ corrected test: 0

Saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114 authorised/dermawise_ssl_candidate_pool.csv

✅ SSL CANDIDATE POOL CREATED


In [ ]:
import os
import pandas as pd

BASE = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

SSL_CANDIDATE = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/"
    "dermawise_ssl_candidate_pool.csv"
)

STRICT_SSL_OUT = os.path.join(
    BASE,
    "fitzpatrick114_ssl_strict_train.csv"
)

# --------------------------------------------------
# LOAD CANDIDATE POOL
# --------------------------------------------------

ssl = pd.read_csv(SSL_CANDIDATE)

print("Candidate pool:", len(ssl))

# --------------------------------------------------
# KEEP ONLY OFFICIAL CORRECTED TRAINING IMAGES
# --------------------------------------------------

strict_ssl = ssl[
    ssl["source_type"] == "corrected_train"
].copy()

print("\n===== STRICT SSL DATASET =====")
print("SSL images:", len(strict_ssl))
print("Unique hashes:", strict_ssl["hash"].nunique())

# --------------------------------------------------
# VERIFY
# --------------------------------------------------

assert len(strict_ssl) == 7975
assert strict_ssl["hash"].nunique() == 7975

print("\nSource:")
print(strict_ssl["source_type"].value_counts())

# --------------------------------------------------
# SAVE
# --------------------------------------------------

strict_ssl.to_csv(
    STRICT_SSL_OUT,
    index=False
)

print("\nSaved:")
print(STRICT_SSL_OUT)

print("\n✅ STRICT SSL MANIFEST READY")

Candidate pool: 13158

===== STRICT SSL DATASET =====
SSL images: 7975
Unique hashes: 7975

Source:
source_type
corrected_train    7975
Name: count, dtype: int64

Saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/fitzpatrick114_ssl_strict_train.csv

✅ STRICT SSL MANIFEST READY


In [ ]:
import os
import zipfile
import pandas as pd
import shutil

# ============================================================
# PATHS
# ============================================================

STRICT_MANIFEST = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "fitzpatrick114_ssl_strict_train.csv"
)

AUTHORIZED_ZIP = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114 authorised/fitzpatrick17k.zip"
)

LOCAL_SSL_DIR = "/content/dermawise_ssl_train"

# ============================================================
# LOAD MANIFEST
# ============================================================

ssl_df = pd.read_csv(STRICT_MANIFEST)

print("===== SSL EXTRACTION =====")
print("Manifest images:", len(ssl_df))

assert len(ssl_df) == 7975

# ============================================================
# START CLEAN LOCAL DIRECTORY
# ============================================================

if os.path.exists(LOCAL_SSL_DIR):
    shutil.rmtree(LOCAL_SSL_DIR)

os.makedirs(
    LOCAL_SSL_DIR,
    exist_ok=True
)

# ============================================================
# EXTRACT ONLY REQUIRED IMAGES
# ============================================================

extracted = 0
missing = []

with zipfile.ZipFile(AUTHORIZED_ZIP, "r") as z:

    zip_names = set(z.namelist())

    for _, row in ssl_df.iterrows():

        zip_path = str(row["zip_path"])

        if zip_path not in zip_names:
            missing.append(zip_path)
            continue

        filename = os.path.basename(zip_path)

        destination = os.path.join(
            LOCAL_SSL_DIR,
            filename
        )

        with z.open(zip_path) as src, \
             open(destination, "wb") as dst:

            shutil.copyfileobj(src, dst)

        extracted += 1

# ============================================================
# VERIFY
# ============================================================

local_files = [
    x for x in os.listdir(LOCAL_SSL_DIR)
    if x.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
]

print("\n===== RESULT =====")

print("Extracted :", extracted)
print("Local files:", len(local_files))
print("Missing   :", len(missing))

if missing:
    print("\nFirst missing paths:")
    for x in missing[:10]:
        print(x)

assert extracted == 7975
assert len(local_files) == 7975
assert len(missing) == 0

print("\nLocal SSL folder:")
print(LOCAL_SSL_DIR)

print("\n✅ ALL 7,975 STRICT SSL IMAGES READY")

===== SSL EXTRACTION =====
Manifest images: 7975

===== RESULT =====
Extracted : 7975
Local files: 7975
Missing   : 0

Local SSL folder:
/content/dermawise_ssl_train

✅ ALL 7,975 STRICT SSL IMAGES READY


In [ ]:
import os
import tensorflow as tf
import numpy as np

# ============================================================
# BASIC SETTINGS
# ============================================================

SSL_DIR = "/content/dermawise_ssl_train"

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

print("TensorFlow:", tf.__version__)

# ============================================================
# GPU CHECK
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

print("\n===== HARDWARE =====")

if gpus:
    print("✅ GPU AVAILABLE")
    for gpu in gpus:
        print(gpu)
else:
    print("⚠️ GPU NOT AVAILABLE")
    print("Do NOT start full SSL training yet.")

# ============================================================
# FIND IMAGES
# ============================================================

image_paths = []

for filename in os.listdir(SSL_DIR):

    if filename.lower().endswith(
        (".jpg", ".jpeg", ".png")
    ):
        image_paths.append(
            os.path.join(
                SSL_DIR,
                filename
            )
        )

image_paths = sorted(image_paths)

print("\n===== DATASET =====")
print("Images found:", len(image_paths))

assert len(image_paths) == 7975

# ============================================================
# IMAGE LOADER
# ============================================================

def load_image(path):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image

# ============================================================
# SIMCLR-STYLE AUGMENTATION
# ============================================================

def augment_view(image):

    image = tf.image.random_flip_left_right(
        image
    )

    image = tf.image.random_flip_up_down(
        image
    )

    image = tf.image.random_brightness(
        image,
        max_delta=0.15 * 255.0
    )

    image = tf.image.random_contrast(
        image,
        lower=0.75,
        upper=1.25
    )

    image = tf.image.random_saturation(
        image,
        lower=0.75,
        upper=1.25
    )

    image = tf.image.random_hue(
        image,
        max_delta=0.05
    )

    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    return image

# ============================================================
# CREATE TWO DIFFERENT VIEWS
# ============================================================

def make_ssl_pair(path):

    image = load_image(path)

    view_1 = augment_view(image)
    view_2 = augment_view(image)

    return view_1, view_2

# ============================================================
# DATASET PIPELINE
# ============================================================

ssl_ds = tf.data.Dataset.from_tensor_slices(
    image_paths
)

ssl_ds = ssl_ds.shuffle(
    len(image_paths),
    seed=SEED,
    reshuffle_each_iteration=True
)

ssl_ds = ssl_ds.map(
    make_ssl_pair,
    num_parallel_calls=tf.data.AUTOTUNE
)

ssl_ds = ssl_ds.batch(
    BATCH_SIZE,
    drop_remainder=True
)

ssl_ds = ssl_ds.prefetch(
    tf.data.AUTOTUNE
)

# ============================================================
# SANITY TEST ONE BATCH
# ============================================================

view1, view2 = next(iter(ssl_ds))

print("\n===== SSL BATCH TEST =====")

print(
    "View 1 shape:",
    view1.shape
)

print(
    "View 2 shape:",
    view2.shape
)

print(
    "View 1 range:",
    float(tf.reduce_min(view1)),
    "to",
    float(tf.reduce_max(view1))
)

print(
    "View 2 range:",
    float(tf.reduce_min(view2)),
    "to",
    float(tf.reduce_max(view2))
)

difference = tf.reduce_mean(
    tf.abs(view1 - view2)
)

print(
    "Average difference between views:",
    float(difference)
)

assert view1.shape == (
    BATCH_SIZE,
    IMG_SIZE,
    IMG_SIZE,
    3
)

assert view2.shape == (
    BATCH_SIZE,
    IMG_SIZE,
    IMG_SIZE,
    3
)

print("\n✅ TWO-VIEW SSL PIPELINE WORKS")

TensorFlow: 2.20.0

===== HARDWARE =====
⚠️ GPU NOT AVAILABLE
Do NOT start full SSL training yet.

===== DATASET =====
Images found: 7975

===== SSL BATCH TEST =====
View 1 shape: (32, 224, 224, 3)
View 2 shape: (32, 224, 224, 3)
View 1 range: 0.0 to 255.0
View 2 range: 0.0 to 255.0
Average difference between views: 49.14908218383789

✅ TWO-VIEW SSL PIPELINE WORKS


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# SETTINGS
# ============================================================

IMG_SIZE = 224
PROJECTION_DIM = 128
TEMPERATURE = 0.1

# ============================================================
# 1. SSL ENCODER
# ============================================================

# IMPORTANT:
# weights=None because this is our actual self-supervised experiment.
# We are NOT starting from ImageNet weights.

backbone = keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights=None,
    include_preprocessing=True
)

inputs = keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    name="image"
)

x = backbone(
    inputs,
    training=True
)

x = layers.GlobalAveragePooling2D(
    name="ssl_global_pool"
)(x)

encoder = keras.Model(
    inputs,
    x,
    name="dermawise_ssl_encoder"
)

# ============================================================
# 2. PROJECTION HEAD
# ============================================================

projection_inputs = keras.Input(
    shape=(encoder.output_shape[-1],)
)

x = layers.Dense(
    256,
    activation="relu"
)(projection_inputs)

x = layers.Dense(
    PROJECTION_DIM
)(x)

projection_head = keras.Model(
    projection_inputs,
    x,
    name="ssl_projection_head"
)

# ============================================================
# 3. FORWARD PASS
# ============================================================

def project(images, training=True):

    features = encoder(
        images,
        training=training
    )

    projections = projection_head(
        features,
        training=training
    )

    projections = tf.math.l2_normalize(
        projections,
        axis=1
    )

    return projections

# ============================================================
# 4. NT-XENT / SIMCLR CONTRASTIVE LOSS
# ============================================================

def contrastive_loss(z1, z2, temperature=TEMPERATURE):

    batch_size = tf.shape(z1)[0]

    z = tf.concat(
        [z1, z2],
        axis=0
    )

    similarity = tf.matmul(
        z,
        z,
        transpose_b=True
    )

    similarity = similarity / temperature

    total = 2 * batch_size

    # Remove self-similarity.
    mask = tf.eye(
        total,
        dtype=tf.bool
    )

    similarity = tf.where(
        mask,
        tf.constant(-1e9, dtype=similarity.dtype),
        similarity
    )

    # Positive pair:
    # view1[i] <-> view2[i]
    labels = tf.concat(
        [
            tf.range(
                batch_size,
                total
            ),
            tf.range(
                0,
                batch_size
            )
        ],
        axis=0
    )

    loss = tf.keras.losses.sparse_categorical_crossentropy(
        labels,
        similarity,
        from_logits=True
    )

    return tf.reduce_mean(loss)

# ============================================================
# 5. TEST ONE EXISTING SSL BATCH
# ============================================================

view1, view2 = next(iter(ssl_ds))

z1 = project(
    view1,
    training=True
)

z2 = project(
    view2,
    training=True
)

loss = contrastive_loss(
    z1,
    z2
)

# ============================================================
# 6. RESULTS
# ============================================================

print("===== SSL MODEL SANITY TEST =====")

print(
    "Encoder output:",
    encoder.output_shape
)

print(
    "Projection shape:",
    z1.shape
)

print(
    "Contrastive loss:",
    float(loss)
)

print(
    "Encoder parameters:",
    f"{encoder.count_params():,}"
)

print(
    "Projection parameters:",
    f"{projection_head.count_params():,}"
)

assert z1.shape == (
    BATCH_SIZE,
    PROJECTION_DIM
)

assert z2.shape == (
    BATCH_SIZE,
    PROJECTION_DIM
)

assert tf.math.is_finite(loss)

print("\n✅ SSL MODEL + NT-XENT LOSS WORKS")
print("⚠️ Full SSL training has NOT been started.")

===== SSL MODEL SANITY TEST =====
Encoder output: (None, 576)
Projection shape: (32, 128)
Contrastive loss: 3.9820988178253174
Encoder parameters: 939,120
Projection parameters: 180,608

✅ SSL MODEL + NT-XENT LOSS WORKS
⚠️ Full SSL training has NOT been started.


In [ ]:
import os

SSL_MODEL_DIR = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/SSL"
)

os.makedirs(
    SSL_MODEL_DIR,
    exist_ok=True
)

ENCODER_PATH = os.path.join(
    SSL_MODEL_DIR,
    "dermawise_ssl_encoder_initial.keras"
)

PROJECTION_PATH = os.path.join(
    SSL_MODEL_DIR,
    "dermawise_ssl_projection_initial.keras"
)

encoder.save(ENCODER_PATH)
projection_head.save(PROJECTION_PATH)

print("===== SSL SETUP SAVED =====")

print("\nEncoder:")
print(ENCODER_PATH)

print("\nProjection head:")
print(PROJECTION_PATH)

print("\n✅ INITIAL SSL MODELS SAVED")
print("⚠️ These are NOT trained SSL models.")

===== SSL SETUP SAVED =====

Encoder:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/SSL/dermawise_ssl_encoder_initial.keras

Projection head:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/SSL/dermawise_ssl_projection_initial.keras

✅ INITIAL SSL MODELS SAVED
⚠️ These are NOT trained SSL models.


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras

TEACHER_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

print("===== KD TEACHER CHECK =====")
print("Exists:", os.path.exists(TEACHER_PATH))

assert os.path.exists(TEACHER_PATH), \
    "Teacher model file not found."

teacher_full = keras.models.load_model(
    TEACHER_PATH,
    compile=False
)

print("\nModel loaded: ✅")
print("Model name:", teacher_full.name)

print("\n===== OUTPUTS =====")

print("Output names:")
print(teacher_full.output_names)

for name, output in zip(
    teacher_full.output_names,
    teacher_full.outputs
):
    print(
        name,
        "->",
        output.shape
    )

print(
    "\nTotal parameters:",
    f"{teacher_full.count_params():,}"
)

print("\n✅ TEACHER READY FOR KD INSPECTION")

===== KD TEACHER CHECK =====
Exists: True

Model loaded: ✅
Model name: DermaWise_Hierarchical_B2

===== OUTPUTS =====
Output names:
ListWrapper(['disease_output', 'nine_output', 'three_output'])
disease_output -> (None, 114)
nine_output -> (None, 9)
three_output -> (None, 3)

Total parameters: 7,946,103

✅ TEACHER READY FOR KD INSPECTION


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 260
NUM_CLASSES = 114

# ============================================================
# 1. EXTRACT 114-CLASS TEACHER
# ============================================================

teacher_disease = keras.Model(
    inputs=teacher_full.input,
    outputs=teacher_full.get_layer(
        "disease_output"
    ).output,
    name="DermaWise_KD_Teacher"
)

teacher_disease.trainable = False

print("===== TEACHER DISEASE MODEL =====")
print("Input :", teacher_disease.input_shape)
print("Output:", teacher_disease.output_shape)
print("Parameters:", f"{teacher_disease.count_params():,}")

assert teacher_disease.output_shape[-1] == 114

# ============================================================
# 2. CREATE LIGHTWEIGHT STUDENT
# ============================================================

student_backbone = keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True
)

student_inputs = keras.Input(
    shape=(IMG_SIZE, IMG_SIZE, 3),
    name="student_image"
)

x = student_backbone(
    student_inputs
)

x = layers.GlobalAveragePooling2D(
    name="student_global_pool"
)(x)

x = layers.Dropout(
    0.30,
    name="student_dropout"
)(x)

# IMPORTANT:
# No softmax here.
# KD works more cleanly with logits.
student_logits = layers.Dense(
    NUM_CLASSES,
    activation=None,
    name="student_logits"
)(x)

student = keras.Model(
    student_inputs,
    student_logits,
    name="DermaWise_MobileNetV3Small_Student"
)

# ============================================================
# 3. VERIFY STUDENT
# ============================================================

print("\n===== STUDENT =====")
print("Input :", student.input_shape)
print("Output:", student.output_shape)
print("Parameters:", f"{student.count_params():,}")

assert student.output_shape[-1] == 114

# ============================================================
# 4. PARAMETER REDUCTION
# ============================================================

teacher_params = teacher_disease.count_params()
student_params = student.count_params()

reduction = (
    1 - student_params / teacher_params
) * 100

print("\n===== SIZE COMPARISON =====")
print("Teacher:", f"{teacher_params:,}")
print("Student:", f"{student_params:,}")
print(
    "Parameter reduction:",
    f"{reduction:.2f}%"
)

# ============================================================
# 5. ONE DUMMY FORWARD PASS
# ============================================================

dummy = tf.zeros(
    (1, IMG_SIZE, IMG_SIZE, 3),
    dtype=tf.float32
)

teacher_probs = teacher_disease(
    dummy,
    training=False
)

student_test_logits = student(
    dummy,
    training=False
)

student_probs = tf.nn.softmax(
    student_test_logits,
    axis=-1
)

print("\n===== FORWARD PASS =====")
print("Teacher:", teacher_probs.shape)
print("Student:", student_probs.shape)

print(
    "Teacher probability sum:",
    float(tf.reduce_sum(teacher_probs[0]))
)

print(
    "Student probability sum:",
    float(tf.reduce_sum(student_probs[0]))
)

assert teacher_probs.shape == (1, 114)
assert student_probs.shape == (1, 114)

print("\n✅ KD TEACHER + STUDENT ARE COMPATIBLE")
print("⚠️ Knowledge distillation training has NOT started.")

===== TEACHER DISEASE MODEL =====
Input : (None, 260, 260, 3)
Output: (None, 114)
Parameters: 7,929,195


/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

===== STUDENT =====
Input : (None, 260, 260, 3)
Output: (None, 114)
Parameters: 1,004,898

===== SIZE COMPARISON =====
Teacher: 7,929,195
Student: 1,004,898
Parameter reduction: 87.33%

===== FORWARD PASS =====
Teacher: (1, 114)
Student: (1, 114)
Teacher probability sum: 1.0
Student probability sum: 0.9999998211860657

✅ KD TEACHER + STUDENT ARE COMPATIBLE
⚠️ Knowledge distillation training has NOT started.


In [ ]:
import os

KD_DIR = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/KD"
)

os.makedirs(
    KD_DIR,
    exist_ok=True
)

STUDENT_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_initial.keras"
)

student.save(STUDENT_PATH)

print("===== KD SETUP SAVED =====")

print("Student:")
print(STUDENT_PATH)

print(
    "\nStudent parameters:",
    f"{student.count_params():,}"
)

print(
    "Teacher parameters:",
    f"{teacher_disease.count_params():,}"
)

print(
    "Parameter reduction:",
    "87.33%"
)

print("\n✅ INITIAL KD STUDENT SAVED")
print("⚠️ Student has NOT been knowledge-distilled yet.")

===== KD SETUP SAVED =====
Student:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_initial.keras

Student parameters: 1,004,898
Teacher parameters: 7,929,195
Parameter reduction: 87.33%

✅ INITIAL KD STUDENT SAVED
⚠️ Student has NOT been knowledge-distilled yet.


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras

KD_DIR = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/KD"
)

STUDENT_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_initial.keras"
)

TFLITE_DIR = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/TFLite"
)

os.makedirs(
    TFLITE_DIR,
    exist_ok=True
)

TFLITE_PATH = os.path.join(
    TFLITE_DIR,
    "dermawise_student_initial_float16.tflite"
)

# ============================================================
# LOAD STUDENT
# ============================================================

student_for_tflite = keras.models.load_model(
    STUDENT_PATH,
    compile=False
)

print("===== TFLITE CONVERSION =====")
print("Keras model loaded: ✅")

# ============================================================
# CONVERTER
# ============================================================

converter = tf.lite.TFLiteConverter.from_keras_model(
    student_for_tflite
)

# Float16 weight quantization
converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.target_spec.supported_types = [
    tf.float16
]

# ============================================================
# CONVERT
# ============================================================

tflite_model = converter.convert()

with open(
    TFLITE_PATH,
    "wb"
) as f:
    f.write(tflite_model)

# ============================================================
# FILE SIZE
# ============================================================

keras_mb = (
    os.path.getsize(STUDENT_PATH)
    / (1024 ** 2)
)

tflite_mb = (
    os.path.getsize(TFLITE_PATH)
    / (1024 ** 2)
)

reduction = (
    1 - tflite_mb / keras_mb
) * 100

print("\n===== FILE SIZE =====")

print(
    "Keras student:",
    f"{keras_mb:.2f} MB"
)

print(
    "Float16 TFLite:",
    f"{tflite_mb:.2f} MB"
)

print(
    "File-size reduction:",
    f"{reduction:.2f}%"
)

print("\nSaved:")
print(TFLITE_PATH)

print("\n✅ FLOAT16 TFLITE CONVERSION SUCCESSFUL")
print(
    "⚠️ This is the INITIAL student, "
    "not the final KD-trained model."
)

===== TFLITE CONVERSION =====
Keras model loaded: ✅
Saved artifact at '/tmp/tmp4ug_a5gq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name='student_image')
Output Type:
  TensorSpec(shape=(None, 114), dtype=tf.float32, name=None)
Captures:
  135484866919696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866914896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866919312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866918352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866918544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866918160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866918736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866918928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866919888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135484866915856: TensorSpec(s

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from PIL import Image

# ============================================================
# PATHS
# ============================================================

STUDENT_PATH = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/KD/"
    "dermawise_kd_student_initial.keras"
)

TFLITE_PATH = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/TFLite/"
    "dermawise_student_initial_float16.tflite"
)

SSL_DIR = "/content/dermawise_ssl_train"

IMG_SIZE = 260

# ============================================================
# VERIFY FILES
# ============================================================

print("===== FILE CHECK =====")

print(
    "Keras student:",
    os.path.exists(STUDENT_PATH)
)

print(
    "TFLite model:",
    os.path.exists(TFLITE_PATH)
)

assert os.path.exists(STUDENT_PATH)
assert os.path.exists(TFLITE_PATH)

# ============================================================
# SELECT ONE REAL IMAGE
# ============================================================

image_files = sorted([
    x for x in os.listdir(SSL_DIR)
    if x.lower().endswith(
        (".jpg", ".jpeg", ".png")
    )
])

assert len(image_files) > 0

TEST_IMAGE = os.path.join(
    SSL_DIR,
    image_files[0]
)

print("\nTest image:")
print(TEST_IMAGE)

# ============================================================
# PREPARE IMAGE
# ============================================================

image = Image.open(
    TEST_IMAGE
).convert("RGB")

image = image.resize(
    (IMG_SIZE, IMG_SIZE)
)

image_array = np.asarray(
    image,
    dtype=np.float32
)

image_array = np.expand_dims(
    image_array,
    axis=0
)

print(
    "\nInput shape:",
    image_array.shape
)

print(
    "Input range:",
    image_array.min(),
    "to",
    image_array.max()
)

# ============================================================
# KERAS INFERENCE
# ============================================================

keras_student = keras.models.load_model(
    STUDENT_PATH,
    compile=False
)

keras_logits = keras_student.predict(
    image_array,
    verbose=0
)

keras_probs = tf.nn.softmax(
    keras_logits,
    axis=-1
).numpy()

# ============================================================
# TFLITE INTERPRETER
# ============================================================

interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

input_details = (
    interpreter.get_input_details()
)

output_details = (
    interpreter.get_output_details()
)

print("\n===== TFLITE MODEL =====")

print(
    "Input:",
    input_details[0]["shape"],
    input_details[0]["dtype"]
)

print(
    "Output:",
    output_details[0]["shape"],
    output_details[0]["dtype"]
)

# ============================================================
# TFLITE INFERENCE
# ============================================================

tflite_input = image_array.astype(
    input_details[0]["dtype"]
)

interpreter.set_tensor(
    input_details[0]["index"],
    tflite_input
)

interpreter.invoke()

tflite_logits = interpreter.get_tensor(
    output_details[0]["index"]
)

tflite_probs = tf.nn.softmax(
    tflite_logits,
    axis=-1
).numpy()

# ============================================================
# COMPARE
# ============================================================

keras_class = int(
    np.argmax(keras_probs[0])
)

tflite_class = int(
    np.argmax(tflite_probs[0])
)

max_difference = float(
    np.max(
        np.abs(
            keras_probs -
            tflite_probs
        )
    )
)

mean_difference = float(
    np.mean(
        np.abs(
            keras_probs -
            tflite_probs
        )
    )
)

print("\n===== COMPARISON =====")

print(
    "Keras predicted index:",
    keras_class
)

print(
    "TFLite predicted index:",
    tflite_class
)

print(
    "Same Top-1 prediction:",
    keras_class == tflite_class
)

print(
    "Maximum probability difference:",
    max_difference
)

print(
    "Mean probability difference:",
    mean_difference
)

assert keras_probs.shape == (1, 114)
assert tflite_probs.shape == (1, 114)

assert np.all(
    np.isfinite(tflite_probs)
)

print("\n✅ TFLITE MODEL EXECUTES SUCCESSFULLY")

if keras_class == tflite_class:
    print(
        "✅ Keras and TFLite Top-1 predictions match"
    )
else:
    print(
        "⚠️ Top-1 differs — inspect numerical differences"
    )

print(
    "\n⚠️ Remember: this verifies deployment only."
)
print(
    "The student is still UNTRAINED for the 114-class task."
)

===== FILE CHECK =====
Keras student: True
TFLite model: True

Test image:
/content/dermawise_ssl_train/0009d89a58dac26bb3c113744b7b4196.jpg

Input shape: (1, 260, 260, 3)
Input range: 0.0 to 255.0

===== TFLITE MODEL =====
Input: [  1 260 260   3] <class 'numpy.float32'>
Output: [  1 114] <class 'numpy.float32'>

===== COMPARISON =====
Keras predicted index: 88
TFLite predicted index: 88
Same Top-1 prediction: True
Maximum probability difference: 0.0008307099342346191
Mean probability difference: 5.3090865549165756e-05

✅ TFLITE MODEL EXECUTES SUCCESSFULLY
✅ Keras and TFLite Top-1 predictions match

⚠️ Remember: this verifies deployment only.
The student is still UNTRAINED for the 114-class task.


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import tensorflow as tf
from tensorflow import keras

# ============================================================
# KD SETTINGS
# ============================================================

TEMPERATURE = 3.0
ALPHA = 0.5

# ============================================================
# DISTILLER
# ============================================================

class Distiller(keras.Model):

    def __init__(
        self,
        student,
        teacher
    ):
        super().__init__()

        self.student = student
        self.teacher = teacher

        self.loss_tracker = keras.metrics.Mean(
            name="loss"
        )

        self.student_loss_tracker = keras.metrics.Mean(
            name="student_loss"
        )

        self.distillation_loss_tracker = keras.metrics.Mean(
            name="distillation_loss"
        )

        self.accuracy = (
            keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        )

    @property
    def metrics(self):

        return [
            self.loss_tracker,
            self.student_loss_tracker,
            self.distillation_loss_tracker,
            self.accuracy
        ]

    def compile(
        self,
        optimizer,
        student_loss_fn,
        distillation_loss_fn,
        alpha=0.5,
        temperature=3.0
    ):

        super().compile(
            optimizer=optimizer
        )

        self.student_loss_fn = (
            student_loss_fn
        )

        self.distillation_loss_fn = (
            distillation_loss_fn
        )

        self.alpha = alpha
        self.temperature = temperature

    def train_step(
        self,
        data
    ):

        # Supports:
        # (images, labels)
        # OR
        # (images, labels, sample_weight)

        if len(data) == 3:

            images, labels, sample_weight = data

        else:

            images, labels = data
            sample_weight = None

        # ----------------------------------------
        # Teacher predictions
        # ----------------------------------------

        teacher_probs = self.teacher(
            images,
            training=False
        )

        # Teacher currently ends in softmax.
        # Recover stable log-probabilities.
        teacher_log_probs = tf.math.log(
            tf.clip_by_value(
                teacher_probs,
                1e-7,
                1.0
            )
        )

        # ----------------------------------------
        # Student forward pass
        # ----------------------------------------

        with tf.GradientTape() as tape:

            student_logits = self.student(
                images,
                training=True
            )

            # HARD LABEL LOSS
            student_loss = self.student_loss_fn(
                labels,
                student_logits,
                sample_weight=sample_weight
            )

            # ------------------------------------
            # TEMPERATURE-SOFTENED TEACHER
            # ------------------------------------

            soft_teacher = tf.nn.softmax(
                teacher_log_probs /
                self.temperature,
                axis=-1
            )

            soft_student = tf.nn.softmax(
                student_logits /
                self.temperature,
                axis=-1
            )

            # ------------------------------------
            # KD LOSS
            # ------------------------------------

            distillation_loss = (
                self.distillation_loss_fn(
                    soft_teacher,
                    soft_student
                )
                *
                (self.temperature ** 2)
            )

            total_loss = (
                self.alpha * student_loss
                +
                (1.0 - self.alpha)
                * distillation_loss
            )

        # ----------------------------------------
        # BACKPROPAGATION
        # ----------------------------------------

        trainable_vars = (
            self.student.trainable_variables
        )

        gradients = tape.gradient(
            total_loss,
            trainable_vars
        )

        self.optimizer.apply_gradients(
            zip(
                gradients,
                trainable_vars
            )
        )

        # ----------------------------------------
        # METRICS
        # ----------------------------------------

        self.loss_tracker.update_state(
            total_loss
        )

        self.student_loss_tracker.update_state(
            student_loss
        )

        self.distillation_loss_tracker.update_state(
            distillation_loss
        )

        self.accuracy.update_state(
            labels,
            student_logits,
            sample_weight=sample_weight
        )

        return {
            metric.name: metric.result()
            for metric in self.metrics
        }


# ============================================================
# CREATE DISTILLER
# ============================================================

distiller = Distiller(
    student=student,
    teacher=teacher_disease
)

distiller.compile(

    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    student_loss_fn=(
        keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        )
    ),

    distillation_loss_fn=(
        keras.losses.KLDivergence()
    ),

    alpha=ALPHA,
    temperature=TEMPERATURE
)

print("===== KD TRAINER =====")

print("Temperature:", TEMPERATURE)
print("Alpha:", ALPHA)

print(
    "Teacher trainable:",
    teacher_disease.trainable
)

print(
    "Student parameters:",
    f"{student.count_params():,}"
)

print("\nKD objective:")
print(
    "50% hard-label cross-entropy"
)
print(
    "50% temperature-softened teacher KL divergence"
)

print("\n✅ REAL KD TRAINER CREATED")
print("⚠️ Training has NOT started.")

===== KD TRAINER =====
Temperature: 3.0
Alpha: 0.5
Teacher trainable: False
Student parameters: 1,004,898

KD objective:
50% hard-label cross-entropy
50% temperature-softened teacher KL divergence

✅ REAL KD TRAINER CREATED
⚠️ Training has NOT started.


In [ ]:
import os
import pandas as pd
import numpy as np

BASE = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TRAIN_CSV = os.path.join(
    BASE,
    "fitzpatrick114_train.csv"
)

VAL_CSV = os.path.join(
    BASE,
    "fitzpatrick114_val.csv"
)

TEST_CSV = os.path.join(
    BASE,
    "fitzpatrick114_test.csv"
)

# ============================================================
# LOAD OFFICIAL SPLITS
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("===== SPLITS =====")
print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

print("\nTrain columns:")
print(train_df.columns.tolist())

# ============================================================
# BUILD SAME SORTED DISEASE MAPPING
# ============================================================

all_labels = sorted(
    train_df["label"]
    .dropna()
    .astype(str)
    .unique()
)

print("\n===== CLASS MAPPING =====")
print("Number of classes:", len(all_labels))

assert len(all_labels) == 114

class_to_index = {
    label: i
    for i, label in enumerate(all_labels)
}

index_to_class = {
    i: label
    for label, i in class_to_index.items()
}

# ============================================================
# VERIFY VAL/TEST LABELS
# ============================================================

train_labels = set(
    train_df["label"].astype(str)
)

val_labels = set(
    val_df["label"].astype(str)
)

test_labels = set(
    test_df["label"].astype(str)
)

print(
    "Validation labels missing from train:",
    len(val_labels - train_labels)
)

print(
    "Test labels missing from train:",
    len(test_labels - train_labels)
)

assert len(val_labels - train_labels) == 0
assert len(test_labels - train_labels) == 0

# ============================================================
# ADD INTEGER TARGETS
# ============================================================

train_df["disease_target"] = (
    train_df["label"]
    .astype(str)
    .map(class_to_index)
)

val_df["disease_target"] = (
    val_df["label"]
    .astype(str)
    .map(class_to_index)
)

test_df["disease_target"] = (
    test_df["label"]
    .astype(str)
    .map(class_to_index)
)

assert train_df["disease_target"].notna().all()
assert val_df["disease_target"].notna().all()
assert test_df["disease_target"].notna().all()

# ============================================================
# DISPLAY MAPPING
# ============================================================

print("\n===== FIRST 15 CLASSES =====")

for i in range(15):
    print(
        f"{i:3d} -> {index_to_class[i]}"
    )

print("\n===== LAST 10 CLASSES =====")

for i in range(104, 114):
    print(
        f"{i:3d} -> {index_to_class[i]}"
    )

# ============================================================
# TARGET RANGES
# ============================================================

print("\n===== TARGET CHECK =====")

print(
    "Train target range:",
    int(train_df["disease_target"].min()),
    "to",
    int(train_df["disease_target"].max())
)

print(
    "Val target range:",
    int(val_df["disease_target"].min()),
    "to",
    int(val_df["disease_target"].max())
)

print(
    "Test target range:",
    int(test_df["disease_target"].min()),
    "to",
    int(test_df["disease_target"].max())
)

print("\n✅ 114-CLASS SUPERVISED MAPPING READY")
print("⚠️ KD TRAINING HAS NOT STARTED")

===== SPLITS =====
Train: 7975
Val  : 1139
Test : 2280

Train columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name', 'image_path', 'exists']

===== CLASS MAPPING =====
Number of classes: 114
Validation labels missing from train: 0
Test labels missing from train: 0

===== FIRST 15 CLASSES =====
  0 -> acanthosis nigricans
  1 -> acne
  2 -> acne vulgaris
  3 -> acquired autoimmune bullous diseaseherpes gestationis
  4 -> acrodermatitis enteropathica
  5 -> actinic keratosis
  6 -> allergic contact dermatitis
  7 -> aplasia cutis
  8 -> basal cell carcinoma
  9 -> basal cell carcinoma morpheiform
 10 -> becker nevus
 11 -> behcets disease
 12 -> calcinosis cutis
 13 -> cheilitis
 14 -> congenital nevus

===== LAST 10 CLASSES =====
104 -> syringoma
105 -> telangiectases
106 -> tick bite
107 -> tuberous sclerosis
108 -> tungiasis
109 -> ur

In [ ]:
import os
import tensorflow as tf
import numpy as np

IMG_SIZE = 260
BATCH_SIZE = 16
SEED = 42

# ============================================================
# CHECK IMAGE PATHS
# ============================================================

print("===== IMAGE PATH CHECK =====")

train_exists = train_df["image_path"].apply(os.path.exists)
val_exists = val_df["image_path"].apply(os.path.exists)

print(
    "Train images available:",
    int(train_exists.sum()),
    "/",
    len(train_df)
)

print(
    "Val images available:",
    int(val_exists.sum()),
    "/",
    len(val_df)
)

assert train_exists.all()
assert val_exists.all()

# ============================================================
# IMAGE LOADER
# ============================================================

def load_supervised_image(path, label):

    image = tf.io.read_file(path)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape(
        [None, None, 3]
    )

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label

# ============================================================
# TRAIN AUGMENTATION
# ============================================================

def augment_supervised(image, label):

    image = tf.image.random_flip_left_right(
        image
    )

    image = tf.image.random_brightness(
        image,
        max_delta=0.08 * 255.0
    )

    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10
    )

    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    return image, label

# ============================================================
# TRAIN DATASET
# ============================================================

train_paths = train_df[
    "image_path"
].astype(str).values

train_targets = train_df[
    "disease_target"
].astype(np.int32).values

train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_paths,
        train_targets
    )
)

train_ds = train_ds.shuffle(
    len(train_paths),
    seed=SEED,
    reshuffle_each_iteration=True
)

train_ds = train_ds.map(
    load_supervised_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.map(
    augment_supervised,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.batch(
    BATCH_SIZE
)

train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

# ============================================================
# VALIDATION DATASET
# ============================================================

val_paths = val_df[
    "image_path"
].astype(str).values

val_targets = val_df[
    "disease_target"
].astype(np.int32).values

val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_paths,
        val_targets
    )
)

val_ds = val_ds.map(
    load_supervised_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.batch(
    BATCH_SIZE
)

val_ds = val_ds.prefetch(
    tf.data.AUTOTUNE
)

# ============================================================
# TEST ONE BATCH
# ============================================================

batch_images, batch_labels = next(
    iter(train_ds)
)

print("\n===== KD BATCH TEST =====")

print(
    "Images:",
    batch_images.shape
)

print(
    "Labels:",
    batch_labels.shape
)

print(
    "Image range:",
    float(tf.reduce_min(batch_images)),
    "to",
    float(tf.reduce_max(batch_images))
)

print(
    "Label range:",
    int(tf.reduce_min(batch_labels)),
    "to",
    int(tf.reduce_max(batch_labels))
)

print(
    "Label dtype:",
    batch_labels.dtype
)

assert batch_images.shape[1:] == (
    260,
    260,
    3
)

assert batch_labels.dtype == tf.int32

print("\n✅ KD DATA PIPELINE WORKS")
print("⚠️ FULL KD TRAINING HAS NOT STARTED")

===== IMAGE PATH CHECK =====
Train images available: 7975 / 7975
Val images available: 1139 / 1139

===== KD BATCH TEST =====
Images: (16, 260, 260, 3)
Labels: (16,)
Image range: 0.0 to 255.0
Label range: 6 to 108
Label dtype: <dtype: 'int32'>

✅ KD DATA PIPELINE WORKS
⚠️ FULL KD TRAINING HAS NOT STARTED


In [ ]:
import numpy as np
import tensorflow as tf

print("===== ONE-BATCH KD SANITY TEST =====")

# Get one batch
kd_images, kd_labels = next(iter(train_ds))

# ------------------------------------------------------------
# Save one student weight tensor BEFORE training
# ------------------------------------------------------------

before = student.trainable_variables[-1].numpy().copy()

# ------------------------------------------------------------
# ONE actual KD training step
# ------------------------------------------------------------

result = distiller.train_on_batch(
    kd_images,
    kd_labels,
    return_dict=True
)

# ------------------------------------------------------------
# Compare student weights AFTER training
# ------------------------------------------------------------

after = student.trainable_variables[-1].numpy().copy()

weight_change = float(
    np.mean(
        np.abs(after - before)
    )
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\n===== KD LOSSES =====")

for key, value in result.items():
    print(
        f"{key}: {float(value):.6f}"
    )

print("\n===== GRADIENT CHECK =====")

print(
    "Mean student weight change:",
    weight_change
)

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

for value in result.values():
    assert np.isfinite(float(value))

assert weight_change > 0

print("\n✅ HARD-LABEL LOSS WORKS")
print("✅ TEACHER DISTILLATION LOSS WORKS")
print("✅ STUDENT WEIGHTS UPDATED")
print("✅ ONE REAL KD OPTIMIZATION STEP COMPLETED")

print(
    "\n⚠️ This was ONLY one batch."
)
print(
    "The student is NOT a trained KD model yet."
)

===== ONE-BATCH KD SANITY TEST =====

===== KD LOSSES =====
loss: 5.105776
student_loss: 5.223235
distillation_loss: 4.988316
accuracy: 0.000000

===== GRADIENT CHECK =====
Mean student weight change: 9.991987462854013e-05

✅ HARD-LABEL LOSS WORKS
✅ TEACHER DISTILLATION LOSS WORKS
✅ STUDENT WEIGHTS UPDATED
✅ ONE REAL KD OPTIMIZATION STEP COMPLETED

⚠️ This was ONLY one batch.
The student is NOT a trained KD model yet.


In [ ]:
import os
import json

KD_DIR = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/KD"
)

STATUS_PATH = os.path.join(
    KD_DIR,
    "kd_sanity_test_results.json"
)

kd_status = {
    "teacher": "DermaWise_Hierarchical_B2",
    "teacher_parameters": 7929195,

    "student": "MobileNetV3Small",
    "student_parameters": 1004898,

    "parameter_reduction_percent": 87.33,

    "num_classes": 114,
    "input_size": [260, 260, 3],

    "temperature": 3.0,
    "alpha": 0.5,

    "hard_label_loss": "SparseCategoricalCrossentropy",
    "distillation_loss": "KLDivergence",

    "sanity_test_batches": 1,

    "sanity_test_total_loss": 5.105776,
    "sanity_test_student_loss": 5.223235,
    "sanity_test_distillation_loss": 4.988316,
    "sanity_test_accuracy": 0.0,

    "mean_weight_change": 9.991987462854013e-05,

    "status": (
        "KD pipeline sanity-tested successfully; "
        "full training pending GPU."
    )
}

with open(
    STATUS_PATH,
    "w"
) as f:
    json.dump(
        kd_status,
        f,
        indent=4
    )

print("===== KD STATUS SAVED =====")
print(STATUS_PATH)

print("\n✅ KD IMPLEMENTATION VERIFIED")
print("⏳ FULL KD TRAINING PENDING GPU")

===== KD STATUS SAVED =====
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/kd_sanity_test_results.json

✅ KD IMPLEMENTATION VERIFIED
⏳ FULL KD TRAINING PENDING GPU


In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

print("\n===== GPU CHECK =====")

if gpus:
    print("✅ GPU AVAILABLE")

    for gpu in gpus:
        print("GPU:", gpu)

    print("\nGPU details:")

    try:
        print(
            tf.config.experimental.get_device_details(
                gpus[0]
            )
        )
    except Exception as e:
        print("Could not read details:", e)

    print("\n🚀 READY FOR FULL SSL TRAINING")

else:
    print("❌ GPU NOT AVAILABLE")
    print(
        "Do NOT start SSL or KD full training."
    )

TensorFlow: 2.20.0

===== GPU CHECK =====
❌ GPU NOT AVAILABLE
Do NOT start SSL or KD full training.


In [ ]:
import time
import tensorflow as tf

print("===== KD CPU SPEED TEST =====")

# Reload CLEAN student so the previous one-batch test
# does not affect this benchmark.
STUDENT_PATH = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/KD/"
    "dermawise_kd_student_initial.keras"
)

clean_student = tf.keras.models.load_model(
    STUDENT_PATH,
    compile=False
)

print("✅ Clean student loaded")

# IMPORTANT:
# Recreate your Distiller using:
# teacher = existing teacher disease model
# student = clean_student
#
# Use the SAME:
# Temperature = 3.0
# Alpha = 0.5
# Adam learning rate = 1e-4

cpu_test_distiller = Distiller(
    student=clean_student,
    teacher=teacher_disease_model,
    temperature=3.0,
    alpha=0.5
)

cpu_test_distiller.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    )
)

# ------------------------------------------------------------
# Benchmark 20 batches
# ------------------------------------------------------------

BENCHMARK_BATCHES = 20

benchmark_ds = train_ds.take(
    BENCHMARK_BATCHES
)

start = time.time()

history = cpu_test_distiller.fit(
    benchmark_ds,
    epochs=1,
    verbose=1
)

elapsed = time.time() - start

seconds_per_batch = (
    elapsed / BENCHMARK_BATCHES
)

batches_per_epoch = (
    len(train_df) + BATCH_SIZE - 1
) // BATCH_SIZE

estimated_epoch_minutes = (
    seconds_per_batch *
    batches_per_epoch / 60
)

print("\n===== CPU BENCHMARK RESULT =====")

print(
    f"20 batches: {elapsed:.2f} seconds"
)

print(
    f"Seconds/batch: {seconds_per_batch:.2f}"
)

print(
    f"Batches/full epoch: {batches_per_epoch}"
)

print(
    f"Estimated full epoch: "
    f"{estimated_epoch_minutes:.1f} minutes"
)

print(
    "\n⚠️ This was only a speed test."
)

print(
    "Do NOT start full training yet."
)

===== KD CPU SPEED TEST =====
✅ Clean student loaded


NameError: name 'teacher_disease_model' is not defined

In [ ]:
import tensorflow as tf
import os

TEACHER_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/"
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

print("===== RELOADING KD TEACHER =====")

assert os.path.exists(TEACHER_PATH), \
    f"Teacher model not found: {TEACHER_PATH}"

# Load complete hierarchical model
teacher_full = tf.keras.models.load_model(
    TEACHER_PATH,
    compile=False
)

print("✅ Full teacher loaded")
print("Teacher name:", teacher_full.name)

print("\nTeacher outputs:")

for i, output in enumerate(teacher_full.outputs):
    print(
        i,
        output.name,
        output.shape
    )

print("\n⚠️ Do not run the CPU benchmark again yet.")

===== RELOADING KD TEACHER =====
✅ Full teacher loaded
Teacher name: DermaWise_Hierarchical_B2

Teacher outputs:
0 keras_tensor_3690 (None, 114)
1 keras_tensor_3692 (None, 9)
2 keras_tensor_3694 (None, 3)

⚠️ Do not run the CPU benchmark again yet.


In [ ]:
import tensorflow as tf

print("===== EXTRACTING KD DISEASE TEACHER =====")

teacher_disease_model = tf.keras.Model(
    inputs=teacher_full.input,
    outputs=teacher_full.outputs[0],
    name="KD_Disease_Teacher"
)

teacher_disease_model.trainable = False

print("✅ 114-class teacher extracted")

print(
    "Input shape:",
    teacher_disease_model.input_shape
)

print(
    "Output shape:",
    teacher_disease_model.output_shape
)

print(
    "Parameters:",
    teacher_disease_model.count_params()
)

print(
    "Trainable:",
    teacher_disease_model.trainable
)

# Critical checks
assert teacher_disease_model.output_shape[-1] == 114
assert teacher_disease_model.trainable is False

print("\n✅ KD TEACHER READY")

===== EXTRACTING KD DISEASE TEACHER =====
✅ 114-class teacher extracted
Input shape: (None, 260, 260, 3)
Output shape: (None, 114)
Parameters: 7929195
Trainable: False

✅ KD TEACHER READY


In [ ]:
print("===== KD COMPONENT CHECK =====")

checks = {
    "Distiller class": "Distiller" in globals(),
    "train_ds": "train_ds" in globals(),
    "train_df": "train_df" in globals(),
    "clean_student": "clean_student" in globals(),
    "teacher_disease_model": "teacher_disease_model" in globals(),
}

for name, status in checks.items():
    print(
        f"{'✅' if status else '❌'} {name}"
    )

if all(checks.values()):
    print("\n✅ ALL COMPONENTS READY FOR CPU BENCHMARK")
else:
    print("\n⚠️ SOME COMPONENTS NEED TO BE RESTORED")

===== KD COMPONENT CHECK =====
✅ Distiller class
✅ train_ds
✅ train_df
✅ clean_student
✅ teacher_disease_model

✅ ALL COMPONENTS READY FOR CPU BENCHMARK


In [ ]:
import time
import tensorflow as tf

print("===== KD CPU SPEED BENCHMARK =====")

STUDENT_PATH = (
    "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/"
    "dermawise_kd_student_initial.keras"
)

# Reload CLEAN student
benchmark_student = tf.keras.models.load_model(
    STUDENT_PATH,
    compile=False
)

print("✅ Clean benchmark student loaded")

# ------------------------------------------------------------
# Create Distiller using the SAME structure as our working test
# ------------------------------------------------------------

benchmark_distiller = Distiller(
    student=benchmark_student,
    teacher=teacher_disease_model
)

# Set KD configuration
benchmark_distiller.temperature = 3.0
benchmark_distiller.alpha = 0.5

benchmark_distiller.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    )
)

print("Temperature:", benchmark_distiller.temperature)
print("Alpha:", benchmark_distiller.alpha)

# ------------------------------------------------------------
# Benchmark only 20 batches
# ------------------------------------------------------------

BENCHMARK_BATCHES = 20

benchmark_ds = train_ds.take(
    BENCHMARK_BATCHES
)

print("\nBenchmark batches:", BENCHMARK_BATCHES)
print("Starting CPU benchmark...\n")

start_time = time.time()

benchmark_distiller.fit(
    benchmark_ds,
    epochs=1,
    verbose=1
)

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Estimate full training time
# ------------------------------------------------------------

seconds_per_batch = elapsed / BENCHMARK_BATCHES

batches_per_epoch = (
    len(train_df) + BATCH_SIZE - 1
) // BATCH_SIZE

estimated_epoch_minutes = (
    seconds_per_batch *
    batches_per_epoch / 60
)

estimated_10_epochs_hours = (
    seconds_per_batch *
    batches_per_epoch *
    10 / 3600
)

print("\n===== CPU BENCHMARK RESULT =====")

print(
    f"20 batches time: {elapsed:.2f} seconds"
)

print(
    f"Seconds per batch: {seconds_per_batch:.2f}"
)

print(
    f"Batches per full epoch: {batches_per_epoch}"
)

print(
    f"Estimated full epoch: "
    f"{estimated_epoch_minutes:.1f} minutes"
)

print(
    f"Estimated 10 epochs: "
    f"{estimated_10_epochs_hours:.1f} hours"
)

print(
    "\n⚠️ Benchmark only — "
    "full KD training has NOT started."
)

===== KD CPU SPEED BENCHMARK =====
✅ Clean benchmark student loaded


TypeError: Distiller.compile() missing 2 required positional arguments: 'student_loss_fn' and 'distillation_loss_fn'

In [ ]:
benchmark_distiller.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    student_loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    distillation_loss_fn=tf.keras.losses.KLDivergence()
)

In [ ]:
benchmark_distiller = Distiller(
    student=benchmark_student,
    teacher=teacher_disease_model
)

benchmark_distiller.temperature = 3.0
benchmark_distiller.alpha = 0.5

benchmark_distiller.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    student_loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    distillation_loss_fn=tf.keras.losses.KLDivergence()
)

print("Temperature:", benchmark_distiller.temperature)
print("Alpha:", benchmark_distiller.alpha)

BENCHMARK_BATCHES = 20
benchmark_ds = train_ds.take(BENCHMARK_BATCHES)

print("\nStarting 20-batch CPU benchmark...")

import time
start_time = time.time()

benchmark_distiller.fit(
    benchmark_ds,
    epochs=1,
    verbose=1
)

elapsed = time.time() - start_time

seconds_per_batch = elapsed / BENCHMARK_BATCHES
batches_per_epoch = (
    len(train_df) + BATCH_SIZE - 1
) // BATCH_SIZE

estimated_epoch_minutes = (
    seconds_per_batch *
    batches_per_epoch / 60
)

estimated_10_epochs_hours = (
    seconds_per_batch *
    batches_per_epoch *
    10 / 3600
)

print("\n===== CPU BENCHMARK RESULT =====")
print(f"20 batches time: {elapsed:.2f} seconds")
print(f"Seconds per batch: {seconds_per_batch:.2f}")
print(f"Batches per full epoch: {batches_per_epoch}")
print(f"Estimated full epoch: {estimated_epoch_minutes:.1f} minutes")
print(f"Estimated 10 epochs: {estimated_10_epochs_hours:.1f} hours")

print("\n⚠️ Benchmark only — full KD training has NOT started.")

Temperature: 3.0
Alpha: 0.5

Starting 20-batch CPU benchmark...
20/20 ━━━━━━━━━━━━━━━━━━━━ 136s 4s/step - accuracy: 0.0188 - distillation_loss: 4.6480 - loss: 4.7845 - student_loss: 4.9210

===== CPU BENCHMARK RESULT =====
20 batches time: 141.04 seconds
Seconds per batch: 7.05
Batches per full epoch: 499
Estimated full epoch: 58.6 minutes
Estimated 10 epochs: 9.8 hours

⚠️ Benchmark only — full KD training has NOT started.


In [ ]:
import os
import tensorflow as tf
import time

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

# Always start from the CLEAN student
STUDENT_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_initial.keras"
)

kd_student = tf.keras.models.load_model(
    STUDENT_PATH,
    compile=False
)

print("✅ Clean KD student loaded")

# Create real distiller
kd_model = Distiller(
    student=kd_student,
    teacher=teacher_disease_model
)

kd_model.temperature = 3.0
kd_model.alpha = 0.5

kd_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    student_loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    distillation_loss_fn=tf.keras.losses.KLDivergence()
)

print("Temperature:", kd_model.temperature)
print("Alpha:", kd_model.alpha)
print("Teacher trainable:", teacher_disease_model.trainable)

print("\n===== STARTING KD EPOCH 1 =====")
print("Expected CPU time: roughly 1 hour")
print("Do not close Colab while this is running.\n")

start = time.time()

history = kd_model.fit(
    train_ds,
    epochs=1,
    verbose=1
)

elapsed = time.time() - start

# Save STUDENT, not the Distiller wrapper
EPOCH1_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch01.keras"
)

kd_student.save(EPOCH1_PATH)

print("\n===== EPOCH 1 COMPLETE =====")
print(
    f"Training time: {elapsed / 60:.1f} minutes"
)

print(
    "Saved:",
    EPOCH1_PATH
)

print("\n✅ KD EPOCH 1 SAVED TO GOOGLE DRIVE")

✅ Clean KD student loaded
Temperature: 3.0
Alpha: 0.5
Teacher trainable: False

===== STARTING KD EPOCH 1 =====
Expected CPU time: roughly 1 hour
Do not close Colab while this is running.

499/499 ━━━━━━━━━━━━━━━━━━━━ 1795s 3s/step - accuracy: 0.0544 - distillation_loss: 3.8870 - loss: 4.1927 - student_loss: 4.4985

===== EPOCH 1 COMPLETE =====
Training time: 30.2 minutes
Saved: /content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_epoch01.keras

✅ KD EPOCH 1 SAVED TO GOOGLE DRIVE


In [ ]:
print("KD student available:", "kd_student" in globals())
print("Validation dataset available:", "val_ds" in globals())
print("Teacher available:", "teacher_disease_model" in globals())

KD student available: False
Validation dataset available: False
Teacher available: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import tensorflow as tf
import pandas as pd
import numpy as np

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"
EPOCH1_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch01.keras"
)

print("TensorFlow:", tf.__version__)
print("Epoch 1 model exists:", os.path.exists(EPOCH1_PATH))

if os.path.exists(EPOCH1_PATH):
    print("Size MB:", round(os.path.getsize(EPOCH1_PATH) / (1024**2), 2))
    print("✅ KD Epoch 1 checkpoint is safe.")
else:
    print("❌ Epoch 1 checkpoint not found.")

Mounted at /content/drive
TensorFlow: 2.20.0
Epoch 1 model exists: True
Size MB: 4.37
✅ KD Epoch 1 checkpoint is safe.


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

EPOCH1_PATH = os.path.join(
    WORK_DIR,
    "KD",
    "dermawise_kd_student_epoch01.keras"
)

VAL_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_val.csv"
)

# -----------------------------
# 1. Load trained KD student
# -----------------------------
kd_student = tf.keras.models.load_model(
    EPOCH1_PATH,
    compile=False
)

print("✅ KD Epoch 1 student loaded")
print("Input shape :", kd_student.input_shape)
print("Output shape:", kd_student.output_shape)

# -----------------------------
# 2. Load validation manifest
# -----------------------------
val_df = pd.read_csv(VAL_PATH)

# Restore exact 114-class mapping from TRAIN manifest
train_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
)

CLASS_NAMES = sorted(train_df["label"].unique().tolist())
class_to_index = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

val_df["class_id"] = val_df["label"].map(class_to_index)

assert len(CLASS_NAMES) == 114
assert val_df["class_id"].notna().all()

print("\nClasses:", len(CLASS_NAMES))
print("Validation images:", len(val_df))

# -----------------------------
# 3. Build validation dataset
# -----------------------------
IMG_SIZE = 260
BATCH_SIZE = 16

def load_val_image(filepath, label):
    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    # MobileNetV3 includes preprocessing internally.
    # Keep images in 0-255 float range.
    image = tf.cast(image, tf.float32)

    return image, label


val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].astype(str).values,
        val_df["class_id"].astype("int32").values
    )
)

val_ds = (
    val_ds
    .map(load_val_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# -----------------------------
# 4. Verify one batch
# -----------------------------
images, labels = next(iter(val_ds))

print("\n===== RESTORATION CHECK =====")
print("Image batch:", images.shape)
print("Labels     :", labels.shape)
print("Pixel range:",
      float(tf.reduce_min(images)),
      "to",
      float(tf.reduce_max(images)))

print("\n✅ READY FOR KD EPOCH 1 VALIDATION")

✅ KD Epoch 1 student loaded
Input shape : (None, 260, 260, 3)
Output shape: (None, 114)

Classes: 114
Validation images: 1139

===== RESTORATION CHECK =====
Image batch: (16, 260, 260, 3)
Labels     : (16,)
Pixel range: 0.0 to 255.0

✅ READY FOR KD EPOCH 1 VALIDATION


In [ ]:
import numpy as np
import tensorflow as tf
import time

print("===== EVALUATING KD EPOCH 1 =====")

start = time.time()

all_logits = []
all_labels = []

for images, labels in val_ds:
    logits = kd_student(images, training=False)

    all_logits.append(logits.numpy())
    all_labels.append(labels.numpy())

logits = np.concatenate(all_logits, axis=0)
y_true = np.concatenate(all_labels, axis=0)

# Convert logits to probabilities
probs = tf.nn.softmax(logits, axis=1).numpy()

# Top-1
top1_pred = np.argmax(probs, axis=1)
top1 = np.mean(top1_pred == y_true)

# Top-3
top3_indices = np.argsort(probs, axis=1)[:, -3:]
top3 = np.mean([
    y_true[i] in top3_indices[i]
    for i in range(len(y_true))
])

# Top-5
top5_indices = np.argsort(probs, axis=1)[:, -5:]
top5 = np.mean([
    y_true[i] in top5_indices[i]
    for i in range(len(y_true))
])

elapsed = time.time() - start

print("\n===== KD EPOCH 1 VALIDATION RESULTS =====")
print(f"Images : {len(y_true)}")
print(f"Top-1  : {top1 * 100:.2f}%")
print(f"Top-3  : {top3 * 100:.2f}%")
print(f"Top-5  : {top5 * 100:.2f}%")
print(f"Time   : {elapsed / 60:.2f} minutes")

===== EVALUATING KD EPOCH 1 =====

===== KD EPOCH 1 VALIDATION RESULTS =====
Images : 1139
Top-1  : 6.94%
Top-3  : 15.10%
Top-5  : 22.04%
Time   : 5.97 minutes


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

print("GPUs:", gpus)

if gpus:
    print("✅ GPU AVAILABLE — we can switch/use GPU")
else:
    print("❌ GPU NOT AVAILABLE")

GPUs: []
❌ GPU NOT AVAILABLE


In [ ]:
import os

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

TEACHER_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

EPOCH1_PATH = os.path.join(
    WORK_DIR,
    "KD",
    "dermawise_kd_student_epoch01.keras"
)

print("Teacher exists:", os.path.exists(TEACHER_PATH))
print("Epoch 1 student exists:", os.path.exists(EPOCH1_PATH))

Teacher exists: True
Epoch 1 student exists: True


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
KD_DIR = os.path.join(WORK_DIR, "KD")

TEACHER_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

EPOCH1_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch01.keras"
)

TRAIN_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

# ============================================================
# 1. LOAD TEACHER
# ============================================================

print("Loading hierarchical teacher...")

teacher_full = tf.keras.models.load_model(
    TEACHER_PATH,
    compile=False
)

# The saved hierarchical model outputs:
# output 0 = disease (114)
# output 1 = nine-way
# output 2 = three-way

teacher_disease_model = tf.keras.Model(
    inputs=teacher_full.input,
    outputs=teacher_full.outputs[0],
    name="Frozen_Disease_Teacher"
)

teacher_disease_model.trainable = False

print("✅ Teacher loaded")
print("Teacher output:", teacher_disease_model.output_shape)
print("Teacher trainable:", teacher_disease_model.trainable)

assert teacher_disease_model.output_shape[-1] == 114


# ============================================================
# 2. LOAD EPOCH-1 STUDENT
# ============================================================

print("\nLoading KD Epoch-1 student...")

kd_student = tf.keras.models.load_model(
    EPOCH1_PATH,
    compile=False
)

print("✅ Epoch-1 student loaded")
print("Student input :", kd_student.input_shape)
print("Student output:", kd_student.output_shape)

assert kd_student.output_shape[-1] == 114


# ============================================================
# 3. RESTORE TRAINING MANIFEST + CLASS MAPPING
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)

CLASS_NAMES = sorted(
    train_df["label"].unique().tolist()
)

class_to_index = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

train_df["class_id"] = (
    train_df["label"]
    .map(class_to_index)
    .astype("int32")
)

assert len(CLASS_NAMES) == 114
assert train_df["class_id"].notna().all()

print("\nTraining images:", len(train_df))
print("Classes:", len(CLASS_NAMES))


# ============================================================
# 4. RESTORE TRAINING DATASET
# ============================================================

IMG_SIZE = 260
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE


def load_train_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def augment_train(image, label):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image,
        max_delta=0.08 * 255.0
    )

    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10
    )

    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    return image, label


train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].astype(str).values,
        train_df["class_id"].values
    )
)

train_ds = (
    train_ds
    .shuffle(
        len(train_df),
        seed=42,
        reshuffle_each_iteration=True
    )
    .map(
        load_train_image,
        num_parallel_calls=AUTOTUNE
    )
    .map(
        augment_train,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


# ============================================================
# 5. VERIFY
# ============================================================

images, labels = next(iter(train_ds))

print("\n===== KD RESTORATION CHECK =====")

print("Batch images :", images.shape)
print("Batch labels :", labels.shape)

print(
    "Pixel range :",
    float(tf.reduce_min(images)),
    "to",
    float(tf.reduce_max(images))
)

print(
    "Training batches:",
    tf.data.experimental.cardinality(train_ds).numpy()
)

print("Teacher frozen:", not teacher_disease_model.trainable)

print("\n✅ READY TO REBUILD DISTILLER")

Loading hierarchical teacher...
✅ Teacher loaded
Teacher output: (None, 114)
Teacher trainable: False

Loading KD Epoch-1 student...
✅ Epoch-1 student loaded
Student input : (None, 260, 260, 3)
Student output: (None, 114)

Training images: 7975
Classes: 114

===== KD RESTORATION CHECK =====
Batch images : (16, 260, 260, 3)
Batch labels : (16,)
Pixel range : 0.0 to 255.0
Training batches: 499
Teacher frozen: True

✅ READY TO REBUILD DISTILLER


In [ ]:
import os
import tensorflow as tf

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

EPOCH2_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch02.keras"
)

# ============================================================
# DISTILLER
# ============================================================

class Distiller(tf.keras.Model):

    def __init__(self, student, teacher):
        super().__init__()
        self.student = student
        self.teacher = teacher

        self.temperature = 3.0
        self.alpha = 0.5

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.student_loss_tracker = tf.keras.metrics.Mean(
            name="student_loss"
        )
        self.distillation_loss_tracker = tf.keras.metrics.Mean(
            name="distillation_loss"
        )
        self.accuracy_tracker = (
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        )

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.student_loss_tracker,
            self.distillation_loss_tracker,
            self.accuracy_tracker
        ]

    def compile(
        self,
        optimizer,
        student_loss_fn,
        distillation_loss_fn
    ):
        super().compile()
        self.optimizer = optimizer
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn

    def train_step(self, data):

        x, y = data

        # Teacher is frozen
        teacher_probs = self.teacher(
            x,
            training=False
        )

        # Convert teacher probabilities into stable logits
        teacher_logits = tf.math.log(
            tf.clip_by_value(
                teacher_probs,
                1e-7,
                1.0
            )
        )

        with tf.GradientTape() as tape:

            # Student outputs raw logits
            student_logits = self.student(
                x,
                training=True
            )

            # Normal supervised loss
            student_loss = self.student_loss_fn(
                y,
                student_logits
            )

            # Softened teacher distribution
            teacher_soft = tf.nn.softmax(
                teacher_logits / self.temperature,
                axis=1
            )

            # Softened student distribution
            student_soft = tf.nn.softmax(
                student_logits / self.temperature,
                axis=1
            )

            distillation_loss = (
                self.distillation_loss_fn(
                    teacher_soft,
                    student_soft
                )
                * (self.temperature ** 2)
            )

            loss = (
                self.alpha * student_loss
                +
                (1.0 - self.alpha)
                * distillation_loss
            )

        gradients = tape.gradient(
            loss,
            self.student.trainable_variables
        )

        self.optimizer.apply_gradients(
            zip(
                gradients,
                self.student.trainable_variables
            )
        )

        self.loss_tracker.update_state(loss)
        self.student_loss_tracker.update_state(student_loss)
        self.distillation_loss_tracker.update_state(
            distillation_loss
        )

        self.accuracy_tracker.update_state(
            y,
            student_logits
        )

        return {
            "loss": self.loss_tracker.result(),
            "student_loss":
                self.student_loss_tracker.result(),
            "distillation_loss":
                self.distillation_loss_tracker.result(),
            "accuracy":
                self.accuracy_tracker.result()
        }


# ============================================================
# BUILD FROM EPOCH-1 STUDENT
# ============================================================

kd_model = Distiller(
    student=kd_student,
    teacher=teacher_disease_model
)

kd_model.temperature = 3.0
kd_model.alpha = 0.5

kd_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    student_loss_fn=
        tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),
    distillation_loss_fn=
        tf.keras.losses.KLDivergence()
)

print("===== KD EPOCH 2 SETUP =====")
print("Starting from: Epoch 1 student")
print("Temperature:", kd_model.temperature)
print("Alpha:", kd_model.alpha)
print("Teacher trainable:",
      teacher_disease_model.trainable)

# ============================================================
# TRAIN ONE MORE EPOCH
# ============================================================

print("\n===== STARTING KD EPOCH 2 =====")

history_epoch2 = kd_model.fit(
    train_ds,
    epochs=1,
    verbose=1
)

# ============================================================
# SAVE IMMEDIATELY
# ============================================================

kd_student.save(EPOCH2_PATH)

print("\n===== EPOCH 2 COMPLETE =====")
print("Saved:", EPOCH2_PATH)
print("Exists:", os.path.exists(EPOCH2_PATH))
print(
    "Size MB:",
    round(
        os.path.getsize(EPOCH2_PATH) / (1024**2),
        2
    )
)

print("\n✅ KD EPOCH 2 SAVED TO GOOGLE DRIVE")

===== KD EPOCH 2 SETUP =====
Starting from: Epoch 1 student
Temperature: 3.0
Alpha: 0.5
Teacher trainable: False

===== STARTING KD EPOCH 2 =====
499/499 ━━━━━━━━━━━━━━━━━━━━ 2571s 5s/step - accuracy: 0.1149 - distillation_loss: 3.0941 - loss: 3.5414 - student_loss: 3.9888

===== EPOCH 2 COMPLETE =====
Saved: /content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_epoch02.keras
Exists: True
Size MB: 4.37

✅ KD EPOCH 2 SAVED TO GOOGLE DRIVE


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

search_root = "/content/drive/MyDrive/DermaWise"

print("Searching for Epoch 2 checkpoint...\n")

matches = []

for root, dirs, files in os.walk(search_root):
    for file in files:
        name = file.lower()

        if (
            "epoch02" in name
            or "epoch2" in name
            or "epoch_02" in name
            or "epoch_2" in name
        ):
            matches.append(
                os.path.join(root, file)
            )

if matches:
    print("✅ FOUND:")
    for path in matches:
        print(path)
else:
    print("❌ No Epoch 2 file found.")

Mounted at /content/drive
Searching for Epoch 2 checkpoint...

✅ FOUND:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_epoch02.keras


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import tensorflow as tf

print("\n===== HARDWARE =====")
print("GPU:", tf.config.list_physical_devices("GPU"))

assert tf.config.list_physical_devices("GPU"), \
    "GPU disappeared — STOP!"

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

print("\n===== KD CHECKPOINTS =====")

for f in sorted(os.listdir(KD_DIR)):
    if f.endswith(".keras"):
        print(f)

print("\n✅ T4 GPU READY")

Mounted at /content/drive

===== HARDWARE =====
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

===== KD CHECKPOINTS =====
dermawise_kd_student_epoch01.keras
dermawise_kd_student_epoch02.keras
dermawise_kd_student_initial.keras

✅ T4 GPU READY


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
KD_DIR = os.path.join(WORK_DIR, "KD")

EPOCH2_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch02.keras"
)

VAL_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_val.csv"
)

TRAIN_PATH = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

assert os.path.exists(EPOCH2_PATH), "❌ Epoch 2 model not found"
assert os.path.exists(VAL_PATH), "❌ Validation CSV not found"

# ============================================================
# CLASS MAPPING
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

CLASS_NAMES = sorted(train_df["label"].unique().tolist())

class_to_index = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

val_df["class_id"] = (
    val_df["label"]
    .map(class_to_index)
    .astype("int32")
)

assert len(CLASS_NAMES) == 114
assert val_df["class_id"].notna().all()

print("Validation images:", len(val_df))
print("Classes:", len(CLASS_NAMES))

# ============================================================
# VALIDATION DATASET
# ============================================================

IMG_SIZE = 260
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_val_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].astype(str).values,
        val_df["class_id"].values
    )
)

val_ds = (
    val_ds
    .map(load_val_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ============================================================
# LOAD EPOCH 2
# ============================================================

print("\nLoading Epoch-2 student...")

student_epoch2 = tf.keras.models.load_model(
    EPOCH2_PATH,
    compile=False
)

print("✅ Epoch-2 student loaded")

# ============================================================
# VALIDATE
# ============================================================

print("\n===== EVALUATING KD EPOCH 2 =====")

top1_correct = 0
top3_correct = 0
top5_correct = 0
total = 0

start = time.time()

for images, labels in val_ds:

    logits = student_epoch2(
        images,
        training=False
    )

    probs = tf.nn.softmax(logits, axis=1)

    labels_np = labels.numpy()

    pred1 = tf.argmax(
        probs,
        axis=1,
        output_type=tf.int32
    ).numpy()

    pred3 = tf.math.top_k(
        probs,
        k=3
    ).indices.numpy()

    pred5 = tf.math.top_k(
        probs,
        k=5
    ).indices.numpy()

    top1_correct += np.sum(pred1 == labels_np)

    top3_correct += sum(
        labels_np[i] in pred3[i]
        for i in range(len(labels_np))
    )

    top5_correct += sum(
        labels_np[i] in pred5[i]
        for i in range(len(labels_np))
    )

    total += len(labels_np)

elapsed = time.time() - start

top1 = 100 * top1_correct / total
top3 = 100 * top3_correct / total
top5 = 100 * top5_correct / total

print("\n===== KD EPOCH 2 VALIDATION RESULTS =====")
print("Images :", total)
print(f"Top-1  : {top1:.2f}%")
print(f"Top-3  : {top3:.2f}%")
print(f"Top-5  : {top5:.2f}%")
print(f"Time   : {elapsed:.2f} seconds")

Validation images: 1139
Classes: 114

Loading Epoch-2 student...
✅ Epoch-2 student loaded

===== EVALUATING KD EPOCH 2 =====

===== KD EPOCH 2 VALIDATION RESULTS =====
Images : 1139
Top-1  : 10.27%
Top-3  : 21.42%
Top-5  : 29.50%
Time   : 227.63 seconds


In [ ]:
import os
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

train_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
)

val_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_val.csv")
)

print("Train:", len(train_df))
print("Val  :", len(val_df))

print("\nExample train path:")
print(train_df["image_path"].iloc[0])

print("\nExample val path:")
print(val_df["image_path"].iloc[0])

print("\nTrain first image exists:",
      os.path.exists(train_df["image_path"].iloc[0]))

print("Val first image exists:",
      os.path.exists(val_df["image_path"].iloc[0]))

Train: 7975
Val  : 1139

Example train path:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C/Categorized_AbbrvName/ne-de_361/ne-de_f2_61_3a96728f.jpg

Example val path:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C/Categorized_AbbrvName/pi-ru-pi_278/pi-ru-pi_f5_215_86507785.jpg

Train first image exists: True
Val first image exists: True


In [ ]:
import os
import shutil
import time
import pandas as pd

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

LOCAL_ROOT = "/content/dermawise_kd"
LOCAL_TRAIN = os.path.join(LOCAL_ROOT, "train")
LOCAL_VAL = os.path.join(LOCAL_ROOT, "val")

os.makedirs(LOCAL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_VAL, exist_ok=True)

train_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_train.csv")
)

val_df = pd.read_csv(
    os.path.join(WORK_DIR, "fitzpatrick114_val.csv")
)

# ------------------------------------------------------------
# COPY FUNCTION
# ------------------------------------------------------------

def copy_images(df, destination, name):

    start = time.time()
    copied = 0
    missing = 0
    local_paths = []

    total = len(df)

    print(f"\nCopying {name}: {total} images...")

    for i, src in enumerate(df["image_path"].astype(str)):

        # Preserve unique original filename.
        filename = os.path.basename(src)

        dst = os.path.join(
            destination,
            filename
        )

        if os.path.exists(src):

            if not os.path.exists(dst):
                shutil.copy2(src, dst)

            copied += 1
            local_paths.append(dst)

        else:
            missing += 1
            local_paths.append(None)

        if (i + 1) % 1000 == 0:
            print(
                f"{name}: {i+1}/{total}"
            )

    elapsed = time.time() - start

    print(f"\n{name} finished")
    print("Available:", copied)
    print("Missing  :", missing)
    print(
        "Time     :",
        round(elapsed / 60, 2),
        "minutes"
    )

    return local_paths


# ------------------------------------------------------------
# COPY TRAIN
# ------------------------------------------------------------

train_df["local_path"] = copy_images(
    train_df,
    LOCAL_TRAIN,
    "TRAIN"
)

# ------------------------------------------------------------
# COPY VALIDATION
# ------------------------------------------------------------

val_df["local_path"] = copy_images(
    val_df,
    LOCAL_VAL,
    "VALIDATION"
)

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

train_valid = train_df["local_path"].apply(
    lambda x: isinstance(x, str) and os.path.exists(x)
).sum()

val_valid = val_df["local_path"].apply(
    lambda x: isinstance(x, str) and os.path.exists(x)
).sum()

print("\n===== LOCAL DATASET CHECK =====")
print("Train:", train_valid, "/", len(train_df))
print("Val  :", val_valid, "/", len(val_df))

assert train_valid == 7975, \
    "❌ Some training images are missing"

assert val_valid == 1139, \
    "❌ Some validation images are missing"

print("\n✅ ALL KD IMAGES ARE ON LOCAL STORAGE")


Copying TRAIN: 7975 images...
TRAIN: 1000/7975
TRAIN: 2000/7975
TRAIN: 3000/7975
TRAIN: 4000/7975
TRAIN: 5000/7975
TRAIN: 6000/7975
TRAIN: 7000/7975

TRAIN finished
Available: 7975
Missing  : 0
Time     : 49.84 minutes

Copying VALIDATION: 1139 images...
VALIDATION: 1000/1139

VALIDATION finished
Available: 1139
Missing  : 0
Time     : 0.06 minutes

===== LOCAL DATASET CHECK =====
Train: 7975 / 7975
Val  : 1139 / 1139

✅ ALL KD IMAGES ARE ON LOCAL STORAGE


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")

print("GPU:", gpus)

if gpus:
    print("✅ GPU AVAILABLE")
else:
    print("❌ GPU NOT AVAILABLE")

Mounted at /content/drive
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU AVAILABLE


In [ ]:
import os
import pandas as pd
import tensorflow as tf

# ============================================================
# PATHS
# ============================================================

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
KD_DIR = os.path.join(WORK_DIR, "KD")

TEACHER_PATH = os.path.join(
    WORK_DIR,
    "dermawise_fitzpatrick114_hierarchical_b2_finetuned_best.keras"
)

EPOCH2_PATH = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch02.keras"
)

TRAIN_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

VAL_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_val.csv"
)

# ============================================================
# SAFETY CHECK
# ============================================================

assert tf.config.list_physical_devices("GPU"), \
    "❌ GPU NOT AVAILABLE — STOP"

assert os.path.exists(TEACHER_PATH), \
    "❌ Teacher missing"

assert os.path.exists(EPOCH2_PATH), \
    "❌ Epoch-2 student missing"

print("✅ GPU available")
print("✅ Teacher found")
print("✅ Epoch-2 student found")

# ============================================================
# LOAD MODELS
# ============================================================

print("\nLoading teacher...")

teacher_full = tf.keras.models.load_model(
    TEACHER_PATH,
    compile=False
)

teacher_disease_model = tf.keras.Model(
    inputs=teacher_full.input,
    outputs=teacher_full.outputs[0],
    name="Frozen_Disease_Teacher"
)

teacher_disease_model.trainable = False

print("✅ Teacher loaded:", teacher_disease_model.output_shape)

print("\nLoading Epoch-2 student...")

kd_student = tf.keras.models.load_model(
    EPOCH2_PATH,
    compile=False
)

print("✅ Student loaded:", kd_student.output_shape)

# ============================================================
# DATA
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

CLASS_NAMES = sorted(
    train_df["label"].unique().tolist()
)

class_to_index = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

train_df["class_id"] = (
    train_df["label"]
    .map(class_to_index)
    .astype("int32")
)

val_df["class_id"] = (
    val_df["label"]
    .map(class_to_index)
    .astype("int32")
)

assert len(CLASS_NAMES) == 114
assert train_df["class_id"].notna().all()
assert val_df["class_id"].notna().all()

print("\nTrain:", len(train_df))
print("Val  :", len(val_df))
print("Classes:", len(CLASS_NAMES))

# ============================================================
# TF.DATA
# ============================================================

IMG_SIZE = 260
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(image, tf.float32)

    return image, label


def augment_image(image, label):

    image = tf.image.random_flip_left_right(image)

    image = tf.image.random_brightness(
        image,
        max_delta=0.08 * 255.0
    )

    image = tf.image.random_contrast(
        image,
        lower=0.90,
        upper=1.10
    )

    image = tf.clip_by_value(
        image,
        0.0,
        255.0
    )

    return image, label


train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].astype(str).values,
        train_df["class_id"].values
    )
)

train_ds = (
    train_ds
    .shuffle(
        len(train_df),
        seed=42,
        reshuffle_each_iteration=True
    )
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .map(augment_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].astype(str).values,
        val_df["class_id"].values
    )
)

val_ds = (
    val_ds
    .map(load_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# ============================================================
# FINAL CHECK
# ============================================================

print("\n===== GPU KD RESTORATION =====")
print("Train batches:",
      tf.data.experimental.cardinality(train_ds).numpy())

print("Val batches:",
      tf.data.experimental.cardinality(val_ds).numpy())

print("Teacher frozen:",
      not teacher_disease_model.trainable)

print("Student starts from: EPOCH 2")

print("\n✅ READY FOR MULTI-EPOCH GPU KD")

✅ GPU available
✅ Teacher found
✅ Epoch-2 student found

Loading teacher...
✅ Teacher loaded: (None, 114)

Loading Epoch-2 student...
✅ Student loaded: (None, 114)

Train: 7975
Val  : 1139
Classes: 114

===== GPU KD RESTORATION =====
Train batches: 499
Val batches: 72
Teacher frozen: True
Student starts from: EPOCH 2

✅ READY FOR MULTI-EPOCH GPU KD


In [ ]:
import os
import tensorflow as tf

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"
os.makedirs(KD_DIR, exist_ok=True)

# ============================================================
# DISTILLER
# ============================================================

class Distiller(tf.keras.Model):

    def __init__(self, student, teacher):
        super().__init__()

        self.student = student
        self.teacher = teacher

        self.temperature = 3.0
        self.alpha = 0.5

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")

        self.student_loss_tracker = tf.keras.metrics.Mean(
            name="student_loss"
        )

        self.distillation_loss_tracker = tf.keras.metrics.Mean(
            name="distillation_loss"
        )

        self.accuracy_tracker = (
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        )

    @property
    def metrics(self):

        return [
            self.loss_tracker,
            self.student_loss_tracker,
            self.distillation_loss_tracker,
            self.accuracy_tracker
        ]

    def compile(
        self,
        optimizer,
        student_loss_fn,
        distillation_loss_fn
    ):

        super().compile()

        self.optimizer = optimizer
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn


    def train_step(self, data):

        x, y = data

        # Frozen teacher
        teacher_probs = self.teacher(
            x,
            training=False
        )

        # Teacher currently outputs softmax probabilities.
        # Convert to stable log-space representation.
        teacher_logits = tf.math.log(
            tf.clip_by_value(
                teacher_probs,
                1e-7,
                1.0
            )
        )

        with tf.GradientTape() as tape:

            # Student produces raw logits
            student_logits = self.student(
                x,
                training=True
            )

            # Hard-label loss
            student_loss = self.student_loss_fn(
                y,
                student_logits
            )

            # Soft teacher/student distributions
            teacher_soft = tf.nn.softmax(
                teacher_logits / self.temperature,
                axis=1
            )

            student_soft = tf.nn.softmax(
                student_logits / self.temperature,
                axis=1
            )

            # Knowledge-distillation loss
            distillation_loss = (
                self.distillation_loss_fn(
                    teacher_soft,
                    student_soft
                )
                * (self.temperature ** 2)
            )

            loss = (
                self.alpha * student_loss
                +
                (1.0 - self.alpha)
                * distillation_loss
            )

        gradients = tape.gradient(
            loss,
            self.student.trainable_variables
        )

        self.optimizer.apply_gradients(
            zip(
                gradients,
                self.student.trainable_variables
            )
        )

        self.loss_tracker.update_state(loss)

        self.student_loss_tracker.update_state(
            student_loss
        )

        self.distillation_loss_tracker.update_state(
            distillation_loss
        )

        self.accuracy_tracker.update_state(
            y,
            student_logits
        )

        return {
            "loss": self.loss_tracker.result(),
            "student_loss":
                self.student_loss_tracker.result(),
            "distillation_loss":
                self.distillation_loss_tracker.result(),
            "accuracy":
                self.accuracy_tracker.result()
        }


# ============================================================
# BUILD FROM OUR EPOCH-2 STUDENT
# ============================================================

distiller = Distiller(
    student=kd_student,
    teacher=teacher_disease_model
)

distiller.temperature = 3.0
distiller.alpha = 0.5

distiller.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),

    student_loss_fn=
        tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        ),

    distillation_loss_fn=
        tf.keras.losses.KLDivergence()
)


# ============================================================
# CHECKPOINT CALLBACK
# ============================================================

class StudentCheckpoint(tf.keras.callbacks.Callback):

    def __init__(self, student, start_epoch):
        super().__init__()

        self.student = student
        self.start_epoch = start_epoch

    def on_epoch_end(self, epoch, logs=None):

        # fit() below runs 6 new epochs.
        # We are continuing AFTER Epoch 2.
        actual_epoch = self.start_epoch + epoch + 1

        save_path = os.path.join(
            KD_DIR,
            f"dermawise_kd_student_epoch{actual_epoch:02d}.keras"
        )

        self.student.save(save_path)

        print(
            f"\n✅ STUDENT EPOCH {actual_epoch} "
            f"SAVED TO GOOGLE DRIVE"
        )

        print(save_path)


checkpoint_callback = StudentCheckpoint(
    student=kd_student,
    start_epoch=2
)


# ============================================================
# FINAL SAFETY CHECK
# ============================================================

assert tf.config.list_physical_devices("GPU"), \
    "GPU unavailable — DO NOT START"

print("===== MULTI-EPOCH KD SETUP =====")
print("Starting checkpoint : Epoch 2")
print("Next epoch          : Epoch 3")
print("Final planned epoch : Epoch 8")
print("Temperature         :", distiller.temperature)
print("Alpha               :", distiller.alpha)
print("Teacher trainable   :", teacher_disease_model.trainable)
print("GPU                 :", tf.config.list_physical_devices("GPU"))

print("\n===== STARTING GPU KD: EPOCHS 3 → 8 =====")


# ============================================================
# TRAIN SIX ADDITIONAL EPOCHS
# ============================================================

history = distiller.fit(
    train_ds,
    epochs=6,
    callbacks=[checkpoint_callback],
    verbose=1
)

print("\n========================================")
print("✅ MULTI-EPOCH KD TRAINING COMPLETE")
print("Epochs 3 → 8 completed")
print("========================================")

===== MULTI-EPOCH KD SETUP =====
Starting checkpoint : Epoch 2
Next epoch          : Epoch 3
Final planned epoch : Epoch 8
Temperature         : 3.0
Alpha               : 0.5
Teacher trainable   : False
GPU                 : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

===== STARTING GPU KD: EPOCHS 3 → 8 =====
Epoch 1/6
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.1453 - distillation_loss: 2.7431 - loss: 3.2139 - student_loss: 3.6846
✅ STUDENT EPOCH 3 SAVED TO GOOGLE DRIVE
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_epoch03.keras
499/499 ━━━━━━━━━━━━━━━━━━━━ 3018s 6s/step - accuracy: 0.1510 - distillation_loss: 2.6312 - loss: 3.1334 - student_loss: 3.6356
Epoch 2/6
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step - accuracy: 0.1884 - distillation_loss: 2.3948 - loss: 2.8931 - student_loss: 3.3914
✅ STUDENT EPOCH 4 SAVED TO GOOGLE DRIVE
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_epoch04.keras
499/499 ━━━━

In [ ]:
import os
import gc
import time
import numpy as np
import tensorflow as tf

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

results = []

print("===== KD CHECKPOINT VALIDATION =====\n")

for epoch in range(3, 9):

    model_path = os.path.join(
        KD_DIR,
        f"dermawise_kd_student_epoch{epoch:02d}.keras"
    )

    assert os.path.exists(model_path), \
        f"Epoch {epoch} checkpoint missing!"

    print(f"Evaluating KD Epoch {epoch}...")

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    top1_correct = 0
    top3_correct = 0
    top5_correct = 0
    total = 0

    start = time.time()

    for images, labels in val_ds:

        logits = model(
            images,
            training=False
        )

        probs = tf.nn.softmax(
            logits,
            axis=1
        )

        labels_np = labels.numpy()

        pred1 = tf.argmax(
            probs,
            axis=1,
            output_type=tf.int32
        ).numpy()

        pred3 = tf.math.top_k(
            probs,
            k=3
        ).indices.numpy()

        pred5 = tf.math.top_k(
            probs,
            k=5
        ).indices.numpy()

        top1_correct += np.sum(
            pred1 == labels_np
        )

        top3_correct += sum(
            labels_np[i] in pred3[i]
            for i in range(len(labels_np))
        )

        top5_correct += sum(
            labels_np[i] in pred5[i]
            for i in range(len(labels_np))
        )

        total += len(labels_np)

    top1 = 100 * top1_correct / total
    top3 = 100 * top3_correct / total
    top5 = 100 * top5_correct / total

    elapsed = time.time() - start

    results.append(
        (epoch, top1, top3, top5)
    )

    print(
        f"Epoch {epoch}: "
        f"Top-1 {top1:.2f}% | "
        f"Top-3 {top3:.2f}% | "
        f"Top-5 {top5:.2f}% | "
        f"{elapsed:.1f}s"
    )

    del model
    gc.collect()


print("\n===== FINAL KD VALIDATION COMPARISON =====")

print(
    "Epoch 1: Top-1 6.94% | "
    "Top-3 15.10% | Top-5 22.04%"
)

print(
    "Epoch 2: Top-1 10.27% | "
    "Top-3 21.42% | Top-5 29.50%"
)

for epoch, top1, top3, top5 in results:

    print(
        f"Epoch {epoch}: "
        f"Top-1 {top1:.2f}% | "
        f"Top-3 {top3:.2f}% | "
        f"Top-5 {top5:.2f}%"
    )

best = max(
    results,
    key=lambda x: x[1]
)

print("\n🏆 BEST EPOCH AMONG 3-8:", best[0])
print(f"Best Top-1: {best[1]:.2f}%")
print(f"Top-3     : {best[2]:.2f}%")
print(f"Top-5     : {best[3]:.2f}%")

===== KD CHECKPOINT VALIDATION =====

Evaluating KD Epoch 3...
Epoch 3: Top-1 14.22% | Top-3 29.06% | Top-5 38.10% | 425.9s
Evaluating KD Epoch 4...
Epoch 4: Top-1 17.65% | Top-3 35.47% | Top-5 44.78% | 20.5s
Evaluating KD Epoch 5...
Epoch 5: Top-1 21.42% | Top-3 39.51% | Top-5 49.96% | 17.4s
Evaluating KD Epoch 6...
Epoch 6: Top-1 23.71% | Top-3 42.67% | Top-5 51.62% | 18.3s
Evaluating KD Epoch 7...
Epoch 7: Top-1 24.67% | Top-3 44.78% | Top-5 54.78% | 20.5s
Evaluating KD Epoch 8...
Epoch 8: Top-1 26.69% | Top-3 44.51% | Top-5 54.96% | 18.5s

===== FINAL KD VALIDATION COMPARISON =====
Epoch 1: Top-1 6.94% | Top-3 15.10% | Top-5 22.04%
Epoch 2: Top-1 10.27% | Top-3 21.42% | Top-5 29.50%
Epoch 3: Top-1 14.22% | Top-3 29.06% | Top-5 38.10%
Epoch 4: Top-1 17.65% | Top-3 35.47% | Top-5 44.78%
Epoch 5: Top-1 21.42% | Top-3 39.51% | Top-5 49.96%
Epoch 6: Top-1 23.71% | Top-3 42.67% | Top-5 51.62%
Epoch 7: Top-1 24.67% | Top-3 44.78% | Top-5 54.78%
Epoch 8: Top-1 26.69% | Top-3 44.51% | Top-5

In [ ]:
import os
import tensorflow as tf

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

assert tf.config.list_physical_devices("GPU"), \
    "❌ GPU unavailable — STOP"


class ContinueCheckpoint(tf.keras.callbacks.Callback):

    def __init__(self, student):
        super().__init__()
        self.student = student

    def on_epoch_end(self, epoch, logs=None):

        # fit() epochs 0-3 correspond to actual KD 9-12
        actual_epoch = 9 + epoch

        path = os.path.join(
            KD_DIR,
            f"dermawise_kd_student_epoch{actual_epoch:02d}.keras"
        )

        self.student.save(path)

        print(
            f"\n✅ KD EPOCH {actual_epoch} "
            f"SAVED TO GOOGLE DRIVE"
        )


checkpoint = ContinueCheckpoint(
    distiller.student
)

print("===== CONTINUING KD =====")
print("Starting from : Epoch 8")
print("Training      : Epochs 9 → 12")
print("GPU           :", tf.config.list_physical_devices("GPU"))

history_9_12 = distiller.fit(
    train_ds,
    epochs=4,
    callbacks=[checkpoint],
    verbose=1
)

print("\n================================")
print("✅ KD EPOCHS 9 → 12 COMPLETE")
print("================================")

===== CONTINUING KD =====
Starting from : Epoch 8
Training      : Epochs 9 → 12
GPU           : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Epoch 1/4
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.3408 - distillation_loss: 1.8924 - loss: 2.2634 - student_loss: 2.6344
✅ KD EPOCH 9 SAVED TO GOOGLE DRIVE
499/499 ━━━━━━━━━━━━━━━━━━━━ 105s 210ms/step - accuracy: 0.3482 - distillation_loss: 1.8808 - loss: 2.2465 - student_loss: 2.6122
Epoch 2/4
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - accuracy: 0.3611 - distillation_loss: 1.8337 - loss: 2.1825 - student_loss: 2.5314
✅ KD EPOCH 10 SAVED TO GOOGLE DRIVE
499/499 ━━━━━━━━━━━━━━━━━━━━ 96s 193ms/step - accuracy: 0.3699 - distillation_loss: 1.8288 - loss: 2.1622 - student_loss: 2.4956
Epoch 3/4
499/499 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step - accuracy: 0.3892 - distillation_loss: 1.7897 - loss: 2.1013 - student_loss: 2.4129
✅ KD EPOCH 11 SAVED TO GOOGLE DRIVE
499/499 ━━━━━━━━━━━━━━━━━━━━ 94s 188ms/step - accuracy: 0

In [ ]:
import os
import gc
import time
import numpy as np
import tensorflow as tf

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

results_9_12 = []

print("===== KD EPOCHS 9-12 VALIDATION =====\n")

for epoch in range(9, 13):

    path = os.path.join(
        KD_DIR,
        f"dermawise_kd_student_epoch{epoch:02d}.keras"
    )

    assert os.path.exists(path), \
        f"❌ Epoch {epoch} checkpoint missing"

    print(f"Evaluating Epoch {epoch}...")

    model = tf.keras.models.load_model(
        path,
        compile=False
    )

    top1_correct = 0
    top3_correct = 0
    top5_correct = 0
    total = 0

    start = time.time()

    for images, labels in val_ds:

        logits = model(
            images,
            training=False
        )

        probs = tf.nn.softmax(
            logits,
            axis=1
        )

        labels_np = labels.numpy()

        pred1 = tf.argmax(
            probs,
            axis=1,
            output_type=tf.int32
        ).numpy()

        pred3 = tf.math.top_k(
            probs,
            k=3
        ).indices.numpy()

        pred5 = tf.math.top_k(
            probs,
            k=5
        ).indices.numpy()

        top1_correct += np.sum(
            pred1 == labels_np
        )

        top3_correct += sum(
            labels_np[i] in pred3[i]
            for i in range(len(labels_np))
        )

        top5_correct += sum(
            labels_np[i] in pred5[i]
            for i in range(len(labels_np))
        )

        total += len(labels_np)

    top1 = 100 * top1_correct / total
    top3 = 100 * top3_correct / total
    top5 = 100 * top5_correct / total

    elapsed = time.time() - start

    results_9_12.append(
        (epoch, top1, top3, top5)
    )

    print(
        f"Epoch {epoch}: "
        f"Top-1 {top1:.2f}% | "
        f"Top-3 {top3:.2f}% | "
        f"Top-5 {top5:.2f}% | "
        f"{elapsed:.1f}s"
    )

    del model
    gc.collect()


print("\n===== VALIDATION SUMMARY =====")

for epoch, top1, top3, top5 in results_9_12:

    print(
        f"Epoch {epoch}: "
        f"Top-1 {top1:.2f}% | "
        f"Top-3 {top3:.2f}% | "
        f"Top-5 {top5:.2f}%"
    )

best = max(
    results_9_12,
    key=lambda x: x[1]
)

print("\n🏆 BEST EPOCH 9-12:", best[0])
print(f"Top-1: {best[1]:.2f}%")
print(f"Top-3: {best[2]:.2f}%")
print(f"Top-5: {best[3]:.2f}%")

===== KD EPOCHS 9-12 VALIDATION =====

Evaluating Epoch 9...
Epoch 9: Top-1 28.01% | Top-3 46.36% | Top-5 56.72% | 27.5s
Evaluating Epoch 10...


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

print("===== SAVED KD CHECKPOINTS =====")

for epoch in range(9, 13):
    path = os.path.join(
        KD_DIR,
        f"dermawise_kd_student_epoch{epoch:02d}.keras"
    )

    print(
        f"Epoch {epoch}:",
        "✅ FOUND" if os.path.exists(path) else "❌ MISSING"
    )

Mounted at /content/drive
===== SAVED KD CHECKPOINTS =====
Epoch 9: ✅ FOUND
Epoch 10: ✅ FOUND
Epoch 11: ✅ FOUND
Epoch 12: ✅ FOUND


In [ ]:
import os
import gc
import time
import numpy as np
import pandas as pd
import tensorflow as tf

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"
KD_DIR = os.path.join(WORK_DIR, "KD")

TRAIN_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

VAL_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_val.csv"
)

# ============================================================
# RESTORE CLASS MAPPING
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

CLASS_NAMES = sorted(
    train_df["label"].unique().tolist()
)

class_to_index = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

val_df["class_id"] = (
    val_df["label"]
    .map(class_to_index)
    .astype("int32")
)

assert len(CLASS_NAMES) == 114
assert val_df["class_id"].notna().all()

print("Validation images:", len(val_df))
print("Classes:", len(CLASS_NAMES))


# ============================================================
# VALIDATION DATASET
# ============================================================

IMG_SIZE = 260
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_val_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


val_ds = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].astype(str).values,
        val_df["class_id"].values
    )
)

val_ds = (
    val_ds
    .map(
        load_val_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Validation batches:",
      tf.data.experimental.cardinality(val_ds).numpy())


# ============================================================
# EVALUATE EPOCHS 9-12
# ============================================================

results = []

print("\n===== KD EPOCHS 9-12 VALIDATION =====\n")

for epoch in range(9, 13):

    path = os.path.join(
        KD_DIR,
        f"dermawise_kd_student_epoch{epoch:02d}.keras"
    )

    assert os.path.exists(path), \
        f"Epoch {epoch} missing!"

    print(f"Evaluating Epoch {epoch}...")

    model = tf.keras.models.load_model(
        path,
        compile=False
    )

    top1_correct = 0
    top3_correct = 0
    top5_correct = 0
    total = 0

    start = time.time()

    for images, labels in val_ds:

        logits = model(
            images,
            training=False
        )

        probs = tf.nn.softmax(
            logits,
            axis=1
        )

        labels_np = labels.numpy()

        pred1 = tf.argmax(
            probs,
            axis=1,
            output_type=tf.int32
        ).numpy()

        pred3 = tf.math.top_k(
            probs,
            k=3
        ).indices.numpy()

        pred5 = tf.math.top_k(
            probs,
            k=5
        ).indices.numpy()

        top1_correct += np.sum(
            pred1 == labels_np
        )

        top3_correct += sum(
            labels_np[i] in pred3[i]
            for i in range(len(labels_np))
        )

        top5_correct += sum(
            labels_np[i] in pred5[i]
            for i in range(len(labels_np))
        )

        total += len(labels_np)

    top1 = 100 * top1_correct / total
    top3 = 100 * top3_correct / total
    top5 = 100 * top5_correct / total

    elapsed = time.time() - start

    results.append(
        (epoch, top1, top3, top5)
    )

    print(
        f"Epoch {epoch}: "
        f"Top-1 {top1:.2f}% | "
        f"Top-3 {top3:.2f}% | "
        f"Top-5 {top5:.2f}% | "
        f"{elapsed/60:.2f} min"
    )

    del model
    gc.collect()


# ============================================================
# RESULTS
# ============================================================

print("\n===== FINAL KD VALIDATION COMPARISON =====")

print(
    "Epoch 8 : "
    "Top-1 26.69% | "
    "Top-3 44.51% | "
    "Top-5 54.96%"
)

for epoch, top1, top3, top5 in results:

    print(
        f"Epoch {epoch}: "
        f"Top-1 {top1:.2f}% | "
        f"Top-3 {top3:.2f}% | "
        f"Top-5 {top5:.2f}%"
    )

all_results = [
    (8, 26.69, 44.51, 54.96)
] + results

best = max(
    all_results,
    key=lambda x: x[1]
)

print("\n🏆 BEST CHECKPOINT BY TOP-1")
print("Epoch :", best[0])
print(f"Top-1 : {best[1]:.2f}%")
print(f"Top-3 : {best[2]:.2f}%")
print(f"Top-5 : {best[3]:.2f}%")

Validation images: 1139
Classes: 114
Validation batches: 36

===== KD EPOCHS 9-12 VALIDATION =====

Evaluating Epoch 9...
Epoch 9: Top-1 28.01% | Top-3 46.36% | Top-5 56.72% | 4.77 min
Evaluating Epoch 10...
Epoch 10: Top-1 28.45% | Top-3 47.76% | Top-5 56.19% | 1.37 min
Evaluating Epoch 11...
Epoch 11: Top-1 29.32% | Top-3 49.52% | Top-5 57.77% | 0.69 min
Evaluating Epoch 12...
Epoch 12: Top-1 30.99% | Top-3 49.43% | Top-5 57.77% | 0.73 min

===== FINAL KD VALIDATION COMPARISON =====
Epoch 8 : Top-1 26.69% | Top-3 44.51% | Top-5 54.96%
Epoch 9: Top-1 28.01% | Top-3 46.36% | Top-5 56.72%
Epoch 10: Top-1 28.45% | Top-3 47.76% | Top-5 56.19%
Epoch 11: Top-1 29.32% | Top-3 49.52% | Top-5 57.77%
Epoch 12: Top-1 30.99% | Top-3 49.43% | Top-5 57.77%

🏆 BEST CHECKPOINT BY TOP-1
Epoch : 12
Top-1 : 30.99%
Top-3 : 49.43%
Top-5 : 57.77%


In [ ]:
import os
import shutil

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

SOURCE = os.path.join(
    KD_DIR,
    "dermawise_kd_student_epoch12.keras"
)

FINAL = os.path.join(
    KD_DIR,
    "dermawise_kd_student_best.keras"
)

assert os.path.exists(SOURCE), "❌ Epoch 12 checkpoint missing"

shutil.copy2(
    SOURCE,
    FINAL
)

print("===== FINAL KD STUDENT =====")
print("Selected checkpoint : Epoch 12")
print("Validation Top-1    : 30.99%")
print("Validation Top-3    : 49.43%")
print("Validation Top-5    : 57.77%")
print()
print("Saved as:")
print(FINAL)
print()
print("Exists:", os.path.exists(FINAL))
print(
    "Size:",
    round(os.path.getsize(FINAL)/(1024**2), 2),
    "MB"
)
print("\n✅ FINAL KD STUDENT LOCKED")

===== FINAL KD STUDENT =====
Selected checkpoint : Epoch 12
Validation Top-1    : 30.99%
Validation Top-3    : 49.43%
Validation Top-5    : 57.77%

Saved as:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/dermawise_kd_student_best.keras

Exists: True
Size: 4.37 MB

✅ FINAL KD STUDENT LOCKED


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    classification_report,
    f1_score,
    recall_score
)

# ============================================================
# PATHS
# ============================================================

WORK_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

KD_MODEL_PATH = os.path.join(
    WORK_DIR,
    "KD",
    "dermawise_kd_student_best.keras"
)

TRAIN_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_train.csv"
)

TEST_CSV = os.path.join(
    WORK_DIR,
    "fitzpatrick114_test.csv"
)

assert os.path.exists(KD_MODEL_PATH), \
    "❌ Final KD model missing"

assert os.path.exists(TEST_CSV), \
    "❌ Test CSV missing"

# ============================================================
# CLASS MAPPING
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

CLASS_NAMES = sorted(
    train_df["label"].unique().tolist()
)

class_to_index = {
    name: i
    for i, name in enumerate(CLASS_NAMES)
}

test_df["class_id"] = (
    test_df["label"]
    .map(class_to_index)
    .astype("int32")
)

assert len(CLASS_NAMES) == 114
assert test_df["class_id"].notna().all()

print("===== OFFICIAL KD TEST =====")
print("Test images:", len(test_df))
print("Classes    :", len(CLASS_NAMES))

# ============================================================
# TEST DATASET
# ============================================================

IMG_SIZE = 260
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_test_image(filepath, label):

    image = tf.io.read_file(filepath)

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )

    image = tf.cast(
        image,
        tf.float32
    )

    return image, label


test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["image_path"].astype(str).values,
        test_df["class_id"].values
    )
)

test_ds = (
    test_ds
    .map(
        load_test_image,
        num_parallel_calls=AUTOTUNE
    )
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print(
    "Test batches:",
    tf.data.experimental.cardinality(test_ds).numpy()
)

# ============================================================
# LOAD LOCKED STUDENT
# ============================================================

print("\nLoading locked KD student...")

model = tf.keras.models.load_model(
    KD_MODEL_PATH,
    compile=False
)

print("✅ KD student loaded")
print("Input :", model.input_shape)
print("Output:", model.output_shape)

# ============================================================
# SINGLE TEST PASS
# ============================================================

print("\nRunning official test evaluation...")

all_probs = []
all_labels = []

start = time.time()

for images, labels in test_ds:

    logits = model(
        images,
        training=False
    )

    probs = tf.nn.softmax(
        logits,
        axis=1
    )

    all_probs.append(
        probs.numpy()
    )

    all_labels.append(
        labels.numpy()
    )

probs = np.concatenate(
    all_probs,
    axis=0
)

y_true = np.concatenate(
    all_labels,
    axis=0
)

elapsed = time.time() - start

assert len(y_true) == 2280
assert probs.shape == (2280, 114)

# ============================================================
# TOP-K
# ============================================================

top1_predictions = np.argmax(
    probs,
    axis=1
)

top1 = np.mean(
    top1_predictions == y_true
)

top3_indices = np.argsort(
    probs,
    axis=1
)[:, -3:]

top5_indices = np.argsort(
    probs,
    axis=1
)[:, -5:]

top3 = np.mean([
    y_true[i] in top3_indices[i]
    for i in range(len(y_true))
])

top5 = np.mean([
    y_true[i] in top5_indices[i]
    for i in range(len(y_true))
])

# ============================================================
# F1 / RECALL
# ============================================================

macro_f1 = f1_score(
    y_true,
    top1_predictions,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    top1_predictions,
    average="weighted",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    top1_predictions,
    average="macro",
    zero_division=0
)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    y_true,
    top1_predictions,
    labels=np.arange(114),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(
    report
).transpose()

REPORT_PATH = os.path.join(
    WORK_DIR,
    "KD",
    "fitzpatrick114_kd_final_classification_report.csv"
)

report_df.to_csv(
    REPORT_PATH,
    index=True
)

# ============================================================
# SAVE PREDICTIONS
# ============================================================

predictions_df = test_df.copy()

predictions_df["true_class_id"] = y_true
predictions_df["predicted_class_id"] = top1_predictions

predictions_df["predicted_label"] = [
    CLASS_NAMES[i]
    for i in top1_predictions
]

predictions_df["top1_score"] = np.max(
    probs,
    axis=1
)

PREDICTIONS_PATH = os.path.join(
    WORK_DIR,
    "KD",
    "fitzpatrick114_kd_final_test_predictions.csv"
)

predictions_df.to_csv(
    PREDICTIONS_PATH,
    index=False
)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n========================================")
print(" FINAL KD STUDENT — OFFICIAL TEST")
print("========================================")

print("Images        :", len(y_true))
print(f"Top-1         : {top1*100:.2f}%")
print(f"Top-3         : {top3*100:.2f}%")
print(f"Top-5         : {top5*100:.2f}%")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Weighted F1   : {weighted_f1:.4f}")
print(f"Macro Recall  : {macro_recall:.4f}")
print(f"Time          : {elapsed/60:.2f} minutes")

print("\nTeacher official test:")
print("Top-1         : 30.44%")
print("Top-3         : 49.30%")
print("Top-5         : 57.89%")
print("Macro F1      : 0.2784")
print("Weighted F1   : 0.2882")
print("Macro Recall  : 0.2785")

print("\nSaved:")
print(REPORT_PATH)
print(PREDICTIONS_PATH)

print("\n✅ FINAL KD TEST COMPLETE")

===== OFFICIAL KD TEST =====
Test images: 2280
Classes    : 114
Test batches: 72

Loading locked KD student...
✅ KD student loaded
Input : (None, 260, 260, 3)
Output: (None, 114)

Running official test evaluation...

 FINAL KD STUDENT — OFFICIAL TEST
Images        : 2280
Top-1         : 31.67%
Top-3         : 50.92%
Top-5         : 60.48%
Macro F1      : 0.2684
Weighted F1   : 0.2923
Macro Recall  : 0.2758
Time          : 9.45 minutes

Teacher official test:
Top-1         : 30.44%
Top-3         : 49.30%
Top-5         : 57.89%
Macro F1      : 0.2784
Weighted F1   : 0.2882
Macro Recall  : 0.2785

Saved:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/fitzpatrick114_kd_final_classification_report.csv
/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD/fitzpatrick114_kd_final_test_predictions.csv

✅ FINAL KD TEST COMPLETE


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

files = [
    "dermawise_kd_student_best.keras",
    "fitzpatrick114_kd_final_classification_report.csv",
    "fitzpatrick114_kd_final_test_predictions.csv"
]

for name in files:
    path = os.path.join(KD_DIR, name)
    print(
        "✅ FOUND" if os.path.exists(path) else "❌ MISSING",
        name
    )

✅ FOUND dermawise_kd_student_best.keras
✅ FOUND fitzpatrick114_kd_final_classification_report.csv
✅ FOUND fitzpatrick114_kd_final_test_predictions.csv


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.metrics import (
    f1_score,
    recall_score
)

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

PRED_PATH = os.path.join(
    KD_DIR,
    "fitzpatrick114_kd_final_test_predictions.csv"
)

df = pd.read_csv(PRED_PATH)

print("Columns:")
print(df.columns.tolist())
print("\nRows:", len(df))

Columns:
['filepath', 'filename', 'diag', 'fst', 'partition', 'md5hash', 'fitzpatrick', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum', 'orig_img_name', 'new_img_name', 'image_path', 'exists', 'class_id', 'true_class_id', 'predicted_class_id', 'predicted_label', 'top1_score']

Rows: 2280


In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report
)

KD_DIR = "/content/drive/MyDrive/DermaWise/Fitzpatrick114/KD"

PRED_PATH = os.path.join(
    KD_DIR,
    "fitzpatrick114_kd_final_test_predictions.csv"
)

df = pd.read_csv(PRED_PATH)

y_true = df["true_class_id"].astype(int).values
y_pred = df["predicted_class_id"].astype(int).values

# -------------------------------
# FINAL METRICS
# -------------------------------

top1 = accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

macro_recall = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

# Per-class F1
report = classification_report(
    y_true,
    y_pred,
    labels=np.arange(114),
    output_dict=True,
    zero_division=0
)

zero_f1 = 0

for i in range(114):
    if report[str(i)]["f1-score"] == 0:
        zero_f1 += 1


print("========================================")
print(" FINAL KD STUDENT — OFFICIAL TEST")
print("========================================")

print("Test images     :", len(df))
print(f"Top-1 Accuracy  : {top1 * 100:.2f}%")
print(f"Macro F1        : {macro_f1:.4f}")
print(f"Weighted F1     : {weighted_f1:.4f}")
print(f"Macro Recall    : {macro_recall:.4f}")
print(f"Zero-F1 classes : {zero_f1}")

print("\n========================================")
print(" HIERARCHICAL TEACHER — OFFICIAL TEST")
print("========================================")

print("Top-1 Accuracy  : 30.44%")
print("Top-3 Accuracy  : 49.30%")
print("Top-5 Accuracy  : 57.89%")
print("Macro F1        : 0.2784")
print("Weighted F1     : 0.2882")
print("Macro Recall    : 0.2785")

print("\nKD student parameters : ~1.005M")
print("Teacher parameters    : ~7.929M")
print("Parameter reduction   : ~87.33%")

 FINAL KD STUDENT — OFFICIAL TEST
Test images     : 2280
Top-1 Accuracy  : 31.67%
Macro F1        : 0.2684
Weighted F1     : 0.2923
Macro Recall    : 0.2758
Zero-F1 classes : 22

 HIERARCHICAL TEACHER — OFFICIAL TEST
Top-1 Accuracy  : 30.44%
Top-3 Accuracy  : 49.30%
Top-5 Accuracy  : 57.89%
Macro F1        : 0.2784
Weighted F1     : 0.2882
Macro Recall    : 0.2785

KD student parameters : ~1.005M
Teacher parameters    : ~7.929M
Parameter reduction   : ~87.33%


In [ ]:
import os
import tensorflow as tf

KD_MODEL = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/KD/"
    "dermawise_kd_student_best.keras"
)

TFLITE_DIR = (
    "/content/drive/MyDrive/DermaWise/"
    "Fitzpatrick114/TFLite"
)

os.makedirs(TFLITE_DIR, exist_ok=True)

TFLITE_PATH = os.path.join(
    TFLITE_DIR,
    "dermawise_kd_student_final_float16.tflite"
)

# Load final trained KD student
model = tf.keras.models.load_model(
    KD_MODEL,
    compile=False
)

print("Loaded:", model.name)
print("Input :", model.input_shape)
print("Output:", model.output_shape)
print("Parameters:", model.count_params())

# --------------------------------
# FLOAT16 TFLITE CONVERSION
# --------------------------------

converter = tf.lite.TFLiteConverter.from_keras_model(
    model
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.target_spec.supported_types = [
    tf.float16
]

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

keras_size = os.path.getsize(KD_MODEL) / (1024**2)
tflite_size = os.path.getsize(TFLITE_PATH) / (1024**2)

print("\n================================")
print(" FINAL FLOAT16 TFLITE MODEL")
print("================================")

print(f"Keras size : {keras_size:.2f} MB")
print(f"TFLite size: {tflite_size:.2f} MB")
print(
    f"Size reduction: "
    f"{(1 - tflite_size/keras_size)*100:.2f}%"
)

print("\nSaved:")
print(TFLITE_PATH)

print("\n✅ FINAL TFLITE CONVERSION COMPLETE")

Loaded: DermaWise_MobileNetV3Small_Student
Input : (None, 260, 260, 3)
Output: (None, 114)
Parameters: 1004898
Saved artifact at '/tmp/tmpp31qnd1o'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name='student_image')
Output Type:
  TensorSpec(shape=(None, 114), dtype=tf.float32, name=None)
Captures:
  140636959248784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959247632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959246864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959248400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959249168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959249360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959249552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959249744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140636959250128: TensorSpec(shape=(), dt

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

BASE = "/content/drive/MyDrive/DermaWise/Fitzpatrick114"

KERAS_PATH = os.path.join(
    BASE,
    "KD",
    "dermawise_kd_student_best.keras"
)

TFLITE_PATH = os.path.join(
    BASE,
    "TFLite",
    "dermawise_kd_student_final_float16.tflite"
)

TEST_CSV = os.path.join(
    BASE,
    "fitzpatrick114_test.csv"
)

# ==========================================
# LOAD ONE REAL TEST IMAGE
# ==========================================

test_df = pd.read_csv(TEST_CSV)

image_path = test_df.iloc[0]["image_path"]

print("Test image:")
print(image_path)

image = tf.io.read_file(image_path)

image = tf.io.decode_image(
    image,
    channels=3,
    expand_animations=False
)

image = tf.image.resize(
    image,
    [260, 260]
)

image = tf.cast(
    image,
    tf.float32
)

image = tf.expand_dims(
    image,
    axis=0
)

print("\nInput shape:", image.shape)
print(
    "Input range:",
    float(tf.reduce_min(image)),
    "to",
    float(tf.reduce_max(image))
)

# ==========================================
# KERAS PREDICTION
# ==========================================

keras_model = tf.keras.models.load_model(
    KERAS_PATH,
    compile=False
)

keras_logits = keras_model(
    image,
    training=False
).numpy()

keras_probs = tf.nn.softmax(
    keras_logits,
    axis=1
).numpy()

keras_pred = int(
    np.argmax(keras_probs[0])
)

# ==========================================
# TFLITE PREDICTION
# ==========================================

interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("\nTFLite input:")
print(input_details[0]["shape"], input_details[0]["dtype"])

print("TFLite output:")
print(output_details[0]["shape"], output_details[0]["dtype"])

interpreter.set_tensor(
    input_details[0]["index"],
    image.numpy().astype(
        input_details[0]["dtype"]
    )
)

interpreter.invoke()

tflite_logits = interpreter.get_tensor(
    output_details[0]["index"]
)

tflite_probs = tf.nn.softmax(
    tflite_logits,
    axis=1
).numpy()

tflite_pred = int(
    np.argmax(tflite_probs[0])
)

# ==========================================
# COMPARE
# ==========================================

max_diff = np.max(
    np.abs(
        keras_probs - tflite_probs
    )
)

mean_diff = np.mean(
    np.abs(
        keras_probs - tflite_probs
    )
)

print("\n====================================")
print(" KERAS vs TFLITE VERIFICATION")
print("====================================")

print("Keras predicted index :", keras_pred)
print("TFLite predicted index:", tflite_pred)

print(
    "Same Top-1 prediction:",
    keras_pred == tflite_pred
)

print(
    "Maximum probability difference:",
    float(max_diff)
)

print(
    "Mean probability difference:",
    float(mean_diff)
)

if (
    keras_pred == tflite_pred
    and max_diff < 0.01
):
    print(
        "\n✅ FINAL TFLITE MODEL VERIFIED"
    )
else:
    print(
        "\n⚠️ Prediction difference detected — inspect before deployment"
    )

Test image:
/content/drive/MyDrive/DermaWise/Fitzpatrick114/Fitzpatrick17k_C/Categorized_AbbrvName/pa_59/pa_f4_17_4f4f5d2d.jpg

Input shape: (1, 260, 260, 3)
Input range: 0.0 to 255.0

TFLite input:
[  1 260 260   3] <class 'numpy.float32'>
TFLite output:
[  1 114] <class 'numpy.float32'>

 KERAS vs TFLITE VERIFICATION
Keras predicted index : 17
TFLite predicted index: 17
Same Top-1 prediction: True
Maximum probability difference: 0.004412621259689331
Mean probability difference: 0.0001332036918029189

✅ FINAL TFLITE MODEL VERIFIED


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
